P4-0 — Global setup, imports, random seed

In [ ]:
# ============================================================
# P4-0 — Global setup, imports, random seed
# Automatic Model Training Pipeline
# ============================================================

import os
import re
import json
import math
import hashlib
import warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from datetime import datetime

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

print("P4-0 ready")
print("Python environment ready")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

P4-1 — Define AutoTrainingConfig

In [ ]:
# ============================================================
# P4-1 — Define AutoTrainingConfig
# ============================================================

@dataclass
class AutoTrainingConfig:
    # ---- Identity ----
    pipeline_name: str = "automatic_model_training_pipeline" 
    process_id: str = "abc"

    # ---- Input ----
    data_path: str = "..." # <--- put file path here for example C:\DatasetTest\data1.csv or just file name if the dataset file is in the same folder as this notebook for example data1.csv

    # ---- Required raw schema ----
    occurred_col: str = "occurred"
    machine_col: str = "mc_no"
    status_col: str = "mc_status"
    process_col: str = "process"

    # ---- Status definition ----
    # 😮 Normal statuses are excluded from prediction targets. <--- define normal status here
    normal_statuses: list = field(default_factory=lambda: ["run, MC_RUN"])

    # "auto" means all statuses not in normal_statuses become target statuses.
    # Or set explicit list, e.g. ["mc_alarm", "mc_stop"]
    target_statuses: str | list = "auto" # <--- target status

    # 😮 Optional status aliases.
    # Keys and values will be normalized automatically. <--- go to status name check notebook and define status names to be in lower case and make sure to not have spacebar, use _ instead of spacebar, below are examples, set them based on your real data
    status_alias_map: dict = field(default_factory=lambda: {
        "RUN": "run",
        "MC_RUN": "mc_run",
        "fullwork": "fullwork",
        "no work": "no_work",
        "m/c stop": "mc_stop",
        "alarm": "alarm",
        
    })

    # 😮 Dashboard/display only. Does not affect training target. <--- define normal status and status you don't want to show on dashboard here
    hidden_statuses: list = field(default_factory=lambda: ["run, MC_RUN"])

    # ---- Data validation ----
    allow_multiple_processes: bool = False
    min_rows: int = 1_000
    min_machines: int = 1
    min_target_rows: int = 100

    # ---- Deduplication ----
    # exact_duplicate_rows: same process, mc_no, occurred, mc_status
    # same_timestamp_conflict_policy:
    #   "drop_conflicts" = drop rows where same process+mc_no+occurred has multiple statuses
    #   "keep"           = keep them, later target labeling may remove zero/invalid gaps
    same_timestamp_conflict_policy: str = "drop_conflicts"

    # ---- Later batches will use these ----
    rolling_windows_min: list = field(default_factory=lambda: [5, 15, 30, 60, 120])
    train_frac: float = 0.70
    val_frac: float = 0.15
    test_frac: float = 0.15

    target_clip_quantile: float = 0.99

    confidence_threshold: float = 0.60

    artifact_dir: str = "./artifacts_phase4"
    artifact_schema_version: str = "phase4_auto_v1"
    feature_version: str = "auto_features_v1"


CFG = AutoTrainingConfig()

print("P4-1 ready")
print(json.dumps(asdict(CFG), indent=2, ensure_ascii=False))

P4-2 — Load cleaned dataset

In [ ]:
# ============================================================
# P4-2 — Load cleaned dataset
# ============================================================

def load_table(path: str) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Data file not found: {path}")

    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    elif suffix in [".parquet", ".pq"]:
        return pd.read_parquet(path)
    elif suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    else:
        raise ValueError(
            f"Unsupported file type: {suffix}. "
            "Supported: .csv, .parquet, .pq, .xlsx, .xls"
        )


df_raw = load_table(CFG.data_path)

print("P4-2 ready")
print("Loaded:", CFG.data_path)
print("Shape:", df_raw.shape)
print("Columns:", list(df_raw.columns))

display(df_raw.head())
display(df_raw.dtypes.to_frame("dtype"))

P4-3 — Standardize schema and normalize status names

In [ ]:
# ============================================================
# P4-3 — Standardize schema and normalize status names
# ============================================================

def normalize_col_name(x: str) -> str:
    return str(x).strip()


def normalize_status_text(x) -> str:
    """
    Canonical status text for modeling.
    Examples:
      'RUN'         -> 'run'
      'mc_fullWork' -> 'mc_fullwork'
      'm/c stop'   -> 'm_c_stop'
      'mc waitpart'-> 'mc_waitpart'
    """
    if pd.isna(x):
        return np.nan

    s = str(x).strip().lower()
    s = s.replace("\u00a0", " ")
    s = re.sub(r"\s+", "_", s)
    s = s.replace("/", "_")
    s = s.replace("-", "_")
    s = re.sub(r"_+", "_", s)
    s = s.strip("_")
    return s


def sanitize_feature_token(x) -> str:
    """
    Stable feature-name-safe token.
    Used later for dynamic one-hot columns.
    """
    s = normalize_status_text(x)
    if pd.isna(s):
        return "missing"
    s = re.sub(r"[^a-zA-Z0-9_]+", "_", s)
    s = re.sub(r"_+", "_", s)
    return s.strip("_")


def canonicalize_status(x, alias_map: dict) -> str:
    s = normalize_status_text(x)

    alias_norm = {
        normalize_status_text(k): normalize_status_text(v)
        for k, v in alias_map.items()
    }

    return alias_norm.get(s, s)


def standardize_schema(df: pd.DataFrame, cfg: AutoTrainingConfig) -> pd.DataFrame:
    out = df.copy()

    # Strip column names
    out.columns = [normalize_col_name(c) for c in out.columns]

    required_raw_cols = [
        cfg.occurred_col,
        cfg.machine_col,
        cfg.status_col,
        cfg.process_col,
    ]

    missing = [c for c in required_raw_cols if c not in out.columns]
    if missing:
        raise ValueError(
            f"Missing required raw columns: {missing}. "
            f"Available columns: {list(out.columns)}"
        )

    # Rename to standard internal schema
    rename_map = {
        cfg.occurred_col: "occurred",
        cfg.machine_col: "mc_no",
        cfg.status_col: "mc_status_raw",
        cfg.process_col: "process",
    }

    out = out.rename(columns=rename_map)

    # Keep only standard required columns for this pipeline phase
    out = out[["occurred", "mc_no", "mc_status_raw", "process"]].copy()

    # Parse timestamp
    out["occurred"] = pd.to_datetime(out["occurred"], errors="coerce")

    # Normalize text columns
    out["mc_no"] = out["mc_no"].astype(str).str.strip()
    out["process"] = out["process"].astype(str).str.strip().str.lower()
    out["mc_status_raw"] = out["mc_status_raw"].astype(str).str.strip()

    # Canonical model status
    out["mc_status"] = out["mc_status_raw"].apply(
        lambda x: canonicalize_status(x, cfg.status_alias_map)
    )

    # Stable feature token for future feature engineering
    out["mc_status_token"] = out["mc_status"].apply(sanitize_feature_token)

    return out


df_std = standardize_schema(df_raw, CFG)

print("P4-3 ready")
print("Standardized shape:", df_std.shape)
print("Columns:", list(df_std.columns))

display(df_std.head())
display(df_std.dtypes.to_frame("dtype"))

print("\nRaw status values:")
display(df_std["mc_status_raw"].value_counts(dropna=False).to_frame("count"))

print("\nCanonical status values:")
display(df_std["mc_status"].value_counts(dropna=False).to_frame("count"))

P4-4 — Define normal_statuses and target_statuses

In [ ]:
# ============================================================
# P4-4 — Define normal_statuses and target_statuses
# ============================================================

NORMAL_STATUSES = set(normalize_status_text(s) for s in CFG.normal_statuses)

ALL_STATUSES = sorted(df_std["mc_status"].dropna().unique().tolist())

if CFG.target_statuses == "auto":
    TARGET_STATUSES = set(s for s in ALL_STATUSES if s not in NORMAL_STATUSES)
else:
    TARGET_STATUSES = set(normalize_status_text(s) for s in CFG.target_statuses)

IGNORED_STATUSES = set(ALL_STATUSES) - NORMAL_STATUSES - TARGET_STATUSES

STATUS_ROLE = {}
for s in ALL_STATUSES:
    if s in NORMAL_STATUSES:
        STATUS_ROLE[s] = "normal"
    elif s in TARGET_STATUSES:
        STATUS_ROLE[s] = "target"
    else:
        STATUS_ROLE[s] = "ignored"

status_role_df = (
    pd.DataFrame({
        "mc_status": ALL_STATUSES,
        "role": [STATUS_ROLE[s] for s in ALL_STATUSES],
        "count": [int((df_std["mc_status"] == s).sum()) for s in ALL_STATUSES],
    })
    .sort_values(["role", "count"], ascending=[True, False])
    .reset_index(drop=True)
)

print("P4-4 ready")
print("NORMAL_STATUSES:", sorted(NORMAL_STATUSES))
print("TARGET_STATUSES:", sorted(TARGET_STATUSES))
print("IGNORED_STATUSES:", sorted(IGNORED_STATUSES))

display(status_role_df)

P4-5 — Deduplicate exact duplicate rows

This removes exact duplicate event rows and conservatively drops same-machine/same-timestamp status conflicts without using any priority rule.

In [ ]:
# ============================================================
# P4-5 — Deduplicate exact duplicate rows
# ============================================================

def deduplicate_events(df: pd.DataFrame, cfg: AutoTrainingConfig) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    # 1. Validate configuration upfront
    if cfg.same_timestamp_conflict_policy not in ["drop_conflicts", "keep"]:
        raise ValueError(
            "same_timestamp_conflict_policy must be either "
            "'drop_conflicts' or 'keep'"
        )

    out = df.copy()
    before_rows = len(out)

    exact_key = ["process", "mc_no", "occurred", "mc_status"]
    exact_duplicate_rows = int(out.duplicated(subset=exact_key, keep="first").sum())
    out = out.drop_duplicates(subset=exact_key, keep="first").copy()
    after_exact_rows = len(out)

    timestamp_key = ["process", "mc_no", "occurred"]
    conflict_groups = (
        out.groupby(timestamp_key)
        .agg(
            n_rows=("mc_status", "size"),
            n_status=("mc_status", "nunique"),
            statuses=("mc_status", lambda x: sorted(set(x))),
        )
        .reset_index()
    )
    conflict_groups = conflict_groups[conflict_groups["n_status"] > 1].copy()
    conflict_group_count = len(conflict_groups)

    # 2. Simplified conflict removal logic
    conflict_rows_removed = 0
    if cfg.same_timestamp_conflict_policy == "drop_conflicts" and conflict_group_count > 0:
        conflict_keys = conflict_groups[timestamp_key].copy()
        conflict_keys["_conflict"] = 1

        out = out.merge(conflict_keys, on=timestamp_key, how="left")
        conflict_rows_removed = int(out["_conflict"].eq(1).sum())
        out = out[out["_conflict"].isna()].drop(columns=["_conflict"]).copy()

    after_rows = len(out)
    out = (
        out.sort_values(["process", "mc_no", "occurred", "mc_status"])
        .reset_index(drop=True)
    )

    report = {
        "before_rows": before_rows,
        "exact_duplicate_rows_removed": exact_duplicate_rows,
        "after_exact_dedup_rows": after_exact_rows,
        "same_timestamp_conflict_groups": int(conflict_group_count),
        "same_timestamp_conflict_rows_removed": int(conflict_rows_removed),
        "after_dedup_rows": after_rows,
        "total_rows_removed": before_rows - after_rows,
        "same_timestamp_conflict_policy": cfg.same_timestamp_conflict_policy,
    }

    return out, report, conflict_groups


df_clean, dedup_report, timestamp_conflict_groups = deduplicate_events(df_std, CFG)

print("P4-5 ready")
print(json.dumps(dedup_report, indent=2, ensure_ascii=False))

if len(timestamp_conflict_groups) > 0:
    print("\nExample same-timestamp conflict groups:")
    display(timestamp_conflict_groups.head(20))
else:
    print("\nNo same-timestamp status conflicts found.")

print("\nCleaned shape:", df_clean.shape)
display(df_clean.head())

P4-6 — Dataset profiling and validation report

In [ ]:
# ============================================================
# P4-6 — Dataset profiling and validation report
# ============================================================

def add_check(checks, name, passed, detail, severity="ERROR"):
    checks.append({
        "check": name,
        "severity": severity,
        "passed": bool(passed),
        "detail": detail,
    })


def build_dataset_profile(df: pd.DataFrame) -> dict:
    return {
        "rows": int(len(df)),
        "columns": list(df.columns),
        "process_count": int(df["process"].nunique()),
        "process_values": sorted(df["process"].dropna().unique().tolist()),
        "machine_count": int(df["mc_no"].nunique()),
        "status_count": int(df["mc_status"].nunique()),
        "time_min": df["occurred"].min(),
        "time_max": df["occurred"].max(),
        "time_span": df["occurred"].max() - df["occurred"].min(),
        "missing_occurred": int(df["occurred"].isna().sum()),
        "missing_mc_no": int(df["mc_no"].isna().sum()),
        "missing_mc_status": int(df["mc_status"].isna().sum()),
        "missing_process": int(df["process"].isna().sum()),
    }


profile = build_dataset_profile(df_clean)

status_profile = (
    df_clean["mc_status"]
    .value_counts(dropna=False)
    .rename_axis("mc_status")
    .reset_index(name="count")
)

status_profile["role"] = status_profile["mc_status"].map(STATUS_ROLE).fillna("unknown")
status_profile["percent"] = status_profile["count"] / len(df_clean) * 100
status_profile = status_profile[["mc_status", "role", "count", "percent"]]

machine_profile = (
    df_clean["mc_no"]
    .value_counts()
    .rename_axis("mc_no")
    .reset_index(name="rows")
)

process_profile = (
    df_clean["process"]
    .value_counts()
    .rename_axis("process")
    .reset_index(name="rows")
)

# Remaining duplicate/conflict checks after dedup
exact_key = ["process", "mc_no", "occurred", "mc_status"]
timestamp_key = ["process", "mc_no", "occurred"]

remaining_exact_duplicates = int(df_clean.duplicated(subset=exact_key).sum())

remaining_timestamp_conflicts = (
    df_clean.groupby(timestamp_key)["mc_status"]
    .nunique()
    .reset_index(name="n_status")
)
remaining_timestamp_conflicts = int((remaining_timestamp_conflicts["n_status"] > 1).sum())

target_row_count = int(df_clean["mc_status"].isin(TARGET_STATUSES).sum())
normal_row_count = int(df_clean["mc_status"].isin(NORMAL_STATUSES).sum())
ignored_row_count = int(df_clean["mc_status"].isin(IGNORED_STATUSES).sum())

checks = []

add_check(
    checks,
    "required_columns_exist",
    all(c in df_clean.columns for c in ["occurred", "mc_no", "mc_status", "process"]),
    "Required internal columns: occurred, mc_no, mc_status, process",
)

add_check(
    checks,
    "row_count_minimum",
    len(df_clean) >= CFG.min_rows,
    f"rows={len(df_clean):,}, minimum={CFG.min_rows:,}",
)

add_check(
    checks,
    "machine_count_minimum",
    df_clean["mc_no"].nunique() >= CFG.min_machines,
    f"machines={df_clean['mc_no'].nunique():,}, minimum={CFG.min_machines:,}",
)

add_check(
    checks,
    "single_process_dataset",
    CFG.allow_multiple_processes or df_clean["process"].nunique() == 1,
    f"process_count={df_clean['process'].nunique()}, processes={sorted(df_clean['process'].unique().tolist())}",
)

add_check(
    checks,
    "no_missing_occurred",
    df_clean["occurred"].notna().all(),
    f"missing_occurred={df_clean['occurred'].isna().sum():,}",
)

add_check(
    checks,
    "no_missing_mc_no",
    df_clean["mc_no"].notna().all() and (df_clean["mc_no"].astype(str).str.strip() != "").all(),
    f"missing_or_blank_mc_no={int(df_clean['mc_no'].isna().sum() + (df_clean['mc_no'].astype(str).str.strip() == '').sum()):,}",
)

add_check(
    checks,
    "no_missing_mc_status",
    df_clean["mc_status"].notna().all() and (df_clean["mc_status"].astype(str).str.strip() != "").all(),
    f"missing_or_blank_mc_status={int(df_clean['mc_status'].isna().sum() + (df_clean['mc_status'].astype(str).str.strip() == '').sum()):,}",
)

add_check(
    checks,
    "normal_statuses_defined",
    len(NORMAL_STATUSES) > 0,
    f"normal_statuses={sorted(NORMAL_STATUSES)}",
)

add_check(
    checks,
    "target_statuses_defined",
    len(TARGET_STATUSES) > 0,
    f"target_statuses={sorted(TARGET_STATUSES)}",
)

add_check(
    checks,
    "target_rows_minimum",
    target_row_count >= CFG.min_target_rows,
    f"target_rows={target_row_count:,}, minimum={CFG.min_target_rows:,}",
)

add_check(
    checks,
    "no_remaining_exact_duplicates",
    remaining_exact_duplicates == 0,
    f"remaining_exact_duplicates={remaining_exact_duplicates:,}",
)

add_check(
    checks,
    "no_remaining_same_timestamp_conflicts",
    remaining_timestamp_conflicts == 0,
    f"remaining_same_timestamp_conflict_groups={remaining_timestamp_conflicts:,}",
    severity="WARNING" if CFG.same_timestamp_conflict_policy == "keep" else "ERROR",
)

configured_normal_missing = sorted(NORMAL_STATUSES - set(ALL_STATUSES))
add_check(
    checks,
    "configured_normal_statuses_present",
    len(configured_normal_missing) == 0,
    f"configured normal statuses not present in data: {configured_normal_missing}",
    severity="WARNING",
)

validation_df = pd.DataFrame(checks)

failed_errors = validation_df[
    (validation_df["severity"] == "ERROR") & (~validation_df["passed"])
]

READY_FOR_BATCH_2 = len(failed_errors) == 0

print("=" * 100)
print("P4-6 — DATASET PROFILE")
print("=" * 100)

print(f"Pipeline:        {CFG.pipeline_name}")
print(f"Process ID:      {CFG.process_id}")
print(f"Data path:       {CFG.data_path}")
print(f"Rows:            {profile['rows']:,}")
print(f"Machines:        {profile['machine_count']:,}")
print(f"Statuses:        {profile['status_count']:,}")
print(f"Processes:       {profile['process_values']}")
print(f"Time range:      {profile['time_min']} -> {profile['time_max']}")
print(f"Time span:       {profile['time_span']}")

print("\nNormal statuses:")
print(sorted(NORMAL_STATUSES))

print("\nTarget statuses:")
print(sorted(TARGET_STATUSES))

print("\nIgnored statuses:")
print(sorted(IGNORED_STATUSES))

print("\nDeduplication report:")
print(json.dumps(dedup_report, indent=2, ensure_ascii=False))

print("\nStatus profile:")
display(status_profile)

print("\nProcess profile:")
display(process_profile)

print("\nMachine row-count summary:")
display(machine_profile["rows"].describe().to_frame("rows"))

print("\nTop 20 machines by row count:")
display(machine_profile.head(20))

print("\nValidation gates:")
display(validation_df)

print("=" * 100)
if READY_FOR_BATCH_2:
    print("READY_FOR_BATCH_2 = True")
    print("Batch 1 passed. Send this output to confirm before moving to Batch 2.")
else:
    print("READY_FOR_BATCH_2 = False")
    print("Batch 1 has failed ERROR gates. Fix before moving to Batch 2.")
print("=" * 100)

P4-7 — Build next-target-event labels

This cell creates the prediction target:
- next_target_time
- next_target_type
- y_time_sec

The target means:

For each machine event row, find the next future event where mc_status is in TARGET_STATUSES.

In [ ]:
# ============================================================
# P4-7 — Build next-target-event labels
# ============================================================

def build_next_target_labels(df: pd.DataFrame) -> pd.DataFrame:
    """Build next-target-event labels per process + machine.

    Target event definition:
      mc_status in TARGET_STATUSES

    For every row:
      next_target_time = next future timestamp where status is target
      next_target_type = status of that next target event
      y_time_sec       = seconds until next target event

    Important:
      If the current row is already a target event, the label is still the NEXT
      future target, not the current row itself.
    """
    out = df.copy()

    out["is_normal_status"] = out["mc_status"].isin(NORMAL_STATUSES).astype(int)
    out["is_target_event"] = out["mc_status"].isin(TARGET_STATUSES).astype(int)
    out["is_ignored_status"] = out["mc_status"].isin(IGNORED_STATUSES).astype(int)

    out = out.sort_values(["process", "mc_no", "occurred", "mc_status"]).reset_index(
        drop=True
    )

    def _label_one_machine(g: pd.DataFrame) -> pd.DataFrame:
        g = g.sort_values(["occurred", "mc_status"]).copy()

        target_time_at_row = g["occurred"].where(g["is_target_event"].eq(1))
        target_type_at_row = g["mc_status"].where(g["is_target_event"].eq(1))

        # shift(-1) makes the label strictly future.
        # bfill fills normal/ignored rows with the next future target.
        g["next_target_time"] = target_time_at_row.shift(-1).bfill()
        g["next_target_type"] = target_type_at_row.shift(-1).bfill()

        g["y_time_sec"] = (
            g["next_target_time"] - g["occurred"]
        ).dt.total_seconds()

        return g

    # --- FIX APPLIED HERE ---
    # 1. Remove group_keys=False so keys are preserved in the temporary MultiIndex
    # 2. Use reset_index(level=...) to safely pull them back into regular columns
    out = (
        out.groupby(["process", "mc_no"])
        .apply(_label_one_machine)
        .reset_index(level=["process", "mc_no"])
        .reset_index(drop=True)
    )

    return out


df_label_raw = build_next_target_labels(df_clean)

print("P4-7 ready")
print("Input rows:", f"{len(df_clean):,}")
print("Rows after raw labeling:", f"{len(df_label_raw):,}")

display(df_label_raw.head(20))

print("\nLabel columns:")
display(
    df_label_raw[
        [
            "process",
            "mc_no",
            "occurred",
            "mc_status",
            "is_normal_status",
            "is_target_event",
            "next_target_time",
            "next_target_type",
            "y_time_sec",
        ]
    ].head(30)
)

P4-8 — Remove invalid target rows

This removes rows where the machine has no future target event, or where the computed target gap is invalid.

In [ ]:
# ============================================================
# P4-8 — Remove invalid target rows
# ============================================================

def clean_labeled_rows(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    before = len(df)

    missing_next_time = int(df["next_target_time"].isna().sum())
    missing_next_type = int(df["next_target_type"].isna().sum())
    missing_y = int(df["y_time_sec"].isna().sum())
    non_positive_y = int((df["y_time_sec"] <= 0).sum())

    # Keep only rows with a valid strictly-future target.
    out = df.copy()
    out = out.dropna(subset=["next_target_time", "next_target_type", "y_time_sec"])
    out = out[out["y_time_sec"] > 0].copy()

    after = len(out)

    report = {
        "before_rows": before,
        "missing_next_target_time_rows": missing_next_time,
        "missing_next_target_type_rows": missing_next_type,
        "missing_y_time_rows": missing_y,
        "non_positive_y_time_rows": non_positive_y,
        "after_valid_label_rows": after,
        "rows_removed": before - after,
        "rows_removed_percent": (before - after) / before * 100 if before else np.nan,
    }

    out = out.sort_values(["process", "mc_no", "occurred", "mc_status"]).reset_index(drop=True)

    return out, report


df_labeled, label_cleaning_report = clean_labeled_rows(df_label_raw)

print("P4-8 ready")
print(json.dumps(label_cleaning_report, indent=2, ensure_ascii=False))

print("\nLabeled shape:", df_labeled.shape)

display(
    df_labeled[
        [
            "process",
            "mc_no",
            "occurred",
            "mc_status",
            "is_normal_status",
            "is_target_event",
            "next_target_time",
            "next_target_type",
            "y_time_sec",
        ]
    ].head(30)
)

P4-9 — Target distribution and label validation report

In [ ]:
# ============================================================
# P4-9 — Target distribution and label validation report
# ============================================================

def seconds_to_readable(sec):
    if pd.isna(sec):
        return np.nan

    sec = float(sec)

    if sec < 60:
        return f"{sec:.1f}s"
    elif sec < 3600:
        return f"{sec / 60:.2f}m"
    elif sec < 86400:
        return f"{sec / 3600:.2f}h"
    else:
        return f"{sec / 86400:.2f}d"


def target_distribution_table(y: pd.Series) -> pd.DataFrame:
    qs = {
        "min": y.min(),
        "p01": y.quantile(0.01),
        "p05": y.quantile(0.05),
        "p10": y.quantile(0.10),
        "p25": y.quantile(0.25),
        "p50_median": y.quantile(0.50),
        "p75": y.quantile(0.75),
        "p90": y.quantile(0.90),
        "p95": y.quantile(0.95),
        "p99": y.quantile(0.99),
        "max": y.max(),
        "mean": y.mean(),
        "std": y.std(),
    }

    out = pd.DataFrame({
        "metric": list(qs.keys()),
        "seconds": list(qs.values()),
    })

    out["readable"] = out["seconds"].apply(seconds_to_readable)
    return out


def build_label_validation_report(df: pd.DataFrame) -> pd.DataFrame:
    checks = []

    def add(name, passed, detail, severity="ERROR"):
        checks.append({
            "check": name,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        })

    add(
        "labeled_rows_exist",
        len(df) > 0,
        f"labeled_rows={len(df):,}",
    )

    add(
        "target_status_rows_exist",
        int(df["is_target_event"].sum()) >= CFG.min_target_rows,
        f"current_row_target_events={int(df['is_target_event'].sum()):,}, minimum={CFG.min_target_rows:,}",
    )

    add(
        "next_target_type_has_no_missing",
        df["next_target_type"].notna().all(),
        f"missing_next_target_type={df['next_target_type'].isna().sum():,}",
    )

    add(
        "next_target_time_has_no_missing",
        df["next_target_time"].notna().all(),
        f"missing_next_target_time={df['next_target_time'].isna().sum():,}",
    )

    add(
        "y_time_has_no_missing",
        df["y_time_sec"].notna().all(),
        f"missing_y_time_sec={df['y_time_sec'].isna().sum():,}",
    )

    add(
        "y_time_strictly_positive",
        (df["y_time_sec"] > 0).all(),
        f"non_positive_y_time_rows={int((df['y_time_sec'] <= 0).sum()):,}",
    )

    normal_in_next_target = sorted(set(df["next_target_type"].unique()) & NORMAL_STATUSES)
    add(
        "normal_statuses_not_in_next_target_type",
        len(normal_in_next_target) == 0,
        f"normal statuses found in next_target_type: {normal_in_next_target}",
    )

    ignored_in_next_target = sorted(set(df["next_target_type"].unique()) & IGNORED_STATUSES)
    add(
        "ignored_statuses_not_in_next_target_type",
        len(ignored_in_next_target) == 0,
        f"ignored statuses found in next_target_type: {ignored_in_next_target}",
    )

    unexpected_next_targets = sorted(set(df["next_target_type"].unique()) - TARGET_STATUSES)
    add(
        "next_target_type_only_contains_target_statuses",
        len(unexpected_next_targets) == 0,
        f"unexpected next target types: {unexpected_next_targets}",
    )

    add(
        "next_target_time_is_future",
        (df["next_target_time"] > df["occurred"]).all(),
        f"rows_where_next_target_not_future={int((df['next_target_time'] <= df['occurred']).sum()):,}",
    )

    add(
        "multiple_target_classes",
        df["next_target_type"].nunique() >= 2,
        f"next_target_type_classes={df['next_target_type'].nunique():,}",
        severity="WARNING",
    )

    return pd.DataFrame(checks)


target_time_dist = target_distribution_table(df_labeled["y_time_sec"])

next_target_type_dist = (
    df_labeled["next_target_type"]
    .value_counts(dropna=False)
    .rename_axis("next_target_type")
    .reset_index(name="count")
)

next_target_type_dist["percent"] = (
    next_target_type_dist["count"] / len(df_labeled) * 100
)

current_status_dist_after_label = (
    df_labeled["mc_status"]
    .value_counts(dropna=False)
    .rename_axis("current_mc_status")
    .reset_index(name="count")
)

current_status_dist_after_label["role"] = current_status_dist_after_label["current_mc_status"].map(STATUS_ROLE)
current_status_dist_after_label["percent"] = (
    current_status_dist_after_label["count"] / len(df_labeled) * 100
)

machine_label_summary = (
    df_labeled.groupby(["process", "mc_no"])
    .agg(
        rows=("mc_no", "size"),
        current_target_rows=("is_target_event", "sum"),
        first_time=("occurred", "min"),
        last_time=("occurred", "max"),
        median_y_sec=("y_time_sec", "median"),
        p90_y_sec=("y_time_sec", lambda x: x.quantile(0.90)),
        max_y_sec=("y_time_sec", "max"),
        next_target_classes=("next_target_type", "nunique"),
    )
    .reset_index()
)

machine_label_row_summary = machine_label_summary["rows"].describe().to_frame("labeled_rows")
machine_target_row_summary = machine_label_summary["current_target_rows"].describe().to_frame("current_target_rows")

low_labeled_machines = machine_label_summary[machine_label_summary["rows"] < 50].sort_values("rows")
zero_current_target_machines = machine_label_summary[machine_label_summary["current_target_rows"] == 0]

label_validation_df = build_label_validation_report(df_labeled)

failed_label_errors = label_validation_df[
    (label_validation_df["severity"] == "ERROR") & (~label_validation_df["passed"])
]

READY_FOR_BATCH_3 = len(failed_label_errors) == 0

print("=" * 100)
print("P4-9 — TARGET DISTRIBUTION AND LABEL VALIDATION REPORT")
print("=" * 100)

print(f"Rows before labeling:       {len(df_clean):,}")
print(f"Rows after valid labeling:  {len(df_labeled):,}")
print(f"Rows removed:               {label_cleaning_report['rows_removed']:,}")
print(f"Rows removed percent:       {label_cleaning_report['rows_removed_percent']:.4f}%")

print("\nNormal statuses:")
print(sorted(NORMAL_STATUSES))

print("\nTarget statuses:")
print(sorted(TARGET_STATUSES))

print("\nLabel cleaning report:")
print(json.dumps(label_cleaning_report, indent=2, ensure_ascii=False))

print("\nTarget time distribution:")
display(target_time_dist)

print("\nNext target type distribution:")
display(next_target_type_dist)

print("\nCurrent status distribution after valid labeling:")
display(current_status_dist_after_label)

print("\nMachine labeled row-count summary:")
display(machine_label_row_summary)

print("\nMachine current-target-row summary:")
display(machine_target_row_summary)

print("\nLowest 20 machines by labeled rows:")
display(low_labeled_machines.head(20))

print("\nMachines with zero current target rows after labeling:")
display(zero_current_target_machines.head(20))

print("\nLabel validation gates:")
display(label_validation_df)

print("=" * 100)
if READY_FOR_BATCH_3:
    print("READY_FOR_BATCH_3 = True")
    print("Batch 2 passed. Send this output to confirm before moving to Batch 3.")
else:
    print("READY_FOR_BATCH_3 = False")
    print("Batch 2 has failed ERROR gates. Fix before moving to Batch 3.")
print("=" * 100)

P4-10 — Build automatic feature set

In [ ]:
# ============================================================
# P4-10 — Build automatic feature set
# ============================================================

def safe_seconds_delta(a, b):
    return (a - b).dt.total_seconds()


def make_one_hot_from_token(series: pd.Series, prefix: str) -> pd.DataFrame:
    """
    Stable one-hot helper.
    Example:
      status_run
      status_mc_alarm
      prev1_status_mc_waitpart
    """
    d = pd.get_dummies(series.fillna("none").astype(str), prefix=prefix, dtype=np.int8)
    d.columns = [sanitize_feature_token(c) for c in d.columns]
    return d


def add_groupwise_last_time(out: pd.DataFrame, condition: pd.Series, col_name: str, include_current: bool = True):
    """
    Create last timestamp where condition was true within each process+machine group.

    include_current=True:
      current row is allowed to update last timestamp.
      Good for time_since_each_status.

    include_current=False:
      current row is excluded.
      Good for time_since_last_target_event before current row.
    """
    temp_time = out["occurred"].where(condition)

    if not include_current:
        temp_time = temp_time.groupby([out["process"], out["mc_no"]]).shift(1)

    out[col_name] = temp_time.groupby([out["process"], out["mc_no"]]).ffill()
    return out


def add_rolling_count_features(
    out: pd.DataFrame,
    value_col: str,
    prefix: str,
    windows_min: list,
) -> pd.DataFrame:
    """
    Time-based rolling sums per process+machine.

    value_col should be numeric 0/1.
    Rolling includes the current row, which is valid because current event is already observed.
    """
    base = out[["process", "mc_no", "occurred", value_col]].copy()
    base = base.sort_values(["process", "mc_no", "occurred"])

    for w in windows_min:
        feature_count = f"{prefix}_{w}m_count"
        feature_rate = f"{prefix}_{w}m_rate_per_min"

        rolled = (
            base
            .set_index("occurred")
            .groupby(["process", "mc_no"])[value_col]
            .rolling(f"{w}min", min_periods=1)
            .sum()
            .reset_index(level=[0, 1], drop=True)
        )

        out[feature_count] = rolled.reindex(out.set_index("occurred").index).to_numpy()

        # Since index reindex by occurred alone can be unsafe when timestamps repeat across machines,
        # recalculate by aligned sorted order instead.
        # The block above is kept readable, but the aligned implementation below is the one used.
        rolled_aligned = (
            base
            .set_index("occurred")
            .groupby(["process", "mc_no"])[value_col]
            .rolling(f"{w}min", min_periods=1)
            .sum()
            .reset_index()
        )

        out[feature_count] = rolled_aligned[value_col].to_numpy()
        out[feature_rate] = out[feature_count] / max(w, 1)

    return out


def build_automatic_features(df: pd.DataFrame, cfg: AutoTrainingConfig) -> tuple[pd.DataFrame, dict]:
    """
    Build process-agnostic effect-only features.

    Uses only:
      occurred
      mc_no
      process
      current/past mc_status
      current/past target-event markers

    Does not use:
      next_target_time
      next_target_type
      y_time_sec
    except those remain in dataframe as labels, not features.
    """
    out = df.copy()
    out = out.sort_values(["process", "mc_no", "occurred", "mc_status"]).reset_index(drop=True)

    group_keys = ["process", "mc_no"]
    g = out.groupby(group_keys, group_keys=False)

    # ------------------------------------------------------------
    # Base temporal features
    # ------------------------------------------------------------
    out["hour"] = out["occurred"].dt.hour.astype(np.int16)
    out["minute"] = out["occurred"].dt.minute.astype(np.int16)
    out["second"] = out["occurred"].dt.second.astype(np.int16)
    out["weekday"] = out["occurred"].dt.weekday.astype(np.int16)
    out["day"] = out["occurred"].dt.day.astype(np.int16)
    out["is_weekend"] = out["weekday"].isin([5, 6]).astype(np.int8)
    out["is_night"] = out["hour"].isin([0, 1, 2, 3, 4, 5]).astype(np.int8)

    out["hour_sin"] = np.sin(2 * np.pi * out["hour"] / 24)
    out["hour_cos"] = np.cos(2 * np.pi * out["hour"] / 24)
    out["minute_sin"] = np.sin(2 * np.pi * out["minute"] / 60)
    out["minute_cos"] = np.cos(2 * np.pi * out["minute"] / 60)
    out["weekday_sin"] = np.sin(2 * np.pi * out["weekday"] / 7)
    out["weekday_cos"] = np.cos(2 * np.pi * out["weekday"] / 7)

    # ------------------------------------------------------------
    # Machine sequence / age features
    # ------------------------------------------------------------
    out["machine_event_index"] = g.cumcount().astype(np.int32)

    first_seen = g["occurred"].transform("min")
    out["time_since_first_seen_machine_sec"] = safe_seconds_delta(out["occurred"], first_seen)

    # ------------------------------------------------------------
    # Previous event gap features
    # ------------------------------------------------------------
    out["prev_event_time"] = g["occurred"].shift(1)
    out["event_gap_sec"] = safe_seconds_delta(out["occurred"], out["prev_event_time"])

    for lag in [1, 2, 3]:
        out[f"event_gap_lag{lag}_sec"] = g["event_gap_sec"].shift(lag)

    out["event_gap_mean_5_sec"] = (
        out.groupby(group_keys)["event_gap_sec"]
        .rolling(5, min_periods=1)
        .mean()
        .reset_index(level=[0, 1], drop=True)
    )

    out["event_gap_std_5_sec"] = (
        out.groupby(group_keys)["event_gap_sec"]
        .rolling(5, min_periods=2)
        .std()
        .reset_index(level=[0, 1], drop=True)
    )

    out["event_gap_median_5_sec"] = (
        out.groupby(group_keys)["event_gap_sec"]
        .rolling(5, min_periods=1)
        .median()
        .reset_index(level=[0, 1], drop=True)
    )

    out["event_gap_min_5_sec"] = (
        out.groupby(group_keys)["event_gap_sec"]
        .rolling(5, min_periods=1)
        .min()
        .reset_index(level=[0, 1], drop=True)
    )

    out["event_gap_max_5_sec"] = (
        out.groupby(group_keys)["event_gap_sec"]
        .rolling(5, min_periods=1)
        .max()
        .reset_index(level=[0, 1], drop=True)
    )

    # ------------------------------------------------------------
    # Current status one-hot
    # ------------------------------------------------------------
    status_ohe = make_one_hot_from_token(out["mc_status_token"], "status")
    out = pd.concat([out, status_ohe], axis=1)

    status_cols = list(status_ohe.columns)

    # ------------------------------------------------------------
    # Previous status sequence features
    # ------------------------------------------------------------
    prev_status_cols = []

    for lag in [1, 2, 3]:
        prev_token = g["mc_status_token"].shift(lag).fillna("none")
        prev_ohe = make_one_hot_from_token(prev_token, f"prev{lag}_status")
        out = pd.concat([out, prev_ohe], axis=1)
        prev_status_cols.extend(prev_ohe.columns.tolist())

    # ------------------------------------------------------------
    # Last target-event type before current row
    # ------------------------------------------------------------
    target_token = out["mc_status_token"].where(out["is_target_event"].eq(1))
    out["last_target_type_token"] = (
        target_token
        .groupby([out["process"], out["mc_no"]])
        .shift(1)
        .groupby([out["process"], out["mc_no"]])
        .ffill()
        .fillna("none")
    )

    last_target_ohe = make_one_hot_from_token(out["last_target_type_token"], "last_target")
    out = pd.concat([out, last_target_ohe], axis=1)
    last_target_cols = list(last_target_ohe.columns)

    # Last target time before current row
    target_time = out["occurred"].where(out["is_target_event"].eq(1))
    out["last_target_time"] = (
        target_time
        .groupby([out["process"], out["mc_no"]])
        .shift(1)
        .groupby([out["process"], out["mc_no"]])
        .ffill()
    )

    out["time_since_last_target_sec"] = safe_seconds_delta(out["occurred"], out["last_target_time"])

    # Number of target events before current row
    out["target_event_count_prior"] = (
        out.groupby(group_keys)["is_target_event"]
        .cumsum()
        .groupby([out["process"], out["mc_no"]])
        .shift(1)
        .fillna(0)
        .astype(np.int32)
    )

    # ------------------------------------------------------------
    # Time since status change / consecutive same status
    # ------------------------------------------------------------
    out["prev_status_token"] = g["mc_status_token"].shift(1)

    out["status_changed"] = (
        out["mc_status_token"].ne(out["prev_status_token"])
        | out["prev_status_token"].isna()
    ).astype(np.int8)

    out["status_segment_id"] = (
        out.groupby(group_keys)["status_changed"]
        .cumsum()
        .astype(np.int32)
    )

    seg_keys = ["process", "mc_no", "status_segment_id"]

    out["status_segment_start_time"] = (
        out.groupby(seg_keys)["occurred"]
        .transform("min")
    )

    out["time_since_status_change_sec"] = safe_seconds_delta(
        out["occurred"],
        out["status_segment_start_time"]
    )

    out["consecutive_same_status_count"] = (
        out.groupby(seg_keys)
        .cumcount()
        .astype(np.int32)
    )

    # ------------------------------------------------------------
    # Time since each canonical status
    # ------------------------------------------------------------
    time_since_status_cols = []

    for status in ALL_STATUSES:
        token = sanitize_feature_token(status)
        col_last = f"_last_time_status_{token}"
        col_feat = f"time_since_status_{token}_sec"

        status_condition = out["mc_status"].eq(status)

        out[col_last] = (
            out["occurred"]
            .where(status_condition)
            .groupby([out["process"], out["mc_no"]])
            .ffill()
        )

        out[col_feat] = safe_seconds_delta(out["occurred"], out[col_last])
        time_since_status_cols.append(col_feat)

    # ------------------------------------------------------------
    # Rolling event / target / status-specific counts and rates
    # ------------------------------------------------------------
    out["_event_one"] = 1
    out["_target_one"] = out["is_target_event"].astype(int)

    rolling_cols = []

    base_sorted_index = out.index

    # Generic event and target rolling features
    for value_col, prefix in [
        ("_event_one", "events"),
        ("_target_one", "targets"),
    ]:
        base = out[["process", "mc_no", "occurred", value_col]].copy()

        for w in cfg.rolling_windows_min:
            count_col = f"{prefix}_{w}m_count"
            rate_col = f"{prefix}_{w}m_rate_per_min"

            rolled = (
                base
                .set_index("occurred")
                .groupby(["process", "mc_no"])[value_col]
                .rolling(f"{w}min", min_periods=1)
                .sum()
                .reset_index()
            )

            out[count_col] = rolled[value_col].to_numpy()
            out[rate_col] = out[count_col] / max(w, 1)

            rolling_cols.extend([count_col, rate_col])

    # Status-specific rolling features
    status_specific_rolling_cols = []

    for status in ALL_STATUSES:
        token = sanitize_feature_token(status)
        temp_col = f"_is_status_{token}"
        out[temp_col] = out["mc_status"].eq(status).astype(int)

        base = out[["process", "mc_no", "occurred", temp_col]].copy()

        for w in cfg.rolling_windows_min:
            count_col = f"status_{token}_{w}m_count"
            rate_col = f"status_{token}_{w}m_rate_per_min"

            rolled = (
                base
                .set_index("occurred")
                .groupby(["process", "mc_no"])[temp_col]
                .rolling(f"{w}min", min_periods=1)
                .sum()
                .reset_index()
            )

            out[count_col] = rolled[temp_col].to_numpy()
            out[rate_col] = out[count_col] / max(w, 1)

            status_specific_rolling_cols.extend([count_col, rate_col])

    rolling_cols.extend(status_specific_rolling_cols)

    # ------------------------------------------------------------
    # Machine-level behavioral features
    # These are initially calculated on all labeled data for feature construction.
    # In Batch 4, train-only maps/fill values will be fitted for final production-safe artifacts.
    # ------------------------------------------------------------
    machine_behavior = (
        out.groupby(["process", "mc_no"])
        .agg(
            mc_rows=("mc_no", "size"),
            mc_target_ratio=("is_target_event", "mean"),
            mc_median_event_gap_sec=("event_gap_sec", "median"),
            mc_median_y_sec=("y_time_sec", "median"),
            mc_p90_y_sec=("y_time_sec", lambda x: x.quantile(0.90)),
            mc_target_classes_seen=("next_target_type", "nunique"),
        )
        .reset_index()
    )

    out = out.merge(machine_behavior, on=["process", "mc_no"], how="left")

    machine_behavior_cols = [
        "mc_rows",
        "mc_target_ratio",
        "mc_median_event_gap_sec",
        "mc_median_y_sec",
        "mc_p90_y_sec",
        "mc_target_classes_seen",
    ]

    # ------------------------------------------------------------
    # Status-level behavioral features
    # ------------------------------------------------------------
    status_behavior = (
        out.groupby("mc_status")
        .agg(
            status_rows=("mc_status", "size"),
            status_target_ratio=("is_target_event", "mean"),
            status_median_y_sec=("y_time_sec", "median"),
            status_p90_y_sec=("y_time_sec", lambda x: x.quantile(0.90)),
        )
        .reset_index()
    )

    out = out.merge(status_behavior, on="mc_status", how="left")

    status_behavior_cols = [
        "status_rows",
        "status_target_ratio",
        "status_median_y_sec",
        "status_p90_y_sec",
    ]

    # ------------------------------------------------------------
    # Normalized timing ratio features
    # ------------------------------------------------------------
    eps = 1.0

    out["time_since_last_target_over_mc_median_y"] = (
        out["time_since_last_target_sec"] / (out["mc_median_y_sec"] + eps)
    )

    out["event_gap_over_mc_median_event_gap"] = (
        out["event_gap_sec"] / (out["mc_median_event_gap_sec"] + eps)
    )

    out["time_since_status_change_over_mc_median_y"] = (
        out["time_since_status_change_sec"] / (out["mc_median_y_sec"] + eps)
    )

    out["target_event_count_prior_over_machine_events"] = (
        out["target_event_count_prior"] / (out["machine_event_index"] + 1)
    )

    ratio_cols = [
        "time_since_last_target_over_mc_median_y",
        "event_gap_over_mc_median_event_gap",
        "time_since_status_change_over_mc_median_y",
        "target_event_count_prior_over_machine_events",
    ]

    # ------------------------------------------------------------
    # Cleanup helper timestamp columns, keep useful metadata columns
    # ------------------------------------------------------------
    helper_drop_cols = []

    helper_drop_cols.extend([c for c in out.columns if c.startswith("_last_time_status_")])
    helper_drop_cols.extend([c for c in out.columns if c.startswith("_is_status_")])
    helper_drop_cols.extend(["_event_one", "_target_one"])

    out = out.drop(columns=[c for c in helper_drop_cols if c in out.columns])

    feature_groups = {
        "base_time_cols": [
            "hour", "minute", "second", "weekday", "day", "is_weekend", "is_night",
            "hour_sin", "hour_cos", "minute_sin", "minute_cos", "weekday_sin", "weekday_cos",
        ],
        "machine_sequence_cols": [
            "machine_event_index",
            "time_since_first_seen_machine_sec",
        ],
        "event_gap_cols": [
            "event_gap_sec",
            "event_gap_lag1_sec",
            "event_gap_lag2_sec",
            "event_gap_lag3_sec",
            "event_gap_mean_5_sec",
            "event_gap_std_5_sec",
            "event_gap_median_5_sec",
            "event_gap_min_5_sec",
            "event_gap_max_5_sec",
        ],
        "status_cols": status_cols,
        "prev_status_cols": prev_status_cols,
        "last_target_cols": last_target_cols,
        "target_history_cols": [
            "time_since_last_target_sec",
            "target_event_count_prior",
        ],
        "status_duration_cols": [
            "status_changed",
            "time_since_status_change_sec",
            "consecutive_same_status_count",
        ],
        "time_since_status_cols": time_since_status_cols,
        "rolling_cols": rolling_cols,
        "machine_behavior_cols": machine_behavior_cols,
        "status_behavior_cols": status_behavior_cols,
        "ratio_cols": ratio_cols,
    }

    metadata = {
        "feature_groups": feature_groups,
        "all_statuses": ALL_STATUSES,
        "normal_statuses": sorted(NORMAL_STATUSES),
        "target_statuses": sorted(TARGET_STATUSES),
        "rolling_windows_min": cfg.rolling_windows_min,
    }

    return out, metadata


df_feat, feature_metadata = build_automatic_features(df_labeled, CFG)

print("P4-10 ready")
print("Feature dataframe shape:", df_feat.shape)

print("\nFeature groups:")
for group_name, cols in feature_metadata["feature_groups"].items():
    print(f"- {group_name}: {len(cols)} columns")

display(df_feat.head())

P4-11 — Build final feature list

In [ ]:
# ============================================================
# P4-11 — Build final feature list
# ============================================================

LABEL_COLS = [
    "next_target_time",
    "next_target_type",
    "y_time_sec",
]

IDENTITY_COLS = [
    "process",
    "mc_no",
    "occurred",
    "mc_status_raw",
    "mc_status",
    "mc_status_token",
]

HELPER_COLS = [
    "is_normal_status",
    "is_target_event",
    "is_ignored_status",
    "prev_event_time",
    "prev_status_token",
    "last_target_time",
    "last_target_type_token",
    "status_segment_id",
    "status_segment_start_time",
]

EXCLUDE_FROM_FEATURES = set(LABEL_COLS + IDENTITY_COLS + HELPER_COLS)

candidate_feature_cols = []

for group_name, cols in feature_metadata["feature_groups"].items():
    for c in cols:
        if c in df_feat.columns and c not in EXCLUDE_FROM_FEATURES:
            candidate_feature_cols.append(c)

# Preserve order while removing duplicates
seen = set()
feature_cols = []
for c in candidate_feature_cols:
    if c not in seen:
        feature_cols.append(c)
        seen.add(c)

# Keep only numeric features
non_numeric_features = [
    c for c in feature_cols
    if not pd.api.types.is_numeric_dtype(df_feat[c])
]

if non_numeric_features:
    print("Dropping non-numeric feature columns:")
    print(non_numeric_features)
    feature_cols = [c for c in feature_cols if c not in non_numeric_features]

# Defensive leakage blocklist
LEAKAGE_PATTERNS = [
    "next_target",
    "y_time",
    "target_time",
]

leakage_suspect_cols = [
    c for c in feature_cols
    if any(p in c.lower() for p in LEAKAGE_PATTERNS)
]

# last_target is allowed because it means previous target type, not future next target.
leakage_suspect_cols = [
    c for c in leakage_suspect_cols
    if not c.startswith("last_target")
]

if leakage_suspect_cols:
    raise ValueError(f"Potential leakage columns found in feature_cols: {leakage_suspect_cols}")

feature_group_summary = []

for group_name, cols in feature_metadata["feature_groups"].items():
    included = [c for c in cols if c in feature_cols]
    feature_group_summary.append({
        "feature_group": group_name,
        "n_features": len(included),
        "examples": included[:8],
    })

feature_group_summary_df = pd.DataFrame(feature_group_summary)

print("P4-11 ready")
print("Total feature columns:", len(feature_cols))

print("\nFeature group summary:")
display(feature_group_summary_df)

print("\nFirst 100 feature columns:")
for i, c in enumerate(feature_cols[:100], start=1):
    print(f"{i:03d}. {c}")

if len(feature_cols) > 100:
    print(f"... {len(feature_cols) - 100} more features")

P4-12 — Feature validation report

In [ ]:
# ============================================================
# P4-12 — Feature validation report
# ============================================================

def build_feature_validation_report(df: pd.DataFrame, feature_cols: list) -> pd.DataFrame:
    checks = []

    def add(name, passed, detail, severity="ERROR"):
        checks.append({
            "check": name,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        })

    add(
        "feature_columns_exist",
        all(c in df.columns for c in feature_cols),
        f"missing_feature_cols={[c for c in feature_cols if c not in df.columns][:20]}",
    )

    add(
        "feature_count_reasonable",
        20 <= len(feature_cols) <= 500,
        f"feature_count={len(feature_cols)}",
        severity="WARNING",
    )

    non_numeric = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(df[c])]
    add(
        "all_features_numeric",
        len(non_numeric) == 0,
        f"non_numeric_features={non_numeric[:20]}",
    )

    duplicate_cols = pd.Series(feature_cols).duplicated().sum()
    add(
        "no_duplicate_feature_names",
        duplicate_cols == 0,
        f"duplicate_feature_name_count={int(duplicate_cols)}",
    )

    leakage_bad = [
        c for c in feature_cols
        if (
            "next_target" in c.lower()
            or c.lower() == "y_time_sec"
            or c.lower().startswith("y_")
        )
    ]

    add(
        "no_obvious_future_label_leakage_features",
        len(leakage_bad) == 0,
        f"leakage_suspect_features={leakage_bad[:20]}",
    )

    all_nan_features = [c for c in feature_cols if df[c].isna().all()]
    add(
        "no_all_nan_features",
        len(all_nan_features) == 0,
        f"all_nan_features={all_nan_features[:20]}",
    )

    constant_features = []
    for c in feature_cols:
        nunique = df[c].nunique(dropna=True)
        if nunique <= 1:
            constant_features.append(c)

    add(
        "constant_features_not_too_many",
        len(constant_features) <= max(10, int(0.10 * len(feature_cols))),
        f"constant_feature_count={len(constant_features)}, examples={constant_features[:20]}",
        severity="WARNING",
    )

    inf_counts = np.isinf(df[feature_cols].select_dtypes(include=[np.number])).sum()
    inf_features = inf_counts[inf_counts > 0].index.tolist()

    add(
        "no_infinite_values",
        len(inf_features) == 0,
        f"features_with_inf={inf_features[:20]}",
    )

    return pd.DataFrame(checks)


missingness_df = pd.DataFrame({
    "feature": feature_cols,
    "missing_count": [int(df_feat[c].isna().sum()) for c in feature_cols],
    "missing_percent": [float(df_feat[c].isna().mean() * 100) for c in feature_cols],
    "nunique": [int(df_feat[c].nunique(dropna=True)) for c in feature_cols],
    "dtype": [str(df_feat[c].dtype) for c in feature_cols],
})

missingness_df = missingness_df.sort_values(
    ["missing_percent", "missing_count"],
    ascending=False
).reset_index(drop=True)

feature_validation_df = build_feature_validation_report(df_feat, feature_cols)

failed_feature_errors = feature_validation_df[
    (feature_validation_df["severity"] == "ERROR") & (~feature_validation_df["passed"])
]

READY_FOR_BATCH_4 = len(failed_feature_errors) == 0

print("=" * 100)
print("P4-12 — FEATURE VALIDATION REPORT")
print("=" * 100)

print(f"Feature dataframe rows:      {len(df_feat):,}")
print(f"Feature dataframe columns:   {df_feat.shape[1]:,}")
print(f"Final feature count:         {len(feature_cols):,}")

print("\nFeature group summary:")
display(feature_group_summary_df)

print("\nTop 30 features by missingness:")
display(missingness_df.head(30))

print("\nFeatures with >50% missing:")
display(missingness_df[missingness_df["missing_percent"] > 50].head(50))

print("\nFeature validation gates:")
display(feature_validation_df)

print("\nSample feature rows:")
sample_cols = IDENTITY_COLS + LABEL_COLS + feature_cols[:25]
sample_cols = [c for c in sample_cols if c in df_feat.columns]
display(df_feat[sample_cols].head(10))

print("=" * 100)
if READY_FOR_BATCH_4:
    print("READY_FOR_BATCH_4 = True")
    print("Batch 3 passed. Send this output to confirm before moving to Batch 4.")
else:
    print("READY_FOR_BATCH_4 = False")
    print("Batch 3 has failed ERROR gates. Fix before moving to Batch 4.")
print("=" * 100)

P4-13 — Per-machine chronological train/val/test split

In [ ]:
# ============================================================
# P4-13 — Per-machine chronological train/val/test split
# ============================================================

MIN_ROWS_FULL_SPLIT = 30

def per_machine_chrono_split(
    df: pd.DataFrame,
    cfg: AutoTrainingConfig,
    min_rows_full_split: int = 30,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Chronological split inside each process + machine.

    Machines with too few rows are kept in train only.
    This prevents tiny machines from producing meaningless val/test slices.
    """
    df_sorted = df.sort_values(["process", "mc_no", "occurred", "mc_status"]).copy()

    split_labels = pd.Series(index=df_sorted.index, dtype="object")
    reports = []

    for (process, mc_no), g in df_sorted.groupby(["process", "mc_no"], sort=False):
        g = g.sort_values(["occurred", "mc_status"])
        idx = g.index.to_numpy()
        n = len(g)

        if n < min_rows_full_split:
            train_idx = idx
            val_idx = np.array([], dtype=idx.dtype)
            test_idx = np.array([], dtype=idx.dtype)
            split_reason = "low_support_train_only"
        else:
            n_train = int(n * cfg.train_frac)
            n_val = int(n * cfg.val_frac)

            # Ensure each split has at least one row for sufficiently large machines.
            n_train = max(n_train, 1)
            n_val = max(n_val, 1)
            n_test = n - n_train - n_val

            if n_test < 1:
                n_test = 1
                n_train = max(n - n_val - n_test, 1)

            train_idx = idx[:n_train]
            val_idx = idx[n_train:n_train + n_val]
            test_idx = idx[n_train + n_val:]

            split_reason = "full_split"

        split_labels.loc[train_idx] = "train"
        split_labels.loc[val_idx] = "val"
        split_labels.loc[test_idx] = "test"

        reports.append({
            "process": process,
            "mc_no": mc_no,
            "total_rows": n,
            "train_rows": len(train_idx),
            "val_rows": len(val_idx),
            "test_rows": len(test_idx),
            "split_reason": split_reason,
            "first_time": g["occurred"].min(),
            "last_time": g["occurred"].max(),
        })

    df_sorted["_split"] = split_labels.values

    train_df = df_sorted[df_sorted["_split"].eq("train")].drop(columns=["_split"]).copy()
    val_df = df_sorted[df_sorted["_split"].eq("val")].drop(columns=["_split"]).copy()
    test_df = df_sorted[df_sorted["_split"].eq("test")].drop(columns=["_split"]).copy()

    split_report_df = pd.DataFrame(reports)

    return train_df, val_df, test_df, split_report_df


train_df, val_df, test_df, split_report_df = per_machine_chrono_split(
    df_feat,
    CFG,
    min_rows_full_split=MIN_ROWS_FULL_SPLIT,
)

split_summary = pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "machines": [
        train_df["mc_no"].nunique(),
        val_df["mc_no"].nunique(),
        test_df["mc_no"].nunique(),
    ],
    "time_min": [
        train_df["occurred"].min(),
        val_df["occurred"].min(),
        test_df["occurred"].min(),
    ],
    "time_max": [
        train_df["occurred"].max(),
        val_df["occurred"].max(),
        test_df["occurred"].max(),
    ],
})

split_summary["row_percent"] = split_summary["rows"] / len(df_feat) * 100

machine_split_reason_summary = (
    split_report_df["split_reason"]
    .value_counts()
    .rename_axis("split_reason")
    .reset_index(name="machine_count")
)

print("P4-13 ready")
print(f"MIN_ROWS_FULL_SPLIT = {MIN_ROWS_FULL_SPLIT}")

print("\nSplit summary:")
display(split_summary)

print("\nMachine split reason summary:")
display(machine_split_reason_summary)

print("\nMachine split row summary:")
display(
    split_report_df[
        ["total_rows", "train_rows", "val_rows", "test_rows"]
    ].describe()
)

print("\nLowest 20 machines by total rows:")
display(split_report_df.sort_values("total_rows").head(20))

P4-14 — Train-only target clipping

In [ ]:
# ============================================================
# P4-14 — Train-only target clipping
# ============================================================

clip_max = float(train_df["y_time_sec"].quantile(CFG.target_clip_quantile))

def apply_target_clip(df: pd.DataFrame, clip_max: float) -> pd.DataFrame:
    out = df.copy()
    out["y_time_sec_clip"] = out["y_time_sec"].clip(lower=0, upper=clip_max)
    out["y_time_was_clipped"] = (out["y_time_sec"] > clip_max).astype(np.int8)
    return out


train_df = apply_target_clip(train_df, clip_max)
val_df = apply_target_clip(val_df, clip_max)
test_df = apply_target_clip(test_df, clip_max)

target_clip_report = pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "clip_max_sec": [clip_max, clip_max, clip_max],
    "clip_max_readable": [seconds_to_readable(clip_max)] * 3,
    "clipped_rows": [
        int(train_df["y_time_was_clipped"].sum()),
        int(val_df["y_time_was_clipped"].sum()),
        int(test_df["y_time_was_clipped"].sum()),
    ],
    "clipped_percent": [
        train_df["y_time_was_clipped"].mean() * 100,
        val_df["y_time_was_clipped"].mean() * 100,
        test_df["y_time_was_clipped"].mean() * 100,
    ],
    "raw_median_sec": [
        train_df["y_time_sec"].median(),
        val_df["y_time_sec"].median(),
        test_df["y_time_sec"].median(),
    ],
    "raw_p90_sec": [
        train_df["y_time_sec"].quantile(0.90),
        val_df["y_time_sec"].quantile(0.90),
        test_df["y_time_sec"].quantile(0.90),
    ],
    "raw_p99_sec": [
        train_df["y_time_sec"].quantile(0.99),
        val_df["y_time_sec"].quantile(0.99),
        test_df["y_time_sec"].quantile(0.99),
    ],
    "raw_max_sec": [
        train_df["y_time_sec"].max(),
        val_df["y_time_sec"].max(),
        test_df["y_time_sec"].max(),
    ],
})

print("P4-14 ready")
print(f"Train-only clip quantile: {CFG.target_clip_quantile}")
print(f"clip_max = {clip_max:.3f} sec ({seconds_to_readable(clip_max)})")

display(target_clip_report)

P4-15 — Overwrite behavior features using train-only maps

This is the leakage-safety fix.

In [ ]:
# ============================================================
# P4-15 — Overwrite behavior features using train-only maps
# ============================================================

MACHINE_BEHAVIOR_COLS = feature_metadata["feature_groups"]["machine_behavior_cols"]
STATUS_BEHAVIOR_COLS = feature_metadata["feature_groups"]["status_behavior_cols"]
RATIO_COLS = feature_metadata["feature_groups"]["ratio_cols"]

def fit_train_only_behavior_maps(train: pd.DataFrame) -> dict:
    """
    Fit machine/status behavior maps using train split only.
    This prevents validation/test leakage.
    """
    machine_behavior_train = (
        train.groupby(["process", "mc_no"])
        .agg(
            mc_rows=("mc_no", "size"),
            mc_target_ratio=("is_target_event", "mean"),
            mc_median_event_gap_sec=("event_gap_sec", "median"),
            mc_median_y_sec=("y_time_sec_clip", "median"),
            mc_p90_y_sec=("y_time_sec_clip", lambda x: x.quantile(0.90)),
            mc_target_classes_seen=("next_target_type", "nunique"),
        )
        .reset_index()
    )

    status_behavior_train = (
        train.groupby("mc_status")
        .agg(
            status_rows=("mc_status", "size"),
            status_target_ratio=("is_target_event", "mean"),
            status_median_y_sec=("y_time_sec_clip", "median"),
            status_p90_y_sec=("y_time_sec_clip", lambda x: x.quantile(0.90)),
        )
        .reset_index()
    )

    global_machine_behavior = {
        "mc_rows": float(train.groupby(["process", "mc_no"]).size().median()),
        "mc_target_ratio": float(train["is_target_event"].mean()),
        "mc_median_event_gap_sec": float(train["event_gap_sec"].median()),
        "mc_median_y_sec": float(train["y_time_sec_clip"].median()),
        "mc_p90_y_sec": float(train["y_time_sec_clip"].quantile(0.90)),
        "mc_target_classes_seen": float(train["next_target_type"].nunique()),
    }

    global_status_behavior = {
        "status_rows": float(train["mc_status"].value_counts().median()),
        "status_target_ratio": float(train["is_target_event"].mean()),
        "status_median_y_sec": float(train["y_time_sec_clip"].median()),
        "status_p90_y_sec": float(train["y_time_sec_clip"].quantile(0.90)),
    }

    return {
        "machine_behavior_train": machine_behavior_train,
        "status_behavior_train": status_behavior_train,
        "global_machine_behavior": global_machine_behavior,
        "global_status_behavior": global_status_behavior,
    }


def apply_train_only_behavior_maps(df: pd.DataFrame, maps: dict) -> pd.DataFrame:
    out = df.copy()

    # Remove previously full-data-derived behavior columns before re-merging.
    out = out.drop(
        columns=[c for c in MACHINE_BEHAVIOR_COLS + STATUS_BEHAVIOR_COLS if c in out.columns],
        errors="ignore",
    )

    machine_map = maps["machine_behavior_train"]
    status_map = maps["status_behavior_train"]

    out = out.merge(machine_map, on=["process", "mc_no"], how="left")
    out = out.merge(status_map, on="mc_status", how="left")

    for c, v in maps["global_machine_behavior"].items():
        if c in out.columns:
            out[c] = out[c].fillna(v)

    for c, v in maps["global_status_behavior"].items():
        if c in out.columns:
            out[c] = out[c].fillna(v)

    # Recompute ratio features using train-only behavior maps.
    eps = 1.0

    out["time_since_last_target_over_mc_median_y"] = (
        out["time_since_last_target_sec"] / (out["mc_median_y_sec"] + eps)
    )

    out["event_gap_over_mc_median_event_gap"] = (
        out["event_gap_sec"] / (out["mc_median_event_gap_sec"] + eps)
    )

    out["time_since_status_change_over_mc_median_y"] = (
        out["time_since_status_change_sec"] / (out["mc_median_y_sec"] + eps)
    )

    out["target_event_count_prior_over_machine_events"] = (
        out["target_event_count_prior"] / (out["machine_event_index"] + 1)
    )

    return out


behavior_maps = fit_train_only_behavior_maps(train_df)

train_df = apply_train_only_behavior_maps(train_df, behavior_maps)
val_df = apply_train_only_behavior_maps(val_df, behavior_maps)
test_df = apply_train_only_behavior_maps(test_df, behavior_maps)

behavior_map_report = {
    "machine_behavior_map_rows": int(len(behavior_maps["machine_behavior_train"])),
    "status_behavior_map_rows": int(len(behavior_maps["status_behavior_train"])),
    "global_machine_behavior": behavior_maps["global_machine_behavior"],
    "global_status_behavior": behavior_maps["global_status_behavior"],
}

print("P4-15 ready")
print(json.dumps(behavior_map_report, indent=2, ensure_ascii=False))

print("\nTrain-only machine behavior map sample:")
display(behavior_maps["machine_behavior_train"].head(10))

print("\nTrain-only status behavior map:")
display(behavior_maps["status_behavior_train"])

P4-16 — Train-only feature fill values

In [ ]:
# ============================================================
# P4-16 — Train-only feature fill values
# ============================================================

def build_feature_fill_values(train: pd.DataFrame, feature_cols: list) -> dict:
    """
    Build train-only fill values.

    Strategy:
      - time_since_status_*: use train p99 as "not seen recently / unseen yet" sentinel
      - time_since_last_target_sec: use train p99
      - event-gap columns: use median
      - all others: use median
      - if no valid value exists: use 0.0
    """
    fill_values = {}

    train_numeric = train[feature_cols].replace([np.inf, -np.inf], np.nan)

    for c in feature_cols:
        s = train_numeric[c]

        if c.startswith("time_since_status_") or c == "time_since_last_target_sec":
            val = s.quantile(0.99)
        else:
            val = s.median()

        if pd.isna(val) or np.isinf(val):
            val = 0.0

        fill_values[c] = float(val)

    return fill_values


feature_fill_values = build_feature_fill_values(train_df, feature_cols)

feature_fill_df = pd.DataFrame({
    "feature": list(feature_fill_values.keys()),
    "fill_value": list(feature_fill_values.values()),
})

feature_fill_df["fill_readable_if_sec"] = np.where(
    feature_fill_df["feature"].str.endswith("_sec"),
    feature_fill_df["fill_value"].apply(seconds_to_readable),
    "",
)

print("P4-16 ready")
print(f"Feature fill values fitted from train only: {len(feature_fill_values):,}")

print("\nTop 30 largest fill values:")
display(feature_fill_df.sort_values("fill_value", ascending=False).head(30))

print("\nFill values for time_since_status features:")
display(feature_fill_df[feature_fill_df["feature"].str.startswith("time_since_status_")])

P4-17 — Machine-balanced sample weights and final train matrices

In [ ]:
# ============================================================
# P4-17 — Machine-balanced sample weights and final train matrices
# ============================================================

def build_model_matrix(df: pd.DataFrame, feature_cols: list, fill_values: dict) -> pd.DataFrame:
    X = df[feature_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(fill_values)

    # Final safety check
    bad_cols = X.columns[X.isna().any()].tolist()
    if bad_cols:
        raise ValueError(f"NaN still found after filling. Example columns: {bad_cols[:20]}")

    inf_counts = np.isinf(X.select_dtypes(include=[np.number])).sum()
    inf_cols = inf_counts[inf_counts > 0].index.tolist()
    if inf_cols:
        raise ValueError(f"Inf still found after filling. Example columns: {inf_cols[:20]}")

    return X.astype(np.float32)


def make_machine_balanced_weights(df: pd.DataFrame) -> np.ndarray:
    """
    Down-weight high-volume machines, up-weight low-volume machines.
    Normalized to mean 1.0.
    """
    counts = df.groupby(["process", "mc_no"]).size()

    keys = list(zip(df["process"], df["mc_no"]))
    raw_w = np.array([1.0 / math.sqrt(counts.loc[k]) for k in keys], dtype=np.float64)

    raw_w = raw_w / raw_w.mean()
    return raw_w.astype(np.float32)


X_train = build_model_matrix(train_df, feature_cols, feature_fill_values)
X_val = build_model_matrix(val_df, feature_cols, feature_fill_values)
X_test = build_model_matrix(test_df, feature_cols, feature_fill_values)

y_train_raw = train_df["y_time_sec"].to_numpy(dtype=np.float64)
y_val_raw = val_df["y_time_sec"].to_numpy(dtype=np.float64)
y_test_raw = test_df["y_time_sec"].to_numpy(dtype=np.float64)

y_train_clip = train_df["y_time_sec_clip"].to_numpy(dtype=np.float64)
y_val_clip = val_df["y_time_sec_clip"].to_numpy(dtype=np.float64)
y_test_clip = test_df["y_time_sec_clip"].to_numpy(dtype=np.float64)

y_type_train = train_df["next_target_type"].astype(str).to_numpy()
y_type_val = val_df["next_target_type"].astype(str).to_numpy()
y_type_test = test_df["next_target_type"].astype(str).to_numpy()

sample_weight_train_eta = make_machine_balanced_weights(train_df)

matrix_report = pd.DataFrame({
    "matrix": [
        "X_train", "X_val", "X_test",
        "y_train_raw", "y_val_raw", "y_test_raw",
        "y_train_clip", "y_val_clip", "y_test_clip",
        "y_type_train", "y_type_val", "y_type_test",
        "sample_weight_train_eta",
    ],
    "shape_or_length": [
        str(X_train.shape), str(X_val.shape), str(X_test.shape),
        len(y_train_raw), len(y_val_raw), len(y_test_raw),
        len(y_train_clip), len(y_val_clip), len(y_test_clip),
        len(y_type_train), len(y_type_val), len(y_type_test),
        len(sample_weight_train_eta),
    ],
})

weight_summary = pd.Series(sample_weight_train_eta).describe().to_frame("sample_weight_train_eta")

split_target_summary = pd.DataFrame({
    "split": ["train", "val", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "machines": [
        train_df["mc_no"].nunique(),
        val_df["mc_no"].nunique(),
        test_df["mc_no"].nunique(),
    ],
    "target_median_sec": [
        train_df["y_time_sec"].median(),
        val_df["y_time_sec"].median(),
        test_df["y_time_sec"].median(),
    ],
    "target_p90_sec": [
        train_df["y_time_sec"].quantile(0.90),
        val_df["y_time_sec"].quantile(0.90),
        test_df["y_time_sec"].quantile(0.90),
    ],
    "target_p99_sec": [
        train_df["y_time_sec"].quantile(0.99),
        val_df["y_time_sec"].quantile(0.99),
        test_df["y_time_sec"].quantile(0.99),
    ],
    "target_max_sec": [
        train_df["y_time_sec"].max(),
        val_df["y_time_sec"].max(),
        test_df["y_time_sec"].max(),
    ],
})

# Check feature-column consistency
same_columns_train_val = list(X_train.columns) == list(X_val.columns)
same_columns_train_test = list(X_train.columns) == list(X_test.columns)

final_checks = []

def add_final_check(name, passed, detail, severity="ERROR"):
    final_checks.append({
        "check": name,
        "severity": severity,
        "passed": bool(passed),
        "detail": detail,
    })

add_final_check(
    "x_matrices_have_same_columns",
    same_columns_train_val and same_columns_train_test,
    f"train_val_same={same_columns_train_val}, train_test_same={same_columns_train_test}",
)

add_final_check(
    "x_train_rows_match_y_train",
    len(X_train) == len(y_train_clip) == len(y_type_train) == len(sample_weight_train_eta),
    f"X_train={len(X_train)}, y_train_clip={len(y_train_clip)}, y_type_train={len(y_type_train)}, weights={len(sample_weight_train_eta)}",
)

add_final_check(
    "x_val_rows_match_y_val",
    len(X_val) == len(y_val_clip) == len(y_type_val),
    f"X_val={len(X_val)}, y_val_clip={len(y_val_clip)}, y_type_val={len(y_type_val)}",
)

add_final_check(
    "x_test_rows_match_y_test",
    len(X_test) == len(y_test_clip) == len(y_type_test),
    f"X_test={len(X_test)}, y_test_clip={len(y_test_clip)}, y_type_test={len(y_type_test)}",
)

add_final_check(
    "no_nan_in_model_matrices",
    not X_train.isna().any().any() and not X_val.isna().any().any() and not X_test.isna().any().any(),
    "NaN check passed for X_train/X_val/X_test",
)

add_final_check(
    "feature_count_matches",
    X_train.shape[1] == len(feature_cols),
    f"X_train_features={X_train.shape[1]}, feature_cols={len(feature_cols)}",
)

add_final_check(
    "clip_max_is_positive",
    clip_max > 0,
    f"clip_max={clip_max:.3f}",
)

add_final_check(
    "train_val_test_non_empty",
    len(train_df) > 0 and len(val_df) > 0 and len(test_df) > 0,
    f"train={len(train_df):,}, val={len(val_df):,}, test={len(test_df):,}",
)

final_matrix_validation_df = pd.DataFrame(final_checks)

failed_matrix_errors = final_matrix_validation_df[
    (final_matrix_validation_df["severity"] == "ERROR") & (~final_matrix_validation_df["passed"])
]

READY_FOR_BATCH_5 = len(failed_matrix_errors) == 0

print("=" * 100)
print("P4-17 — FINAL TRAIN/VAL/TEST MATRIX REPORT")
print("=" * 100)

print(f"Final feature count: {len(feature_cols):,}")
print(f"clip_max: {clip_max:.3f} sec ({seconds_to_readable(clip_max)})")

print("\nSplit summary:")
display(split_summary)

print("\nMachine split reason summary:")
display(machine_split_reason_summary)

print("\nTarget clipping report:")
display(target_clip_report)

print("\nSplit target summary:")
display(split_target_summary)

print("\nMatrix report:")
display(matrix_report)

print("\nSample weight summary:")
display(weight_summary)

print("\nBehavior map report:")
print(json.dumps(behavior_map_report, indent=2, ensure_ascii=False))

print("\nFinal matrix validation gates:")
display(final_matrix_validation_df)

print("=" * 100)
if READY_FOR_BATCH_5:
    print("READY_FOR_BATCH_5 = True")
    print("Batch 4 passed. Send this output to confirm before moving to Batch 5.")
else:
    print("READY_FOR_BATCH_5 = False")
    print("Batch 4 has failed ERROR gates. Fix before moving to Batch 5.")
print("=" * 100)

P4-17.1 — Cap machine-balanced sample weights

In [ ]:
# ============================================================
# P4-17.1 — Cap machine-balanced sample weights
# ============================================================

SAMPLE_WEIGHT_CAP = 5.0

sample_weight_train_eta_uncapped = sample_weight_train_eta.copy()

sample_weight_train_eta = np.minimum(
    sample_weight_train_eta,
    SAMPLE_WEIGHT_CAP
).astype(np.float32)

# Renormalize to mean 1.0
sample_weight_train_eta = (
    sample_weight_train_eta / sample_weight_train_eta.mean()
).astype(np.float32)

weight_cap_report = pd.DataFrame({
    "metric": [
        "cap_value",
        "uncapped_min",
        "uncapped_mean",
        "uncapped_p50",
        "uncapped_p95",
        "uncapped_p99",
        "uncapped_max",
        "capped_min",
        "capped_mean",
        "capped_p50",
        "capped_p95",
        "capped_p99",
        "capped_max",
        "rows_affected_by_cap",
        "rows_affected_by_cap_percent",
    ],
    "value": [
        SAMPLE_WEIGHT_CAP,
        float(np.min(sample_weight_train_eta_uncapped)),
        float(np.mean(sample_weight_train_eta_uncapped)),
        float(np.quantile(sample_weight_train_eta_uncapped, 0.50)),
        float(np.quantile(sample_weight_train_eta_uncapped, 0.95)),
        float(np.quantile(sample_weight_train_eta_uncapped, 0.99)),
        float(np.max(sample_weight_train_eta_uncapped)),
        float(np.min(sample_weight_train_eta)),
        float(np.mean(sample_weight_train_eta)),
        float(np.quantile(sample_weight_train_eta, 0.50)),
        float(np.quantile(sample_weight_train_eta, 0.95)),
        float(np.quantile(sample_weight_train_eta, 0.99)),
        float(np.max(sample_weight_train_eta)),
        int((sample_weight_train_eta_uncapped > SAMPLE_WEIGHT_CAP).sum()),
        float((sample_weight_train_eta_uncapped > SAMPLE_WEIGHT_CAP).mean() * 100),
    ],
})

print("P4-17.1 ready")
display(weight_cap_report)

P4-18 — Build ETA and next-type baselines

In [ ]:
# ============================================================
# P4-18 — Build ETA and next-type baselines
# ============================================================

def mode_or_none(x):
    m = x.mode(dropna=True)
    if len(m) == 0:
        return None
    return str(m.iloc[0])


def fit_eta_baselines(train: pd.DataFrame) -> dict:
    """
    Fit simple train-only ETA baselines.

    Uses clipped target for robust baseline fitting.
    Evaluated later against raw y_time_sec.
    """
    eta_maps = {}

    eta_maps["global_median"] = float(train["y_time_sec_clip"].median())

    eta_maps["status_median"] = (
        train.groupby("mc_status")["y_time_sec_clip"]
        .median()
        .astype(float)
        .to_dict()
    )

    eta_maps["machine_median"] = (
        train.groupby(["process", "mc_no"])["y_time_sec_clip"]
        .median()
        .astype(float)
        .to_dict()
    )

    eta_maps["machine_status_median"] = (
        train.groupby(["process", "mc_no", "mc_status"])["y_time_sec_clip"]
        .median()
        .astype(float)
        .to_dict()
    )

    return eta_maps


def predict_eta_baselines(df: pd.DataFrame, eta_maps: dict) -> pd.DataFrame:
    """
    Predict ETA using multiple baseline strategies.
    """
    global_median = eta_maps["global_median"]

    status_map = eta_maps["status_median"]
    machine_map = eta_maps["machine_median"]
    machine_status_map = eta_maps["machine_status_median"]

    pred_global = np.full(len(df), global_median, dtype=np.float64)

    pred_status = np.array([
        status_map.get(status, global_median)
        for status in df["mc_status"]
    ], dtype=np.float64)

    pred_machine = np.array([
        machine_map.get((process, mc_no), global_median)
        for process, mc_no in zip(df["process"], df["mc_no"])
    ], dtype=np.float64)

    pred_machine_status = np.array([
        machine_status_map.get(
            (process, mc_no, status),
            machine_map.get((process, mc_no), status_map.get(status, global_median))
        )
        for process, mc_no, status in zip(df["process"], df["mc_no"], df["mc_status"])
    ], dtype=np.float64)

    return pd.DataFrame({
        "global_median_eta": pred_global,
        "status_median_eta": pred_status,
        "machine_median_eta": pred_machine,
        "machine_status_median_eta": pred_machine_status,
    }, index=df.index)


def fit_type_baselines(train: pd.DataFrame) -> dict:
    """
    Fit train-only next-target-type baselines.
    """
    type_maps = {}

    global_mode = mode_or_none(train["next_target_type"])
    type_maps["global_mode"] = global_mode

    type_maps["status_mode"] = (
        train.groupby("mc_status")["next_target_type"]
        .agg(mode_or_none)
        .to_dict()
    )

    type_maps["machine_mode"] = (
        train.groupby(["process", "mc_no"])["next_target_type"]
        .agg(mode_or_none)
        .to_dict()
    )

    type_maps["machine_status_mode"] = (
        train.groupby(["process", "mc_no", "mc_status"])["next_target_type"]
        .agg(mode_or_none)
        .to_dict()
    )

    return type_maps


def predict_type_baselines(df: pd.DataFrame, type_maps: dict) -> pd.DataFrame:
    global_mode = type_maps["global_mode"]

    status_map = type_maps["status_mode"]
    machine_map = type_maps["machine_mode"]
    machine_status_map = type_maps["machine_status_mode"]

    pred_global = np.array([global_mode] * len(df), dtype=object)

    pred_status = np.array([
        status_map.get(status, global_mode)
        for status in df["mc_status"]
    ], dtype=object)

    pred_machine = np.array([
        machine_map.get((process, mc_no), global_mode)
        for process, mc_no in zip(df["process"], df["mc_no"])
    ], dtype=object)

    pred_machine_status = np.array([
        machine_status_map.get(
            (process, mc_no, status),
            machine_map.get((process, mc_no), status_map.get(status, global_mode))
        )
        for process, mc_no, status in zip(df["process"], df["mc_no"], df["mc_status"])
    ], dtype=object)

    return pd.DataFrame({
        "global_mode_type": pred_global,
        "status_mode_type": pred_status,
        "machine_mode_type": pred_machine,
        "machine_status_mode_type": pred_machine_status,
    }, index=df.index)


eta_baseline_maps = fit_eta_baselines(train_df)
type_baseline_maps = fit_type_baselines(train_df)

eta_baseline_pred_val = predict_eta_baselines(val_df, eta_baseline_maps)
eta_baseline_pred_test = predict_eta_baselines(test_df, eta_baseline_maps)

type_baseline_pred_val = predict_type_baselines(val_df, type_baseline_maps)
type_baseline_pred_test = predict_type_baselines(test_df, type_baseline_maps)

print("P4-18 ready")

print("\nETA baseline global median:")
print(f"{eta_baseline_maps['global_median']:.3f} sec ({seconds_to_readable(eta_baseline_maps['global_median'])})")

print("\nType baseline global mode:")
print(type_baseline_maps["global_mode"])

print("\nETA baseline predictions sample:")
display(eta_baseline_pred_val.head())

print("\nType baseline predictions sample:")
display(type_baseline_pred_val.head())

P4-19 — Baseline evaluation report

In [ ]:
# ============================================================
# P4-19 — Baseline evaluation report
# ============================================================

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

def smape(y_true, y_pred, eps=1e-9):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / np.maximum(denom, eps)) * 100


def hit_rate(y_true, y_pred, tolerance_sec):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return np.mean(np.abs(y_true - y_pred) <= tolerance_sec) * 100


def evaluate_eta_predictions(y_true_raw, y_pred, name, split):
    y_true_raw = np.asarray(y_true_raw, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    rmse = math.sqrt(mean_squared_error(y_true_raw, y_pred))

    return {
        "split": split,
        "baseline": name,
        "MAE_sec": mean_absolute_error(y_true_raw, y_pred),
        "RMSE_sec": rmse,
        "MedAE_sec": median_absolute_error(y_true_raw, y_pred),
        "SMAPE_percent": smape(y_true_raw, y_pred),
        "Hit_2m_percent": hit_rate(y_true_raw, y_pred, 120),
        "Hit_5m_percent": hit_rate(y_true_raw, y_pred, 300),
        "Hit_10m_percent": hit_rate(y_true_raw, y_pred, 600),
        "pred_median_sec": float(np.median(y_pred)),
        "pred_p90_sec": float(np.quantile(y_pred, 0.90)),
        "actual_median_sec": float(np.median(y_true_raw)),
        "actual_p90_sec": float(np.quantile(y_true_raw, 0.90)),
    }


def evaluate_type_predictions(y_true, y_pred, name, split):
    y_true = np.asarray(y_true).astype(str)
    y_pred = np.asarray(y_pred).astype(str)

    return {
        "split": split,
        "baseline": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }


eta_baseline_rows = []

for split_name, df_split, y_true_raw, pred_df in [
    ("val", val_df, y_val_raw, eta_baseline_pred_val),
    ("test_reference", test_df, y_test_raw, eta_baseline_pred_test),
]:
    for col in pred_df.columns:
        eta_baseline_rows.append(
            evaluate_eta_predictions(
                y_true_raw=y_true_raw,
                y_pred=pred_df[col].to_numpy(),
                name=col,
                split=split_name,
            )
        )

eta_baseline_report = pd.DataFrame(eta_baseline_rows)

eta_baseline_report_sorted = eta_baseline_report.sort_values(
    ["split", "MedAE_sec", "Hit_5m_percent"],
    ascending=[True, True, False],
).reset_index(drop=True)

type_baseline_rows = []

for split_name, y_true_type, pred_df in [
    ("val", y_type_val, type_baseline_pred_val),
    ("test_reference", y_type_test, type_baseline_pred_test),
]:
    for col in pred_df.columns:
        type_baseline_rows.append(
            evaluate_type_predictions(
                y_true=y_true_type,
                y_pred=pred_df[col].to_numpy(),
                name=col,
                split=split_name,
            )
        )

type_baseline_report = pd.DataFrame(type_baseline_rows)

type_baseline_report_sorted = type_baseline_report.sort_values(
    ["split", "weighted_f1", "accuracy"],
    ascending=[True, False, False],
).reset_index(drop=True)

# Best validation baselines
best_eta_baseline_name = (
    eta_baseline_report[eta_baseline_report["split"].eq("val")]
    .sort_values(["MedAE_sec", "Hit_5m_percent"], ascending=[True, False])
    .iloc[0]["baseline"]
)

best_type_baseline_name = (
    type_baseline_report[type_baseline_report["split"].eq("val")]
    .sort_values(["weighted_f1", "accuracy"], ascending=[False, False])
    .iloc[0]["baseline"]
)

best_eta_baseline_val = eta_baseline_report[
    (eta_baseline_report["split"].eq("val")) &
    (eta_baseline_report["baseline"].eq(best_eta_baseline_name))
].iloc[0].to_dict()

best_type_baseline_val = type_baseline_report[
    (type_baseline_report["split"].eq("val")) &
    (type_baseline_report["baseline"].eq(best_type_baseline_name))
].iloc[0].to_dict()

# Detailed classification report for best validation type baseline
best_type_val_pred = type_baseline_pred_val[best_type_baseline_name].astype(str).to_numpy()

type_classes_sorted = sorted(set(y_type_val) | set(best_type_val_pred))

best_type_classification_report_dict = classification_report(
    y_type_val,
    best_type_val_pred,
    labels=type_classes_sorted,
    output_dict=True,
    zero_division=0,
)

best_type_classification_report_df = pd.DataFrame(
    best_type_classification_report_dict
).T

best_type_confusion = pd.DataFrame(
    confusion_matrix(y_type_val, best_type_val_pred, labels=type_classes_sorted),
    index=[f"true_{c}" for c in type_classes_sorted],
    columns=[f"pred_{c}" for c in type_classes_sorted],
)

baseline_summary = {
    "best_eta_baseline_val": best_eta_baseline_name,
    "best_eta_baseline_val_MedAE_sec": float(best_eta_baseline_val["MedAE_sec"]),
    "best_eta_baseline_val_Hit_5m_percent": float(best_eta_baseline_val["Hit_5m_percent"]),
    "best_eta_baseline_val_Hit_10m_percent": float(best_eta_baseline_val["Hit_10m_percent"]),
    "best_type_baseline_val": best_type_baseline_name,
    "best_type_baseline_val_accuracy": float(best_type_baseline_val["accuracy"]),
    "best_type_baseline_val_macro_f1": float(best_type_baseline_val["macro_f1"]),
    "best_type_baseline_val_weighted_f1": float(best_type_baseline_val["weighted_f1"]),
}

print("=" * 100)
print("P4-19 — BASELINE EVALUATION REPORT")
print("=" * 100)

print("\nBaseline summary:")
print(json.dumps(baseline_summary, indent=2, ensure_ascii=False))

print("\nETA baseline report:")
display(eta_baseline_report_sorted)

print("\nNext-type baseline report:")
display(type_baseline_report_sorted)

print(f"\nBest validation next-type baseline: {best_type_baseline_name}")
print("\nClassification report for best validation next-type baseline:")
display(best_type_classification_report_df)

print("\nConfusion matrix for best validation next-type baseline:")
display(best_type_confusion)

print("=" * 100)
print("READY_FOR_BATCH_6 = True")
print("Batch 5 complete. Send this output before moving to Batch 6.")
print("=" * 100)

P4-20 — Train candidate P50 ETA models

In [ ]:
# ============================================================
# P4-20 — Train candidate P50 ETA models
# ============================================================

import lightgbm as lgb
from lightgbm import LGBMRegressor

def pinball_loss(y_true, y_pred, alpha):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    diff = y_true - y_pred
    return np.mean(np.maximum(alpha * diff, (alpha - 1) * diff))


def transform_target(y, mode):
    y = np.asarray(y, dtype=np.float64)

    if mode == "raw":
        return y

    if mode == "log1p":
        return np.log1p(y)

    raise ValueError(f"Unknown target transform mode: {mode}")


def inverse_transform_pred(pred, mode):
    pred = np.asarray(pred, dtype=np.float64)

    if mode == "raw":
        return pred

    if mode == "log1p":
        return np.expm1(pred)

    raise ValueError(f"Unknown target transform mode: {mode}")


def clean_eta_pred(pred, clip_upper=None):
    pred = np.asarray(pred, dtype=np.float64)
    pred = np.nan_to_num(pred, nan=0.0, posinf=clip_upper if clip_upper is not None else 1e9, neginf=0.0)
    pred = np.maximum(pred, 1.0)

    if clip_upper is not None:
        pred = np.minimum(pred, clip_upper)

    return pred


def train_lgbm_quantile_candidate(
    alpha,
    target_mode,
    params,
    X_train,
    y_train_clip,
    X_val,
    y_val_clip,
    sample_weight_train,
    model_name,
):
    y_train_model = transform_target(y_train_clip, target_mode)
    y_val_model = transform_target(y_val_clip, target_mode)

    model = LGBMRegressor(
        objective="quantile",
        alpha=alpha,
        device_type="gpu",
        max_bin=63,             # 🚀 Tailored for OpenCL kernel hardware speed
        gpu_use_dp=False,       # 🚀 Prevents the CPU from sending heavy float64 data
        n_jobs=8,               # 🚀 Exact physical core match for your i7-10700
        n_estimators=params.get("n_estimators", 3000),
        learning_rate=params.get("learning_rate", 0.03),
        num_leaves=params.get("num_leaves", 63),
        max_depth=params.get("max_depth", -1),
        min_child_samples=params.get("min_child_samples", 100),
        subsample=params.get("subsample", 0.85),
        subsample_freq=params.get("subsample_freq", 1),
        colsample_bytree=params.get("colsample_bytree", 0.85),
        reg_alpha=params.get("reg_alpha", 0.2),
        reg_lambda=params.get("reg_lambda", 0.5),
        random_state=SEED,
        # n_jobs=-1,
        # force_col_wise=True,
        verbosity=-1,
    )

    model.fit(
        X_train,
        y_train_model,
        sample_weight=sample_weight_train,
        eval_set=[(X_val, y_val_model)],
        eval_metric="quantile",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=params.get("early_stopping_rounds", 150),
                verbose=False,
            ),
            lgb.log_evaluation(period=200),
        ],
    )

    pred_val_model_space = model.predict(X_val, num_iteration=model.best_iteration_)
    pred_val_sec = inverse_transform_pred(pred_val_model_space, target_mode)
    pred_val_sec = clean_eta_pred(pred_val_sec, clip_upper=clip_max)

    return {
        "model_name": model_name,
        "model": model,
        "alpha": alpha,
        "target_mode": target_mode,
        "params": params,
        "best_iteration": int(model.best_iteration_ or params.get("n_estimators", 3000)),
        "pred_val_sec": pred_val_sec,
    }


ETA_CANDIDATE_PARAMS = [
    {
        "candidate_id": "balanced_63",
        "learning_rate": 0.03,
        "n_estimators": 4000,
        "num_leaves": 63,
        "min_child_samples": 100,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.2,
        "reg_lambda": 0.5,
        "early_stopping_rounds": 150,
    },
    {
        "candidate_id": "regularized_63",
        "learning_rate": 0.03,
        "n_estimators": 4000,
        "num_leaves": 63,
        "min_child_samples": 200,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.5,
        "reg_lambda": 1.0,
        "early_stopping_rounds": 150,
    },
    {
        "candidate_id": "compact_31",
        "learning_rate": 0.04,
        "n_estimators": 3000,
        "num_leaves": 31,
        "min_child_samples": 100,
        "subsample": 0.90,
        "subsample_freq": 1,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.2,
        "reg_lambda": 0.5,
        "early_stopping_rounds": 120,
    },
]

TARGET_MODES = ["raw", "log1p"]

p50_candidates = []

for params in ETA_CANDIDATE_PARAMS:
    for target_mode in TARGET_MODES:
        candidate_id = params["candidate_id"]
        model_name = f"p50_{candidate_id}_{target_mode}"

        print("=" * 100)
        print(f"Training {model_name}")
        print("=" * 100)

        result = train_lgbm_quantile_candidate(
            alpha=0.50,
            target_mode=target_mode,
            params=params,
            X_train=X_train,
            y_train_clip=y_train_clip,
            X_val=X_val,
            y_val_clip=y_val_clip,
            sample_weight_train=sample_weight_train_eta,
            model_name=model_name,
        )

        pred = result["pred_val_sec"]

        row = evaluate_eta_predictions(
            y_true_raw=y_val_raw,
            y_pred=pred,
            name=model_name,
            split="val",
        )

        row.update({
            "alpha": 0.50,
            "target_mode": target_mode,
            "candidate_id": candidate_id,
            "best_iteration": result["best_iteration"],
            "pinball50_clip": pinball_loss(y_val_clip, pred, 0.50),
        })

        result["metrics"] = row
        p50_candidates.append(result)

p50_candidate_report = pd.DataFrame([c["metrics"] for c in p50_candidates])

p50_candidate_report_sorted = p50_candidate_report.sort_values(
    ["MedAE_sec", "Hit_5m_percent", "MAE_sec"],
    ascending=[True, False, True],
).reset_index(drop=True)

print("P4-20 ready")
print("\nP50 candidate report:")
display(p50_candidate_report_sorted)

P4-21 — Train candidate P90 ETA models

In [ ]:
# ============================================================
# P4-21 — Train candidate P90 ETA models
# ============================================================

p90_candidates = []

for params in ETA_CANDIDATE_PARAMS:
    for target_mode in TARGET_MODES:
        candidate_id = params["candidate_id"]
        model_name = f"p90_{candidate_id}_{target_mode}"

        print("=" * 100)
        print(f"Training {model_name}")
        print("=" * 100)

        result = train_lgbm_quantile_candidate(
            alpha=0.90,
            target_mode=target_mode,
            params=params,
            X_train=X_train,
            y_train_clip=y_train_clip,
            X_val=X_val,
            y_val_clip=y_val_clip,
            sample_weight_train=sample_weight_train_eta,
            model_name=model_name,
        )

        pred = result["pred_val_sec"]

        coverage_raw = float(np.mean(y_val_raw <= pred) * 100)
        coverage_clip = float(np.mean(y_val_clip <= pred) * 100)

        row = evaluate_eta_predictions(
            y_true_raw=y_val_raw,
            y_pred=pred,
            name=model_name,
            split="val",
        )

        row.update({
            "alpha": 0.90,
            "target_mode": target_mode,
            "candidate_id": candidate_id,
            "best_iteration": result["best_iteration"],
            "pinball90_clip": pinball_loss(y_val_clip, pred, 0.90),
            "P90_coverage_raw_percent": coverage_raw,
            "P90_coverage_clip_percent": coverage_clip,
            "coverage_error_from_90_raw": abs(coverage_raw - 90.0),
            "coverage_error_from_90_clip": abs(coverage_clip - 90.0),
            "pred_p50_sec": float(np.quantile(pred, 0.50)),
            "pred_p90_sec": float(np.quantile(pred, 0.90)),
            "pred_p95_sec": float(np.quantile(pred, 0.95)),
        })

        result["metrics"] = row
        p90_candidates.append(result)

p90_candidate_report = pd.DataFrame([c["metrics"] for c in p90_candidates])

p90_candidate_report_sorted = p90_candidate_report.sort_values(
    ["coverage_error_from_90_raw", "pinball90_clip", "pred_p90_sec"],
    ascending=[True, True, True],
).reset_index(drop=True)

print("P4-21 ready")
print("\nP90 candidate report:")
display(p90_candidate_report_sorted)

P4-22 — Select best ETA models by validation metrics

In [ ]:
# ============================================================
# P4-22 — Select best ETA models by validation metrics
# ============================================================

# -----------------------------
# Select P50
# Priority:
#   1. lowest MedAE
#   2. highest Hit@5m
#   3. highest Hit@10m
#   4. lowest MAE
# -----------------------------

best_p50_row = (
    p50_candidate_report
    .sort_values(
        ["MedAE_sec", "Hit_5m_percent", "Hit_10m_percent", "MAE_sec"],
        ascending=[True, False, False, True],
    )
    .iloc[0]
)

best_p50_name = best_p50_row["baseline"]

best_p50_candidate = next(
    c for c in p50_candidates
    if c["model_name"] == best_p50_name
)

lgbm_p50_final = best_p50_candidate["model"]
p50_target_mode_final = best_p50_candidate["target_mode"]
p50_pred_val = best_p50_candidate["pred_val_sec"]


# -----------------------------
# Select P90
# Priority:
#   1. raw coverage closest to 90%
#   2. lower pinball90
#   3. smaller median P90 prediction
# -----------------------------

best_p90_row = (
    p90_candidate_report
    .sort_values(
        ["coverage_error_from_90_raw", "pinball90_clip", "pred_p50_sec"],
        ascending=[True, True, True],
    )
    .iloc[0]
)

best_p90_name = best_p90_row["baseline"]

best_p90_candidate = next(
    c for c in p90_candidates
    if c["model_name"] == best_p90_name
)

lgbm_p90_final = best_p90_candidate["model"]
p90_target_mode_final = best_p90_candidate["target_mode"]
p90_pred_val_raw = best_p90_candidate["pred_val_sec"]

# Enforce P90 >= P50 for validation preview.
# More formal calibration/post-processing happens in Batch 7.
p90_pred_val = np.maximum(p90_pred_val_raw, p50_pred_val + 1.0)

p90_coverage_raw_after_floor = float(np.mean(y_val_raw <= p90_pred_val) * 100)
p90_coverage_clip_after_floor = float(np.mean(y_val_clip <= p90_pred_val) * 100)

selected_eta_summary = {
    "selected_p50_model": best_p50_name,
    "selected_p50_target_mode": p50_target_mode_final,
    "selected_p50_best_iteration": int(best_p50_candidate["best_iteration"]),
    "selected_p50_val_MedAE_sec": float(best_p50_row["MedAE_sec"]),
    "selected_p50_val_Hit_5m_percent": float(best_p50_row["Hit_5m_percent"]),
    "selected_p50_val_Hit_10m_percent": float(best_p50_row["Hit_10m_percent"]),

    "selected_p90_model": best_p90_name,
    "selected_p90_target_mode": p90_target_mode_final,
    "selected_p90_best_iteration": int(best_p90_candidate["best_iteration"]),
    "selected_p90_val_raw_coverage_percent_before_floor": float(best_p90_row["P90_coverage_raw_percent"]),
    "selected_p90_val_clip_coverage_percent_before_floor": float(best_p90_row["P90_coverage_clip_percent"]),
    "selected_p90_val_raw_coverage_percent_after_floor": p90_coverage_raw_after_floor,
    "selected_p90_val_clip_coverage_percent_after_floor": p90_coverage_clip_after_floor,

    "best_eta_baseline_val": best_eta_baseline_name,
    "best_eta_baseline_val_MedAE_sec": float(best_eta_baseline_val["MedAE_sec"]),
    "best_eta_baseline_val_Hit_5m_percent": float(best_eta_baseline_val["Hit_5m_percent"]),
    "best_eta_baseline_val_Hit_10m_percent": float(best_eta_baseline_val["Hit_10m_percent"]),
}

p50_vs_baseline = {
    "p50_MedAE_improvement_sec": float(best_eta_baseline_val["MedAE_sec"] - best_p50_row["MedAE_sec"]),
    "p50_Hit5m_improvement_point": float(best_p50_row["Hit_5m_percent"] - best_eta_baseline_val["Hit_5m_percent"]),
    "p50_Hit10m_improvement_point": float(best_p50_row["Hit_10m_percent"] - best_eta_baseline_val["Hit_10m_percent"]),
}

print("=" * 100)
print("P4-22 — SELECTED ETA MODELS")
print("=" * 100)

print("\nSelected ETA summary:")
print(json.dumps(selected_eta_summary, indent=2, ensure_ascii=False))

print("\nP50 vs best ETA baseline:")
print(json.dumps(p50_vs_baseline, indent=2, ensure_ascii=False))

print("\nSorted P50 candidates:")
display(p50_candidate_report_sorted)

print("\nSorted P90 candidates:")
display(p90_candidate_report_sorted)

print("=" * 100)
print("P4-22 ready")
print("=" * 100)

P4-23 — ETA validation report

In [ ]:
# ============================================================
# P4-23 — ETA validation report
# ============================================================

def evaluate_p90_interval(y_true_raw, y_true_clip, p50_pred, p90_pred, split="val"):
    p50_pred = np.asarray(p50_pred, dtype=np.float64)
    p90_pred = np.asarray(p90_pred, dtype=np.float64)

    interval_width = p90_pred - p50_pred

    return {
        "split": split,
        "P90_coverage_raw_percent": float(np.mean(y_true_raw <= p90_pred) * 100),
        "P90_coverage_clip_percent": float(np.mean(y_true_clip <= p90_pred) * 100),
        "P90_under_coverage_raw_percent": float(np.mean(y_true_raw > p90_pred) * 100),
        "P90_median_width_sec": float(np.median(interval_width)),
        "P90_p90_width_sec": float(np.quantile(interval_width, 0.90)),
        "P90_mean_width_sec": float(np.mean(interval_width)),
        "P90_invalid_width_rows": int(np.sum(interval_width < 0)),
        "P50_pred_median_sec": float(np.median(p50_pred)),
        "P90_pred_median_sec": float(np.median(p90_pred)),
        "actual_median_sec": float(np.median(y_true_raw)),
        "actual_p90_sec": float(np.quantile(y_true_raw, 0.90)),
    }


p50_val_metrics = evaluate_eta_predictions(
    y_true_raw=y_val_raw,
    y_pred=p50_pred_val,
    name="lgbm_p50_selected",
    split="val",
)

p50_val_metrics.update({
    "pinball50_clip": pinball_loss(y_val_clip, p50_pred_val, 0.50),
})

p90_val_metrics = evaluate_p90_interval(
    y_true_raw=y_val_raw,
    y_true_clip=y_val_clip,
    p50_pred=p50_pred_val,
    p90_pred=p90_pred_val,
    split="val",
)

p90_val_metrics.update({
    "pinball90_clip": pinball_loss(y_val_clip, p90_pred_val, 0.90),
})

eta_validation_summary = {
    "selected_p50_model": best_p50_name,
    "selected_p90_model": best_p90_name,

    "p50_val_MedAE_sec": float(p50_val_metrics["MedAE_sec"]),
    "p50_val_MAE_sec": float(p50_val_metrics["MAE_sec"]),
    "p50_val_RMSE_sec": float(p50_val_metrics["RMSE_sec"]),
    "p50_val_Hit_2m_percent": float(p50_val_metrics["Hit_2m_percent"]),
    "p50_val_Hit_5m_percent": float(p50_val_metrics["Hit_5m_percent"]),
    "p50_val_Hit_10m_percent": float(p50_val_metrics["Hit_10m_percent"]),

    "p90_val_coverage_raw_percent": float(p90_val_metrics["P90_coverage_raw_percent"]),
    "p90_val_coverage_clip_percent": float(p90_val_metrics["P90_coverage_clip_percent"]),
    "p90_val_median_width_sec": float(p90_val_metrics["P90_median_width_sec"]),
    "p90_val_p90_width_sec": float(p90_val_metrics["P90_p90_width_sec"]),

    "baseline_val_MedAE_sec": float(best_eta_baseline_val["MedAE_sec"]),
    "baseline_val_Hit_5m_percent": float(best_eta_baseline_val["Hit_5m_percent"]),
    "baseline_val_Hit_10m_percent": float(best_eta_baseline_val["Hit_10m_percent"]),
}

eta_validation_comparison = pd.DataFrame([
    {
        "model": "best_baseline_" + best_eta_baseline_name,
        "MedAE_sec": best_eta_baseline_val["MedAE_sec"],
        "MAE_sec": best_eta_baseline_val["MAE_sec"],
        "RMSE_sec": best_eta_baseline_val["RMSE_sec"],
        "Hit_2m_percent": best_eta_baseline_val["Hit_2m_percent"],
        "Hit_5m_percent": best_eta_baseline_val["Hit_5m_percent"],
        "Hit_10m_percent": best_eta_baseline_val["Hit_10m_percent"],
        "SMAPE_percent": best_eta_baseline_val["SMAPE_percent"],
    },
    {
        "model": "lgbm_p50_selected",
        "MedAE_sec": p50_val_metrics["MedAE_sec"],
        "MAE_sec": p50_val_metrics["MAE_sec"],
        "RMSE_sec": p50_val_metrics["RMSE_sec"],
        "Hit_2m_percent": p50_val_metrics["Hit_2m_percent"],
        "Hit_5m_percent": p50_val_metrics["Hit_5m_percent"],
        "Hit_10m_percent": p50_val_metrics["Hit_10m_percent"],
        "SMAPE_percent": p50_val_metrics["SMAPE_percent"],
    },
])

# Per-current-status validation preview
val_eta_preview = val_df[
    ["process", "mc_no", "occurred", "mc_status", "next_target_type", "y_time_sec", "y_time_sec_clip"]
].copy()

val_eta_preview["p50_pred_sec"] = p50_pred_val
val_eta_preview["p90_pred_sec"] = p90_pred_val
val_eta_preview["abs_error_p50_sec"] = np.abs(val_eta_preview["y_time_sec"] - val_eta_preview["p50_pred_sec"])
val_eta_preview["p90_covered_raw"] = val_eta_preview["y_time_sec"] <= val_eta_preview["p90_pred_sec"]

eta_by_current_status = (
    val_eta_preview
    .groupby("mc_status")
    .agg(
        rows=("mc_status", "size"),
        actual_median_sec=("y_time_sec", "median"),
        actual_p90_sec=("y_time_sec", lambda x: x.quantile(0.90)),
        p50_pred_median_sec=("p50_pred_sec", "median"),
        MedAE_sec=("abs_error_p50_sec", "median"),
        MAE_sec=("abs_error_p50_sec", "mean"),
        Hit_5m_percent=("abs_error_p50_sec", lambda x: (x <= 300).mean() * 100),
        Hit_10m_percent=("abs_error_p50_sec", lambda x: (x <= 600).mean() * 100),
        P90_coverage_raw_percent=("p90_covered_raw", lambda x: x.mean() * 100),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

eta_by_next_target_type = (
    val_eta_preview
    .groupby("next_target_type")
    .agg(
        rows=("next_target_type", "size"),
        actual_median_sec=("y_time_sec", "median"),
        actual_p90_sec=("y_time_sec", lambda x: x.quantile(0.90)),
        p50_pred_median_sec=("p50_pred_sec", "median"),
        MedAE_sec=("abs_error_p50_sec", "median"),
        MAE_sec=("abs_error_p50_sec", "mean"),
        Hit_5m_percent=("abs_error_p50_sec", lambda x: (x <= 300).mean() * 100),
        Hit_10m_percent=("abs_error_p50_sec", lambda x: (x <= 600).mean() * 100),
        P90_coverage_raw_percent=("p90_covered_raw", lambda x: x.mean() * 100),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

eta_validation_checks = []

def add_eta_check(name, passed, detail, severity="WARNING"):
    eta_validation_checks.append({
        "check": name,
        "severity": severity,
        "passed": bool(passed),
        "detail": detail,
    })

add_eta_check(
    "p50_beats_baseline_medae",
    p50_val_metrics["MedAE_sec"] <= best_eta_baseline_val["MedAE_sec"],
    f"p50_MedAE={p50_val_metrics['MedAE_sec']:.3f}, baseline_MedAE={best_eta_baseline_val['MedAE_sec']:.3f}",
)

add_eta_check(
    "p50_hit5m_at_least_baseline_minus_1pt",
    p50_val_metrics["Hit_5m_percent"] >= best_eta_baseline_val["Hit_5m_percent"] - 1.0,
    f"p50_Hit5m={p50_val_metrics['Hit_5m_percent']:.3f}, baseline_Hit5m={best_eta_baseline_val['Hit_5m_percent']:.3f}",
)

add_eta_check(
    "p90_raw_coverage_reasonable_before_calibration",
    85.0 <= p90_val_metrics["P90_coverage_raw_percent"] <= 95.0,
    f"P90_raw_coverage={p90_val_metrics['P90_coverage_raw_percent']:.3f}%",
)

add_eta_check(
    "p90_width_non_negative",
    p90_val_metrics["P90_invalid_width_rows"] == 0,
    f"invalid_width_rows={p90_val_metrics['P90_invalid_width_rows']}",
    severity="ERROR",
)

eta_validation_checks_df = pd.DataFrame(eta_validation_checks)

failed_eta_errors = eta_validation_checks_df[
    (eta_validation_checks_df["severity"] == "ERROR") & (~eta_validation_checks_df["passed"])
]

READY_FOR_BATCH_7 = len(failed_eta_errors) == 0

print("=" * 100)
print("P4-23 — ETA VALIDATION REPORT")
print("=" * 100)

print("\nETA validation summary:")
print(json.dumps(eta_validation_summary, indent=2, ensure_ascii=False))

print("\nETA validation comparison:")
display(eta_validation_comparison)

print("\nP90 interval metrics:")
display(pd.DataFrame([p90_val_metrics]))

print("\nETA by current status:")
display(eta_by_current_status)

print("\nETA by next target type:")
display(eta_by_next_target_type)

print("\nETA validation checks:")
display(eta_validation_checks_df)

print("\nTop 30 largest P50 validation errors:")
display(
    val_eta_preview
    .sort_values("abs_error_p50_sec", ascending=False)
    .head(30)
)

print("=" * 100)
if READY_FOR_BATCH_7:
    print("READY_FOR_BATCH_7 = True")
    print("Batch 6 complete. Send this output before moving to Batch 7.")
else:
    print("READY_FOR_BATCH_7 = False")
    print("Batch 6 has failed ERROR gates. Fix before moving to Batch 7.")
print("=" * 100)

P4-24 — Calibrate P90 globally and per machine with shrinkage

In [ ]:
# ============================================================
# P4-24 — Calibrate P90 globally and per machine with shrinkage
# ============================================================

P90_TARGET_COVERAGE = 0.90
P90_GLOBAL_MULTIPLIER_GRID = np.linspace(0.50, 3.00, 501)
P90_MACHINE_MULTIPLIER_GRID = np.linspace(0.50, 3.00, 501)

P90_MIN_MACHINE_CAL_ROWS = 200
P90_SHRINKAGE_K = 500
P90_MULTIPLIER_CLIP = (0.60, 3.00)

def coverage_at_multiplier(y_true, pred, multiplier):
    pred_adj = np.asarray(pred, dtype=np.float64) * float(multiplier)
    return float(np.mean(np.asarray(y_true, dtype=np.float64) <= pred_adj))


def find_multiplier_for_target_coverage(
    y_true,
    pred,
    target_coverage=0.90,
    grid=None,
):
    if grid is None:
        grid = np.linspace(0.50, 3.00, 501)

    y_true = np.asarray(y_true, dtype=np.float64)
    pred = np.asarray(pred, dtype=np.float64)

    coverages = np.array([
        coverage_at_multiplier(y_true, pred, m)
        for m in grid
    ])

    idx = int(np.argmin(np.abs(coverages - target_coverage)))

    return {
        "multiplier": float(grid[idx]),
        "coverage": float(coverages[idx]),
        "coverage_percent": float(coverages[idx] * 100),
        "coverage_error": float(abs(coverages[idx] - target_coverage)),
    }


def shrink_machine_multiplier(machine_multiplier, global_multiplier, n, shrinkage_k=500):
    """
    Shrink noisy per-machine calibration toward global multiplier.

    More rows -> more machine-specific.
    Fewer rows -> closer to global.
    """
    w = n / (n + shrinkage_k)
    return float(w * machine_multiplier + (1.0 - w) * global_multiplier)


# Base P90 before calibration
p90_pred_val_base = np.asarray(p90_pred_val, dtype=np.float64)
p50_pred_val_base = np.asarray(p50_pred_val, dtype=np.float64)

# Global calibration
p90_global_calibration = find_multiplier_for_target_coverage(
    y_true=y_val_raw,
    pred=p90_pred_val_base,
    target_coverage=P90_TARGET_COVERAGE,
    grid=P90_GLOBAL_MULTIPLIER_GRID,
)

p90_multiplier_global = float(
    np.clip(
        p90_global_calibration["multiplier"],
        P90_MULTIPLIER_CLIP[0],
        P90_MULTIPLIER_CLIP[1],
    )
)

# Per-machine calibration on validation split
val_calib_df = val_df[
    ["process", "mc_no", "occurred", "mc_status", "next_target_type", "y_time_sec", "y_time_sec_clip"]
].copy()

val_calib_df["p50_pred_base_sec"] = p50_pred_val_base
val_calib_df["p90_pred_base_sec"] = p90_pred_val_base

p90_machine_rows = []

p90_multiplier_by_machine = {}

for (process, mc_no), g in val_calib_df.groupby(["process", "mc_no"]):
    n = len(g)

    base_cov = float(np.mean(g["y_time_sec"].to_numpy() <= g["p90_pred_base_sec"].to_numpy()) * 100)

    if n >= P90_MIN_MACHINE_CAL_ROWS:
        local_cal = find_multiplier_for_target_coverage(
            y_true=g["y_time_sec"].to_numpy(),
            pred=g["p90_pred_base_sec"].to_numpy(),
            target_coverage=P90_TARGET_COVERAGE,
            grid=P90_MACHINE_MULTIPLIER_GRID,
        )

        raw_local_multiplier = float(local_cal["multiplier"])
        raw_local_coverage = float(local_cal["coverage_percent"])

        shrunk_multiplier = shrink_machine_multiplier(
            machine_multiplier=raw_local_multiplier,
            global_multiplier=p90_multiplier_global,
            n=n,
            shrinkage_k=P90_SHRINKAGE_K,
        )

        shrunk_multiplier = float(
            np.clip(
                shrunk_multiplier,
                P90_MULTIPLIER_CLIP[0],
                P90_MULTIPLIER_CLIP[1],
            )
        )

        p90_multiplier_by_machine[(process, mc_no)] = shrunk_multiplier

        final_multiplier = shrunk_multiplier
        cal_source = "shrunk_machine"
    else:
        raw_local_multiplier = np.nan
        raw_local_coverage = np.nan
        final_multiplier = p90_multiplier_global
        cal_source = "global_low_support"

    final_cov = float(
        np.mean(
            g["y_time_sec"].to_numpy()
            <= g["p90_pred_base_sec"].to_numpy() * final_multiplier
        ) * 100
    )

    p90_machine_rows.append({
        "process": process,
        "mc_no": mc_no,
        "rows": n,
        "base_coverage_percent": base_cov,
        "raw_local_multiplier": raw_local_multiplier,
        "raw_local_coverage_percent": raw_local_coverage,
        "final_multiplier": final_multiplier,
        "final_coverage_percent": final_cov,
        "calibration_source": cal_source,
    })

p90_machine_calibration_df = pd.DataFrame(p90_machine_rows)

# Apply global + per-machine multiplier to validation
def apply_p90_calibration(df, p90_base_pred, global_multiplier, by_machine):
    multipliers = np.array([
        by_machine.get((process, mc_no), global_multiplier)
        for process, mc_no in zip(df["process"], df["mc_no"])
    ], dtype=np.float64)

    return np.asarray(p90_base_pred, dtype=np.float64) * multipliers, multipliers


p90_pred_val_calibrated_raw, p90_val_multipliers = apply_p90_calibration(
    df=val_df,
    p90_base_pred=p90_pred_val_base,
    global_multiplier=p90_multiplier_global,
    by_machine=p90_multiplier_by_machine,
)

# Enforce P90 >= P50 after calibration
p90_pred_val_calibrated = np.maximum(
    p90_pred_val_calibrated_raw,
    p50_pred_val_base + 1.0,
)

p90_calibration_summary = {
    "target_coverage_percent": P90_TARGET_COVERAGE * 100,
    "base_global_coverage_percent": float(np.mean(y_val_raw <= p90_pred_val_base) * 100),
    "global_multiplier": p90_multiplier_global,
    "global_multiplier_coverage_percent": float(
        np.mean(y_val_raw <= (p90_pred_val_base * p90_multiplier_global)) * 100
    ),
    "final_calibrated_coverage_percent": float(np.mean(y_val_raw <= p90_pred_val_calibrated) * 100),
    "final_calibrated_clip_coverage_percent": float(np.mean(y_val_clip <= p90_pred_val_calibrated) * 100),
    "machine_calibration_min_rows": P90_MIN_MACHINE_CAL_ROWS,
    "machine_calibration_shrinkage_k": P90_SHRINKAGE_K,
    "machine_multipliers_count": int(len(p90_multiplier_by_machine)),
    "multiplier_clip": P90_MULTIPLIER_CLIP,
    "multiplier_median": float(np.median(p90_val_multipliers)),
    "multiplier_p10": float(np.quantile(p90_val_multipliers, 0.10)),
    "multiplier_p90": float(np.quantile(p90_val_multipliers, 0.90)),
    "multiplier_min": float(np.min(p90_val_multipliers)),
    "multiplier_max": float(np.max(p90_val_multipliers)),
}

print("P4-24 ready")

print("\nP90 calibration summary:")
print(json.dumps(p90_calibration_summary, indent=2, ensure_ascii=False))

print("\nP90 machine calibration summary:")
display(
    p90_machine_calibration_df[
        [
            "rows",
            "base_coverage_percent",
            "raw_local_multiplier",
            "final_multiplier",
            "final_coverage_percent",
        ]
    ].describe()
)

print("\nLowest 30 final machine P90 coverage after calibration:")
display(
    p90_machine_calibration_df
    .sort_values(["final_coverage_percent", "rows"], ascending=[True, False])
    .head(30)
)

print("\nHighest 30 final machine P90 coverage after calibration:")
display(
    p90_machine_calibration_df
    .sort_values(["final_coverage_percent", "rows"], ascending=[False, False])
    .head(30)
)

P4-25 — Calibrate P50 per-machine bias

In [ ]:
# ============================================================
# P4-25 — Calibrate P50 per-machine bias
# ============================================================

P50_MIN_MACHINE_CAL_ROWS = 200
P50_SHRINKAGE_K = 500
P50_SCALE_CLIP = (0.70, 1.30)

def safe_ratio(numerator, denominator, default=1.0):
    if denominator is None or pd.isna(denominator) or denominator <= 0:
        return default
    return float(numerator / denominator)


p50_machine_rows = []
p50_scale_by_machine = {}

global_true_median = float(np.median(y_val_clip))
global_pred_median = float(np.median(p50_pred_val_base))
p50_scale_global_raw = safe_ratio(global_true_median, global_pred_median, default=1.0)

p50_scale_global = float(
    np.clip(
        p50_scale_global_raw,
        P50_SCALE_CLIP[0],
        P50_SCALE_CLIP[1],
    )
)

val_p50_calib_df = val_df[
    ["process", "mc_no", "occurred", "mc_status", "next_target_type", "y_time_sec", "y_time_sec_clip"]
].copy()

val_p50_calib_df["p50_pred_base_sec"] = p50_pred_val_base

for (process, mc_no), g in val_p50_calib_df.groupby(["process", "mc_no"]):
    n = len(g)

    true_median = float(g["y_time_sec_clip"].median())
    pred_median = float(g["p50_pred_base_sec"].median())

    raw_scale = safe_ratio(true_median, pred_median, default=p50_scale_global)

    if n >= P50_MIN_MACHINE_CAL_ROWS:
        w = n / (n + P50_SHRINKAGE_K)
        shrunk_scale = w * raw_scale + (1.0 - w) * p50_scale_global
        final_scale = float(np.clip(shrunk_scale, P50_SCALE_CLIP[0], P50_SCALE_CLIP[1]))
        scale_source = "shrunk_machine"
        p50_scale_by_machine[(process, mc_no)] = final_scale
    else:
        final_scale = p50_scale_global
        scale_source = "global_low_support"

    p50_machine_rows.append({
        "process": process,
        "mc_no": mc_no,
        "rows": n,
        "true_median_sec": true_median,
        "pred_median_sec": pred_median,
        "raw_scale": raw_scale,
        "final_scale": final_scale,
        "scale_source": scale_source,
    })

p50_machine_calibration_df = pd.DataFrame(p50_machine_rows)

def apply_p50_calibration(df, p50_base_pred, global_scale, by_machine):
    scales = np.array([
        by_machine.get((process, mc_no), global_scale)
        for process, mc_no in zip(df["process"], df["mc_no"])
    ], dtype=np.float64)

    return np.asarray(p50_base_pred, dtype=np.float64) * scales, scales


p50_pred_val_calibrated_raw, p50_val_scales = apply_p50_calibration(
    df=val_df,
    p50_base_pred=p50_pred_val_base,
    global_scale=p50_scale_global,
    by_machine=p50_scale_by_machine,
)

p50_pred_val_calibrated = clean_eta_pred(
    p50_pred_val_calibrated_raw,
    clip_upper=clip_max,
)

# Preserve P90 >= P50 after P50 scaling
p90_pred_val_calibrated = np.maximum(
    p90_pred_val_calibrated,
    p50_pred_val_calibrated + 1.0,
)

p50_calibrated_metrics = evaluate_eta_predictions(
    y_true_raw=y_val_raw,
    y_pred=p50_pred_val_calibrated,
    name="lgbm_p50_calibrated",
    split="val",
)

p50_calibration_summary = {
    "global_true_median_sec": global_true_median,
    "global_pred_median_sec": global_pred_median,
    "global_scale_raw": p50_scale_global_raw,
    "global_scale_final": p50_scale_global,
    "machine_scale_count": int(len(p50_scale_by_machine)),
    "machine_calibration_min_rows": P50_MIN_MACHINE_CAL_ROWS,
    "machine_calibration_shrinkage_k": P50_SHRINKAGE_K,
    "scale_clip": P50_SCALE_CLIP,
    "scale_median": float(np.median(p50_val_scales)),
    "scale_p10": float(np.quantile(p50_val_scales, 0.10)),
    "scale_p90": float(np.quantile(p50_val_scales, 0.90)),
    "scale_min": float(np.min(p50_val_scales)),
    "scale_max": float(np.max(p50_val_scales)),
    "base_p50_MedAE_sec": float(p50_val_metrics["MedAE_sec"]),
    "calibrated_p50_MedAE_sec": float(p50_calibrated_metrics["MedAE_sec"]),
    "base_p50_Hit_5m_percent": float(p50_val_metrics["Hit_5m_percent"]),
    "calibrated_p50_Hit_5m_percent": float(p50_calibrated_metrics["Hit_5m_percent"]),
}

print("P4-25 ready")

print("\nP50 calibration summary:")
print(json.dumps(p50_calibration_summary, indent=2, ensure_ascii=False))

print("\nP50 machine calibration summary:")
display(
    p50_machine_calibration_df[
        ["rows", "true_median_sec", "pred_median_sec", "raw_scale", "final_scale"]
    ].describe()
)

print("\nMost downscaled machines:")
display(
    p50_machine_calibration_df
    .sort_values("final_scale", ascending=True)
    .head(30)
)

print("\nMost upscaled machines:")
display(
    p50_machine_calibration_df
    .sort_values("final_scale", ascending=False)
    .head(30)
)

P4-26 — Build ETA floor/median/cap fallback maps

In [ ]:
# ============================================================
# P4-26 — Build ETA floor/median/cap fallback maps
# ============================================================

ETA_MIN_FLOOR_SEC = 1.0
ETA_MAX_CAP_SEC = float(clip_max)

def q05(x):
    return float(np.quantile(x, 0.05))

def q50(x):
    return float(np.quantile(x, 0.50))

def q95(x):
    return float(np.quantile(x, 0.95))

def build_eta_postprocess_maps(train: pd.DataFrame) -> dict:
    """
    Build train-only hierarchical ETA fallback maps.

    Fallback order later:
      machine + current status
      machine
      current status
      global
    """
    y_col = "y_time_sec_clip"

    global_stats = {
        "floor": float(max(ETA_MIN_FLOOR_SEC, train[y_col].quantile(0.01))),
        "median": float(train[y_col].median()),
        "cap": float(min(ETA_MAX_CAP_SEC, train[y_col].quantile(0.99))),
    }

    by_machine_status_df = (
        train.groupby(["process", "mc_no", "mc_status"])[y_col]
        .agg(
            rows="size",
            floor=q05,
            median=q50,
            cap=q95,
        )
        .reset_index()
    )

    by_machine_df = (
        train.groupby(["process", "mc_no"])[y_col]
        .agg(
            rows="size",
            floor=q05,
            median=q50,
            cap=q95,
        )
        .reset_index()
    )

    by_status_df = (
        train.groupby(["mc_status"])[y_col]
        .agg(
            rows="size",
            floor=q05,
            median=q50,
            cap=q95,
        )
        .reset_index()
    )

    # Clip floors/caps to safe global limits
    for d in [by_machine_status_df, by_machine_df, by_status_df]:
        d["floor"] = d["floor"].clip(lower=ETA_MIN_FLOOR_SEC, upper=ETA_MAX_CAP_SEC)
        d["median"] = d["median"].clip(lower=ETA_MIN_FLOOR_SEC, upper=ETA_MAX_CAP_SEC)
        d["cap"] = d["cap"].clip(lower=ETA_MIN_FLOOR_SEC, upper=ETA_MAX_CAP_SEC)

        # Ensure cap >= median >= floor
        d["median"] = np.maximum(d["median"], d["floor"])
        d["cap"] = np.maximum(d["cap"], d["median"] + 1.0)

    by_machine_status = {
        (r["process"], r["mc_no"], r["mc_status"]): {
            "rows": int(r["rows"]),
            "floor": float(r["floor"]),
            "median": float(r["median"]),
            "cap": float(r["cap"]),
        }
        for _, r in by_machine_status_df.iterrows()
    }

    by_machine = {
        (r["process"], r["mc_no"]): {
            "rows": int(r["rows"]),
            "floor": float(r["floor"]),
            "median": float(r["median"]),
            "cap": float(r["cap"]),
        }
        for _, r in by_machine_df.iterrows()
    }

    by_status = {
        r["mc_status"]: {
            "rows": int(r["rows"]),
            "floor": float(r["floor"]),
            "median": float(r["median"]),
            "cap": float(r["cap"]),
        }
        for _, r in by_status_df.iterrows()
    }

    return {
        "global": global_stats,
        "by_machine_status": by_machine_status,
        "by_machine": by_machine,
        "by_status": by_status,
        "by_machine_status_df": by_machine_status_df,
        "by_machine_df": by_machine_df,
        "by_status_df": by_status_df,
    }


def get_eta_bounds(row, maps):
    process = row["process"]
    mc_no = row["mc_no"]
    status = row["mc_status"]

    key_ms = (process, mc_no, status)
    key_m = (process, mc_no)

    if key_ms in maps["by_machine_status"]:
        return maps["by_machine_status"][key_ms], "machine_status"

    if key_m in maps["by_machine"]:
        return maps["by_machine"][key_m], "machine"

    if status in maps["by_status"]:
        return maps["by_status"][status], "status"

    return maps["global"], "global"


def apply_eta_postprocess(df, p50_pred, p90_pred, maps):
    p50 = np.asarray(p50_pred, dtype=np.float64).copy()
    p90 = np.asarray(p90_pred, dtype=np.float64).copy()

    floors = np.zeros(len(df), dtype=np.float64)
    caps = np.zeros(len(df), dtype=np.float64)
    medians = np.zeros(len(df), dtype=np.float64)
    sources = []

    for i, (_, row) in enumerate(df.iterrows()):
        bounds, source = get_eta_bounds(row, maps)
        floors[i] = bounds["floor"]
        medians[i] = bounds["median"]
        caps[i] = bounds["cap"]
        sources.append(source)

    # P50: bound to floor/cap.
    p50_pp = np.clip(p50, floors, caps)

    # P90: should be >= P50 + 1.
    # Do not make P90 too tiny; let it be at least median or P50+1.
    p90_pp = np.maximum(p90, p50_pp + 1.0)
    p90_pp = np.maximum(p90_pp, medians)
    p90_pp = np.clip(p90_pp, p50_pp + 1.0, ETA_MAX_CAP_SEC * 1.50)

    return p50_pp, p90_pp, pd.Series(sources, index=df.index, name="eta_bound_source")


eta_postprocess_maps = build_eta_postprocess_maps(train_df)

p50_pred_val_final, p90_pred_val_final, eta_bound_source_val = apply_eta_postprocess(
    df=val_df,
    p50_pred=p50_pred_val_calibrated,
    p90_pred=p90_pred_val_calibrated,
    maps=eta_postprocess_maps,
)

eta_postprocess_summary = {
    "global": eta_postprocess_maps["global"],
    "by_machine_status_count": int(len(eta_postprocess_maps["by_machine_status"])),
    "by_machine_count": int(len(eta_postprocess_maps["by_machine"])),
    "by_status_count": int(len(eta_postprocess_maps["by_status"])),
    "eta_min_floor_sec": ETA_MIN_FLOOR_SEC,
    "eta_max_cap_sec": ETA_MAX_CAP_SEC,
}

print("P4-26 ready")

print("\nETA postprocess summary:")
print(json.dumps(eta_postprocess_summary, indent=2, ensure_ascii=False))

print("\nETA bound source distribution on validation:")
display(
    eta_bound_source_val
    .value_counts()
    .rename_axis("eta_bound_source")
    .reset_index(name="rows")
)

print("\nBy-status postprocess map:")
display(eta_postprocess_maps["by_status_df"])

print("\nBy-machine postprocess map summary:")
display(
    eta_postprocess_maps["by_machine_df"][
        ["rows", "floor", "median", "cap"]
    ].describe()
)

print("\nBy-machine-status postprocess map summary:")
display(
    eta_postprocess_maps["by_machine_status_df"][
        ["rows", "floor", "median", "cap"]
    ].describe()
)

P4-27 — Validation report after calibration

In [ ]:
# ============================================================
# P4-27 — Validation report after calibration
# ============================================================

p50_final_val_metrics = evaluate_eta_predictions(
    y_true_raw=y_val_raw,
    y_pred=p50_pred_val_final,
    name="lgbm_p50_final_calibrated_postprocessed",
    split="val",
)

p90_final_val_metrics = evaluate_p90_interval(
    y_true_raw=y_val_raw,
    y_true_clip=y_val_clip,
    p50_pred=p50_pred_val_final,
    p90_pred=p90_pred_val_final,
    split="val",
)

p90_final_val_metrics.update({
    "pinball90_clip": pinball_loss(y_val_clip, p90_pred_val_final, 0.90),
})

eta_calibration_comparison = pd.DataFrame([
    {
        "stage": "baseline_machine_status_median",
        "P50_MedAE_sec": best_eta_baseline_val["MedAE_sec"],
        "P50_MAE_sec": best_eta_baseline_val["MAE_sec"],
        "P50_Hit_2m_percent": best_eta_baseline_val["Hit_2m_percent"],
        "P50_Hit_5m_percent": best_eta_baseline_val["Hit_5m_percent"],
        "P50_Hit_10m_percent": best_eta_baseline_val["Hit_10m_percent"],
        "P90_coverage_raw_percent": np.nan,
        "P90_median_width_sec": np.nan,
        "P90_p90_width_sec": np.nan,
    },
    {
        "stage": "lgbm_raw_selected",
        "P50_MedAE_sec": p50_val_metrics["MedAE_sec"],
        "P50_MAE_sec": p50_val_metrics["MAE_sec"],
        "P50_Hit_2m_percent": p50_val_metrics["Hit_2m_percent"],
        "P50_Hit_5m_percent": p50_val_metrics["Hit_5m_percent"],
        "P50_Hit_10m_percent": p50_val_metrics["Hit_10m_percent"],
        "P90_coverage_raw_percent": p90_val_metrics["P90_coverage_raw_percent"],
        "P90_median_width_sec": p90_val_metrics["P90_median_width_sec"],
        "P90_p90_width_sec": p90_val_metrics["P90_p90_width_sec"],
    },
    {
        "stage": "lgbm_calibrated_postprocessed",
        "P50_MedAE_sec": p50_final_val_metrics["MedAE_sec"],
        "P50_MAE_sec": p50_final_val_metrics["MAE_sec"],
        "P50_Hit_2m_percent": p50_final_val_metrics["Hit_2m_percent"],
        "P50_Hit_5m_percent": p50_final_val_metrics["Hit_5m_percent"],
        "P50_Hit_10m_percent": p50_final_val_metrics["Hit_10m_percent"],
        "P90_coverage_raw_percent": p90_final_val_metrics["P90_coverage_raw_percent"],
        "P90_median_width_sec": p90_final_val_metrics["P90_median_width_sec"],
        "P90_p90_width_sec": p90_final_val_metrics["P90_p90_width_sec"],
    },
])

val_eta_final_df = val_df[
    ["process", "mc_no", "occurred", "mc_status", "next_target_type", "y_time_sec", "y_time_sec_clip"]
].copy()

val_eta_final_df["p50_pred_sec"] = p50_pred_val_final
val_eta_final_df["p90_pred_sec"] = p90_pred_val_final
val_eta_final_df["abs_error_p50_sec"] = np.abs(val_eta_final_df["y_time_sec"] - val_eta_final_df["p50_pred_sec"])
val_eta_final_df["p90_covered_raw"] = val_eta_final_df["y_time_sec"] <= val_eta_final_df["p90_pred_sec"]
val_eta_final_df["p90_width_sec"] = val_eta_final_df["p90_pred_sec"] - val_eta_final_df["p50_pred_sec"]
val_eta_final_df["eta_bound_source"] = eta_bound_source_val.values

eta_final_by_current_status = (
    val_eta_final_df
    .groupby("mc_status")
    .agg(
        rows=("mc_status", "size"),
        actual_median_sec=("y_time_sec", "median"),
        actual_p90_sec=("y_time_sec", lambda x: x.quantile(0.90)),
        p50_pred_median_sec=("p50_pred_sec", "median"),
        p90_pred_median_sec=("p90_pred_sec", "median"),
        MedAE_sec=("abs_error_p50_sec", "median"),
        MAE_sec=("abs_error_p50_sec", "mean"),
        Hit_5m_percent=("abs_error_p50_sec", lambda x: (x <= 300).mean() * 100),
        Hit_10m_percent=("abs_error_p50_sec", lambda x: (x <= 600).mean() * 100),
        P90_coverage_raw_percent=("p90_covered_raw", lambda x: x.mean() * 100),
        P90_median_width_sec=("p90_width_sec", "median"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

eta_final_by_next_target_type = (
    val_eta_final_df
    .groupby("next_target_type")
    .agg(
        rows=("next_target_type", "size"),
        actual_median_sec=("y_time_sec", "median"),
        actual_p90_sec=("y_time_sec", lambda x: x.quantile(0.90)),
        p50_pred_median_sec=("p50_pred_sec", "median"),
        p90_pred_median_sec=("p90_pred_sec", "median"),
        MedAE_sec=("abs_error_p50_sec", "median"),
        MAE_sec=("abs_error_p50_sec", "mean"),
        Hit_5m_percent=("abs_error_p50_sec", lambda x: (x <= 300).mean() * 100),
        Hit_10m_percent=("abs_error_p50_sec", lambda x: (x <= 600).mean() * 100),
        P90_coverage_raw_percent=("p90_covered_raw", lambda x: x.mean() * 100),
        P90_median_width_sec=("p90_width_sec", "median"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

eta_final_by_machine = (
    val_eta_final_df
    .groupby(["process", "mc_no"])
    .agg(
        rows=("mc_no", "size"),
        actual_median_sec=("y_time_sec", "median"),
        actual_p90_sec=("y_time_sec", lambda x: x.quantile(0.90)),
        p50_pred_median_sec=("p50_pred_sec", "median"),
        MedAE_sec=("abs_error_p50_sec", "median"),
        Hit_5m_percent=("abs_error_p50_sec", lambda x: (x <= 300).mean() * 100),
        Hit_10m_percent=("abs_error_p50_sec", lambda x: (x <= 600).mean() * 100),
        P90_coverage_raw_percent=("p90_covered_raw", lambda x: x.mean() * 100),
        P90_median_width_sec=("p90_width_sec", "median"),
    )
    .reset_index()
)

eta_final_by_machine_n200 = eta_final_by_machine[eta_final_by_machine["rows"] >= 200].copy()

p90_machine_n200_min_coverage = (
    float(eta_final_by_machine_n200["P90_coverage_raw_percent"].min())
    if len(eta_final_by_machine_n200) > 0 else np.nan
)

p90_machine_n200_median_coverage = (
    float(eta_final_by_machine_n200["P90_coverage_raw_percent"].median())
    if len(eta_final_by_machine_n200) > 0 else np.nan
)

eta_calibration_checks = []

def add_cal_check(name, passed, detail, severity="WARNING"):
    eta_calibration_checks.append({
        "check": name,
        "severity": severity,
        "passed": bool(passed),
        "detail": detail,
    })

add_cal_check(
    "p50_final_medae_not_worse_than_raw_by_more_than_2s",
    p50_final_val_metrics["MedAE_sec"] <= p50_val_metrics["MedAE_sec"] + 2.0,
    f"final_MedAE={p50_final_val_metrics['MedAE_sec']:.3f}, raw_MedAE={p50_val_metrics['MedAE_sec']:.3f}",
)

add_cal_check(
    "p50_final_beats_baseline_medae",
    p50_final_val_metrics["MedAE_sec"] <= best_eta_baseline_val["MedAE_sec"],
    f"final_MedAE={p50_final_val_metrics['MedAE_sec']:.3f}, baseline_MedAE={best_eta_baseline_val['MedAE_sec']:.3f}",
)

add_cal_check(
    "p50_final_hit5m_beats_baseline",
    p50_final_val_metrics["Hit_5m_percent"] >= best_eta_baseline_val["Hit_5m_percent"],
    f"final_Hit5m={p50_final_val_metrics['Hit_5m_percent']:.3f}, baseline_Hit5m={best_eta_baseline_val['Hit_5m_percent']:.3f}",
)

add_cal_check(
    "p90_final_global_coverage_target_range",
    88.0 <= p90_final_val_metrics["P90_coverage_raw_percent"] <= 93.0,
    f"P90_final_coverage={p90_final_val_metrics['P90_coverage_raw_percent']:.3f}%",
)

add_cal_check(
    "p90_final_width_non_negative",
    p90_final_val_metrics["P90_invalid_width_rows"] == 0,
    f"invalid_width_rows={p90_final_val_metrics['P90_invalid_width_rows']}",
    severity="ERROR",
)

add_cal_check(
    "p90_machine_n200_median_coverage_reasonable",
    pd.isna(p90_machine_n200_median_coverage) or p90_machine_n200_median_coverage >= 84.0,
    f"machine_n200_median_coverage={p90_machine_n200_median_coverage}",
)

eta_calibration_checks_df = pd.DataFrame(eta_calibration_checks)

failed_cal_errors = eta_calibration_checks_df[
    (eta_calibration_checks_df["severity"] == "ERROR") & (~eta_calibration_checks_df["passed"])
]

READY_FOR_BATCH_8 = len(failed_cal_errors) == 0

eta_final_validation_summary = {
    "p50_final_MedAE_sec": float(p50_final_val_metrics["MedAE_sec"]),
    "p50_final_MAE_sec": float(p50_final_val_metrics["MAE_sec"]),
    "p50_final_Hit_2m_percent": float(p50_final_val_metrics["Hit_2m_percent"]),
    "p50_final_Hit_5m_percent": float(p50_final_val_metrics["Hit_5m_percent"]),
    "p50_final_Hit_10m_percent": float(p50_final_val_metrics["Hit_10m_percent"]),
    "p90_final_coverage_raw_percent": float(p90_final_val_metrics["P90_coverage_raw_percent"]),
    "p90_final_coverage_clip_percent": float(p90_final_val_metrics["P90_coverage_clip_percent"]),
    "p90_final_median_width_sec": float(p90_final_val_metrics["P90_median_width_sec"]),
    "p90_final_p90_width_sec": float(p90_final_val_metrics["P90_p90_width_sec"]),
    "p90_machine_n200_count": int(len(eta_final_by_machine_n200)),
    "p90_machine_n200_min_coverage": p90_machine_n200_min_coverage,
    "p90_machine_n200_median_coverage": p90_machine_n200_median_coverage,
}

print("=" * 100)
print("P4-27 — ETA VALIDATION REPORT AFTER CALIBRATION")
print("=" * 100)

print("\nFinal ETA validation summary:")
print(json.dumps(eta_final_validation_summary, indent=2, ensure_ascii=False))

print("\nCalibration comparison:")
display(eta_calibration_comparison)

print("\nP90 calibration summary:")
print(json.dumps(p90_calibration_summary, indent=2, ensure_ascii=False))

print("\nP50 calibration summary:")
print(json.dumps(p50_calibration_summary, indent=2, ensure_ascii=False))

print("\nETA final by current status:")
display(eta_final_by_current_status)

print("\nETA final by next target type:")
display(eta_final_by_next_target_type)

print("\nPer-machine P90 coverage summary:")
display(
    eta_final_by_machine[
        ["rows", "MedAE_sec", "Hit_5m_percent", "Hit_10m_percent", "P90_coverage_raw_percent", "P90_median_width_sec"]
    ].describe()
)

print("\nPer-machine P90 coverage summary for n>=200:")
display(
    eta_final_by_machine_n200[
        ["rows", "MedAE_sec", "Hit_5m_percent", "Hit_10m_percent", "P90_coverage_raw_percent", "P90_median_width_sec"]
    ].describe()
)

print("\nLowest 30 P90 coverage machines, n>=200:")
display(
    eta_final_by_machine_n200
    .sort_values("P90_coverage_raw_percent", ascending=True)
    .head(30)
)

print("\nHighest 30 P90 coverage machines, n>=200:")
display(
    eta_final_by_machine_n200
    .sort_values("P90_coverage_raw_percent", ascending=False)
    .head(30)
)

print("\nETA calibration validation checks:")
display(eta_calibration_checks_df)

print("\nTop 30 largest final P50 validation errors:")
display(
    val_eta_final_df
    .sort_values("abs_error_p50_sec", ascending=False)
    .head(30)
)

print("=" * 100)
if READY_FOR_BATCH_8:
    print("READY_FOR_BATCH_8 = True")
    print("Batch 7 complete. Send this output before moving to Batch 8.")
else:
    print("READY_FOR_BATCH_8 = False")
    print("Batch 7 has failed ERROR gates. Fix before moving to Batch 8.")
print("=" * 100)

P4-28 — Train dynamic LightGBM next-type classifier

In [ ]:
# ============================================================
# P4-28 — Train dynamic LightGBM next-type classifier
# ============================================================

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss
from lightgbm import LGBMClassifier

# ------------------------------------------------------------
# Label encoding
# ------------------------------------------------------------

type_label_encoder = LabelEncoder()

y_type_train_enc = type_label_encoder.fit_transform(y_type_train)
y_type_val_enc = type_label_encoder.transform(y_type_val)
y_type_test_enc = type_label_encoder.transform(y_type_test)

type_classes = type_label_encoder.classes_.tolist()
type_class_to_idx = {c: int(i) for i, c in enumerate(type_classes)}
type_idx_to_class = {int(i): c for i, c in enumerate(type_classes)}

n_type_classes = len(type_classes)

print("Type classes:")
print(type_classes)

# ------------------------------------------------------------
# Class frequency and soft class weights
# ------------------------------------------------------------

type_train_counts = pd.Series(y_type_train).value_counts().sort_index()
type_val_counts = pd.Series(y_type_val).value_counts().sort_index()
type_test_counts = pd.Series(y_type_test).value_counts().sort_index()

type_class_frequency_report = pd.DataFrame({
    "class": type_classes,
    "train_count": [int(type_train_counts.get(c, 0)) for c in type_classes],
    "val_count": [int(type_val_counts.get(c, 0)) for c in type_classes],
    "test_count": [int(type_test_counts.get(c, 0)) for c in type_classes],
})

type_class_frequency_report["train_percent"] = (
    type_class_frequency_report["train_count"] / len(y_type_train) * 100
)

type_class_frequency_report["val_percent"] = (
    type_class_frequency_report["val_count"] / len(y_type_val) * 100
)

type_class_frequency_report["test_percent"] = (
    type_class_frequency_report["test_count"] / len(y_type_test) * 100
)

# Soft class weight:
# Full inverse frequency is too aggressive, so use sqrt inverse frequency.
total_train_type = len(y_type_train)
class_weight_by_class = {}

for c in type_classes:
    count_c = max(int(type_train_counts.get(c, 0)), 1)
    class_weight_by_class[c] = math.sqrt(total_train_type / (n_type_classes * count_c))

# Normalize class weights around mean 1
mean_class_weight = np.mean(list(class_weight_by_class.values()))
class_weight_by_class = {
    c: float(w / mean_class_weight)
    for c, w in class_weight_by_class.items()
}

class_weight_by_idx = {
    type_class_to_idx[c]: class_weight_by_class[c]
    for c in type_classes
}

class_weight_report = pd.DataFrame({
    "class": type_classes,
    "class_index": [type_class_to_idx[c] for c in type_classes],
    "train_count": [int(type_train_counts.get(c, 0)) for c in type_classes],
    "train_percent": [float(type_train_counts.get(c, 0) / total_train_type * 100) for c in type_classes],
    "soft_class_weight": [class_weight_by_class[c] for c in type_classes],
})

# Classifier sample weight = capped machine-balanced weight × soft class weight
sample_weight_train_type = np.array([
    sample_weight_train_eta[i] * class_weight_by_class[y_type_train[i]]
    for i in range(len(y_type_train))
], dtype=np.float32)

TYPE_SAMPLE_WEIGHT_CAP = 8.0

sample_weight_train_type = np.minimum(
    sample_weight_train_type,
    TYPE_SAMPLE_WEIGHT_CAP
).astype(np.float32)

sample_weight_train_type = (
    sample_weight_train_type / sample_weight_train_type.mean()
).astype(np.float32)

type_weight_summary = pd.Series(sample_weight_train_type).describe().to_frame("sample_weight_train_type")

print("P4-28 class setup ready")

print("\nType class frequency report:")
display(type_class_frequency_report)

print("\nClass weight report:")
display(class_weight_report)

print("\nType sample weight summary:")
display(type_weight_summary)

# ------------------------------------------------------------
# Candidate classifier training
# ------------------------------------------------------------

TYPE_CANDIDATE_PARAMS = [
    {
        "candidate_id": "balanced_63",
        "learning_rate": 0.03,
        "n_estimators": 3000,
        "num_leaves": 63,
        "min_child_samples": 100,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.2,
        "reg_lambda": 0.5,
        "early_stopping_rounds": 120,
    },
    {
        "candidate_id": "regularized_63",
        "learning_rate": 0.03,
        "n_estimators": 3000,
        "num_leaves": 63,
        "min_child_samples": 200,
        "subsample": 0.85,
        "subsample_freq": 1,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.5,
        "reg_lambda": 1.0,
        "early_stopping_rounds": 120,
    },
    {
        "candidate_id": "compact_31",
        "learning_rate": 0.04,
        "n_estimators": 2500,
        "num_leaves": 31,
        "min_child_samples": 100,
        "subsample": 0.90,
        "subsample_freq": 1,
        "colsample_bytree": 0.90,
        "reg_alpha": 0.2,
        "reg_lambda": 0.5,
        "early_stopping_rounds": 100,
    },
]


def train_lgbm_type_candidate(
    params,
    X_train,
    y_train_enc,
    X_val,
    y_val_enc,
    sample_weight_train,
    model_name,
):
    model = LGBMClassifier(
        objective="multiclass",
        num_class=n_type_classes,
        device_type="gpu",
        max_bin=63,             # 🚀 Tailored for OpenCL kernel hardware speed
        gpu_use_dp=False,       # 🚀 Prevents the CPU from sending heavy float64 data
        n_jobs=8,               # 🚀 Exact physical core match for your i7-10700
        n_estimators=params.get("n_estimators", 3000),
        learning_rate=params.get("learning_rate", 0.03),
        num_leaves=params.get("num_leaves", 63),
        max_depth=params.get("max_depth", -1),
        min_child_samples=params.get("min_child_samples", 100),
        subsample=params.get("subsample", 0.85),
        subsample_freq=params.get("subsample_freq", 1),
        colsample_bytree=params.get("colsample_bytree", 0.85),
        reg_alpha=params.get("reg_alpha", 0.2),
        reg_lambda=params.get("reg_lambda", 0.5),
        random_state=SEED,
        # n_jobs=-1,
        # force_col_wise=True,
        verbosity=-1,
    )

    model.fit(
        X_train,
        y_train_enc,
        sample_weight=sample_weight_train,
        eval_set=[(X_val, y_val_enc)],
        eval_metric="multi_logloss",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=params.get("early_stopping_rounds", 120),
                verbose=False,
            ),
            lgb.log_evaluation(period=200),
        ],
    )

    proba_val = model.predict_proba(X_val, num_iteration=model.best_iteration_)
    pred_val_enc = np.argmax(proba_val, axis=1)
    conf_val = np.max(proba_val, axis=1)

    pred_val_type = type_label_encoder.inverse_transform(pred_val_enc)

    return {
        "model_name": model_name,
        "model": model,
        "params": params,
        "best_iteration": int(model.best_iteration_ or params.get("n_estimators", 3000)),
        "proba_val": proba_val,
        "pred_val_enc": pred_val_enc,
        "pred_val_type": pred_val_type,
        "conf_val": conf_val,
    }


type_candidates = []

for params in TYPE_CANDIDATE_PARAMS:
    candidate_id = params["candidate_id"]
    model_name = f"type_{candidate_id}"

    print("=" * 100)
    print(f"Training {model_name}")
    print("=" * 100)

    result = train_lgbm_type_candidate(
        params=params,
        X_train=X_train,
        y_train_enc=y_type_train_enc,
        X_val=X_val,
        y_val_enc=y_type_val_enc,
        sample_weight_train=sample_weight_train_type,
        model_name=model_name,
    )

    pred = result["pred_val_type"]
    proba = result["proba_val"]

    row = evaluate_type_predictions(
        y_true=y_type_val,
        y_pred=pred,
        name=model_name,
        split="val",
    )

    try:
        val_logloss = log_loss(
            y_type_val_enc,
            proba,
            labels=list(range(n_type_classes)),
        )
    except Exception:
        val_logloss = np.nan

    row.update({
        "candidate_id": candidate_id,
        "best_iteration": result["best_iteration"],
        "multi_logloss": float(val_logloss),
        "mean_confidence": float(np.mean(result["conf_val"])),
        "median_confidence": float(np.median(result["conf_val"])),
        "p10_confidence": float(np.quantile(result["conf_val"], 0.10)),
        "p90_confidence": float(np.quantile(result["conf_val"], 0.90)),
    })

    result["metrics"] = row
    type_candidates.append(result)

type_candidate_report = pd.DataFrame([c["metrics"] for c in type_candidates])

type_candidate_report_sorted = type_candidate_report.sort_values(
    ["weighted_f1", "accuracy", "macro_f1", "multi_logloss"],
    ascending=[False, False, False, True],
).reset_index(drop=True)

# Select classifier
best_type_row = type_candidate_report_sorted.iloc[0]
best_type_name = best_type_row["baseline"]

best_type_candidate = next(
    c for c in type_candidates
    if c["model_name"] == best_type_name
)

lgbm_type_final = best_type_candidate["model"]
type_pred_val = best_type_candidate["pred_val_type"]
type_proba_val = best_type_candidate["proba_val"]
type_conf_val = best_type_candidate["conf_val"]
type_pred_val_enc = best_type_candidate["pred_val_enc"]

type_selection_summary = {
    "selected_type_model": best_type_name,
    "selected_type_best_iteration": int(best_type_candidate["best_iteration"]),
    "selected_type_val_accuracy": float(best_type_row["accuracy"]),
    "selected_type_val_macro_f1": float(best_type_row["macro_f1"]),
    "selected_type_val_weighted_f1": float(best_type_row["weighted_f1"]),
    "selected_type_val_multi_logloss": float(best_type_row["multi_logloss"]),
    "best_type_baseline_val": best_type_baseline_name,
    "best_type_baseline_val_accuracy": float(best_type_baseline_val["accuracy"]),
    "best_type_baseline_val_macro_f1": float(best_type_baseline_val["macro_f1"]),
    "best_type_baseline_val_weighted_f1": float(best_type_baseline_val["weighted_f1"]),
}

type_vs_baseline = {
    "accuracy_improvement_point": float(best_type_row["accuracy"] - best_type_baseline_val["accuracy"]),
    "macro_f1_improvement_point": float(best_type_row["macro_f1"] - best_type_baseline_val["macro_f1"]),
    "weighted_f1_improvement_point": float(best_type_row["weighted_f1"] - best_type_baseline_val["weighted_f1"]),
}

print("=" * 100)
print("P4-28 — TYPE CLASSIFIER TRAINING COMPLETE")
print("=" * 100)

print("\nType selection summary:")
print(json.dumps(type_selection_summary, indent=2, ensure_ascii=False))

print("\nType vs best baseline:")
print(json.dumps(type_vs_baseline, indent=2, ensure_ascii=False))

print("\nType candidate report:")
display(type_candidate_report_sorted)

P4-29 — Classification and confidence report

In [ ]:
# ============================================================
# P4-29 — Classification and confidence report
# ============================================================

TYPE_CONF_THRESHOLD_GLOBAL = CFG.confidence_threshold

# ------------------------------------------------------------
# Core validation report
# ------------------------------------------------------------

type_val_report_dict = classification_report(
    y_type_val,
    type_pred_val,
    labels=type_classes,
    output_dict=True,
    zero_division=0,
)

type_val_report_df = pd.DataFrame(type_val_report_dict).T

type_val_confusion_df = pd.DataFrame(
    confusion_matrix(y_type_val, type_pred_val, labels=type_classes),
    index=[f"true_{c}" for c in type_classes],
    columns=[f"pred_{c}" for c in type_classes],
)

# ------------------------------------------------------------
# Confidence-filtered report
# ------------------------------------------------------------

type_val_eval_df = val_df[
    ["process", "mc_no", "occurred", "mc_status", "next_target_type", "y_time_sec"]
].copy()

type_val_eval_df["type_pred"] = type_pred_val
type_val_eval_df["type_conf"] = type_conf_val
type_val_eval_df["type_correct"] = (
    type_val_eval_df["type_pred"].astype(str)
    == type_val_eval_df["next_target_type"].astype(str)
)

type_val_eval_df["display_type_global_threshold"] = (
    type_val_eval_df["type_conf"] >= TYPE_CONF_THRESHOLD_GLOBAL
)

displayed_df = type_val_eval_df[type_val_eval_df["display_type_global_threshold"]].copy()
hidden_df = type_val_eval_df[~type_val_eval_df["display_type_global_threshold"]].copy()

confidence_threshold_summary = {
    "global_confidence_threshold": TYPE_CONF_THRESHOLD_GLOBAL,
    "total_rows": int(len(type_val_eval_df)),
    "displayed_rows": int(len(displayed_df)),
    "hidden_rows": int(len(hidden_df)),
    "displayed_percent": float(len(displayed_df) / len(type_val_eval_df) * 100),
    "hidden_percent": float(len(hidden_df) / len(type_val_eval_df) * 100),
    "accuracy_all_rows": float(type_val_eval_df["type_correct"].mean()),
    "accuracy_displayed_rows": float(displayed_df["type_correct"].mean()) if len(displayed_df) else np.nan,
    "accuracy_hidden_rows": float(hidden_df["type_correct"].mean()) if len(hidden_df) else np.nan,
    "mean_confidence_all": float(type_val_eval_df["type_conf"].mean()),
    "median_confidence_all": float(type_val_eval_df["type_conf"].median()),
    "p10_confidence_all": float(type_val_eval_df["type_conf"].quantile(0.10)),
    "p90_confidence_all": float(type_val_eval_df["type_conf"].quantile(0.90)),
}

confidence_by_prediction = (
    type_val_eval_df
    .groupby("type_pred")
    .agg(
        rows=("type_pred", "size"),
        accuracy=("type_correct", "mean"),
        mean_conf=("type_conf", "mean"),
        median_conf=("type_conf", "median"),
        p10_conf=("type_conf", lambda x: x.quantile(0.10)),
        p90_conf=("type_conf", lambda x: x.quantile(0.90)),
        displayed_percent=("display_type_global_threshold", lambda x: x.mean() * 100),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

confidence_by_true_type = (
    type_val_eval_df
    .groupby("next_target_type")
    .agg(
        rows=("next_target_type", "size"),
        accuracy=("type_correct", "mean"),
        mean_conf=("type_conf", "mean"),
        median_conf=("type_conf", "median"),
        displayed_percent=("display_type_global_threshold", lambda x: x.mean() * 100),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

# ------------------------------------------------------------
# Threshold sweep
# ------------------------------------------------------------

threshold_rows = []

for th in np.arange(0.30, 0.951, 0.05):
    mask = type_val_eval_df["type_conf"] >= th
    subset = type_val_eval_df[mask]

    threshold_rows.append({
        "threshold": round(float(th), 2),
        "displayed_rows": int(mask.sum()),
        "displayed_percent": float(mask.mean() * 100),
        "accuracy_displayed": float(subset["type_correct"].mean()) if len(subset) else np.nan,
        "hidden_percent": float((~mask).mean() * 100),
    })

threshold_sweep_df = pd.DataFrame(threshold_rows)

# ------------------------------------------------------------
# Optional per-class display threshold
# Goal: maintain at least target precision when possible.
# These are saved later for future use, but Phase 5 can still use global 0.60.
# ------------------------------------------------------------

PER_CLASS_TARGET_PRECISION = 0.80
PER_CLASS_MIN_DISPLAY_PERCENT = 5.0

per_class_threshold_rows = []
type_conf_threshold_by_class = {}

for c in type_classes:
    pred_c_df = type_val_eval_df[type_val_eval_df["type_pred"].eq(c)].copy()

    if len(pred_c_df) == 0:
        type_conf_threshold_by_class[c] = TYPE_CONF_THRESHOLD_GLOBAL
        per_class_threshold_rows.append({
            "class": c,
            "predicted_rows": 0,
            "selected_threshold": TYPE_CONF_THRESHOLD_GLOBAL,
            "precision_at_threshold": np.nan,
            "displayed_percent_within_predicted_class": 0.0,
            "reason": "no_predictions",
        })
        continue

    selected_threshold = TYPE_CONF_THRESHOLD_GLOBAL
    selected_precision = float(pred_c_df["type_correct"].mean())
    selected_display_percent = 100.0
    reason = "global_threshold_default"

    for th in np.arange(0.30, 0.951, 0.01):
        mask = pred_c_df["type_conf"] >= th
        display_percent = float(mask.mean() * 100)

        if mask.sum() == 0:
            continue

        precision = float(pred_c_df.loc[mask, "type_correct"].mean())

        if precision >= PER_CLASS_TARGET_PRECISION and display_percent >= PER_CLASS_MIN_DISPLAY_PERCENT:
            selected_threshold = round(float(th), 2)
            selected_precision = precision
            selected_display_percent = display_percent
            reason = "meets_precision_target"
            break

    type_conf_threshold_by_class[c] = selected_threshold

    per_class_threshold_rows.append({
        "class": c,
        "predicted_rows": int(len(pred_c_df)),
        "selected_threshold": selected_threshold,
        "precision_at_threshold": selected_precision,
        "displayed_percent_within_predicted_class": selected_display_percent,
        "reason": reason,
    })

per_class_threshold_df = pd.DataFrame(per_class_threshold_rows)

# ------------------------------------------------------------
# Compare against baseline
# ------------------------------------------------------------

type_validation_comparison = pd.DataFrame([
    {
        "model": "best_baseline_" + best_type_baseline_name,
        "accuracy": best_type_baseline_val["accuracy"],
        "macro_f1": best_type_baseline_val["macro_f1"],
        "weighted_f1": best_type_baseline_val["weighted_f1"],
    },
    {
        "model": "lgbm_type_selected",
        "accuracy": best_type_row["accuracy"],
        "macro_f1": best_type_row["macro_f1"],
        "weighted_f1": best_type_row["weighted_f1"],
    },
    {
        "model": f"lgbm_type_selected_displayed_conf_ge_{TYPE_CONF_THRESHOLD_GLOBAL}",
        "accuracy": confidence_threshold_summary["accuracy_displayed_rows"],
        "macro_f1": np.nan,
        "weighted_f1": np.nan,
    },
])

# ------------------------------------------------------------
# Validation checks
# ------------------------------------------------------------

type_validation_checks = []

def add_type_check(name, passed, detail, severity="WARNING"):
    type_validation_checks.append({
        "check": name,
        "severity": severity,
        "passed": bool(passed),
        "detail": detail,
    })

add_type_check(
    "type_classifier_weighted_f1_at_least_baseline_minus_2pt",
    best_type_row["weighted_f1"] >= best_type_baseline_val["weighted_f1"] - 0.02,
    f"lgbm_weighted_f1={best_type_row['weighted_f1']:.4f}, baseline_weighted_f1={best_type_baseline_val['weighted_f1']:.4f}",
)

add_type_check(
    "type_classifier_macro_f1_at_least_baseline_minus_2pt",
    best_type_row["macro_f1"] >= best_type_baseline_val["macro_f1"] - 0.02,
    f"lgbm_macro_f1={best_type_row['macro_f1']:.4f}, baseline_macro_f1={best_type_baseline_val['macro_f1']:.4f}",
)

add_type_check(
    "displayed_accuracy_above_all_row_accuracy",
    confidence_threshold_summary["accuracy_displayed_rows"] >= confidence_threshold_summary["accuracy_all_rows"],
    f"displayed_acc={confidence_threshold_summary['accuracy_displayed_rows']:.4f}, all_acc={confidence_threshold_summary['accuracy_all_rows']:.4f}",
)

add_type_check(
    "global_threshold_displays_some_rows",
    confidence_threshold_summary["displayed_rows"] > 0,
    f"displayed_rows={confidence_threshold_summary['displayed_rows']:,}",
    severity="ERROR",
)

add_type_check(
    "all_classes_known",
    set(type_classes) == set(TARGET_STATUSES),
    f"type_classes={type_classes}, target_statuses={sorted(TARGET_STATUSES)}",
    severity="ERROR",
)

type_validation_checks_df = pd.DataFrame(type_validation_checks)

failed_type_errors = type_validation_checks_df[
    (type_validation_checks_df["severity"] == "ERROR") & (~type_validation_checks_df["passed"])
]

READY_FOR_BATCH_9 = len(failed_type_errors) == 0

print("=" * 100)
print("P4-29 — CLASSIFICATION AND CONFIDENCE REPORT")
print("=" * 100)

print("\nType selection summary:")
print(json.dumps(type_selection_summary, indent=2, ensure_ascii=False))

print("\nType vs baseline:")
print(json.dumps(type_vs_baseline, indent=2, ensure_ascii=False))

print("\nType class frequency report:")
display(type_class_frequency_report)

print("\nType validation comparison:")
display(type_validation_comparison)

print("\nFull validation classification report:")
display(type_val_report_df)

print("\nValidation confusion matrix:")
display(type_val_confusion_df)

print("\nConfidence threshold summary:")
print(json.dumps(confidence_threshold_summary, indent=2, ensure_ascii=False))

print("\nConfidence by predicted type:")
display(confidence_by_prediction)

print("\nConfidence by true next target type:")
display(confidence_by_true_type)

print("\nThreshold sweep:")
display(threshold_sweep_df)

print("\nOptional per-class display thresholds:")
display(per_class_threshold_df)

print("\nType validation checks:")
display(type_validation_checks_df)

print("\nTop 30 low-confidence validation predictions:")
display(
    type_val_eval_df
    .sort_values("type_conf", ascending=True)
    .head(30)
)

print("\nTop 30 high-confidence wrong validation predictions:")
display(
    type_val_eval_df[
        ~type_val_eval_df["type_correct"]
    ]
    .sort_values("type_conf", ascending=False)
    .head(30)
)

print("=" * 100)
if READY_FOR_BATCH_9:
    print("READY_FOR_BATCH_9 = True")
    print("Batch 8 complete. Send this output before moving to Batch 9.")
else:
    print("READY_FOR_BATCH_9 = False")
    print("Batch 8 has failed ERROR gates. Fix before moving to Batch 9.")
print("=" * 100)

P4-30 — Final ETA test evaluation

In [ ]:
# ============================================================
# P4-30 — Final ETA test evaluation
# ============================================================

def predict_lgbm_eta_model(model, X, target_mode, clip_upper):
    pred_model_space = model.predict(X, num_iteration=model.best_iteration_)
    pred_sec = inverse_transform_pred(pred_model_space, target_mode)
    pred_sec = clean_eta_pred(pred_sec, clip_upper=clip_upper)
    return pred_sec


# ------------------------------------------------------------
# Raw selected model predictions on test
# ------------------------------------------------------------

p50_pred_test_base = predict_lgbm_eta_model(
    model=lgbm_p50_final,
    X=X_test,
    target_mode=p50_target_mode_final,
    clip_upper=clip_max,
)

p90_pred_test_base = predict_lgbm_eta_model(
    model=lgbm_p90_final,
    X=X_test,
    target_mode=p90_target_mode_final,
    clip_upper=clip_max,
)

p90_pred_test_base = np.maximum(
    p90_pred_test_base,
    p50_pred_test_base + 1.0,
)

# ------------------------------------------------------------
# Apply P90 calibration
# ------------------------------------------------------------

p90_pred_test_calibrated_raw, p90_test_multipliers = apply_p90_calibration(
    df=test_df,
    p90_base_pred=p90_pred_test_base,
    global_multiplier=p90_multiplier_global,
    by_machine=p90_multiplier_by_machine,
)

p90_pred_test_calibrated = np.maximum(
    p90_pred_test_calibrated_raw,
    p50_pred_test_base + 1.0,
)

# ------------------------------------------------------------
# Apply P50 calibration
# ------------------------------------------------------------

p50_pred_test_calibrated_raw, p50_test_scales = apply_p50_calibration(
    df=test_df,
    p50_base_pred=p50_pred_test_base,
    global_scale=p50_scale_global,
    by_machine=p50_scale_by_machine,
)

p50_pred_test_calibrated = clean_eta_pred(
    p50_pred_test_calibrated_raw,
    clip_upper=clip_max,
)

p90_pred_test_calibrated = np.maximum(
    p90_pred_test_calibrated,
    p50_pred_test_calibrated + 1.0,
)

# ------------------------------------------------------------
# Apply post-processing
# ------------------------------------------------------------

p50_pred_test_final, p90_pred_test_final, eta_bound_source_test = apply_eta_postprocess(
    df=test_df,
    p50_pred=p50_pred_test_calibrated,
    p90_pred=p90_pred_test_calibrated,
    maps=eta_postprocess_maps,
)

# ------------------------------------------------------------
# Evaluate ETA on test
# ------------------------------------------------------------

p50_test_base_metrics = evaluate_eta_predictions(
    y_true_raw=y_test_raw,
    y_pred=p50_pred_test_base,
    name="lgbm_p50_base",
    split="test",
)

p50_test_final_metrics = evaluate_eta_predictions(
    y_true_raw=y_test_raw,
    y_pred=p50_pred_test_final,
    name="lgbm_p50_final_calibrated_postprocessed",
    split="test",
)

p90_test_base_metrics = evaluate_p90_interval(
    y_true_raw=y_test_raw,
    y_true_clip=y_test_clip,
    p50_pred=p50_pred_test_base,
    p90_pred=p90_pred_test_base,
    split="test",
)

p90_test_final_metrics = evaluate_p90_interval(
    y_true_raw=y_test_raw,
    y_true_clip=y_test_clip,
    p50_pred=p50_pred_test_final,
    p90_pred=p90_pred_test_final,
    split="test",
)

p90_test_final_metrics.update({
    "pinball90_clip": pinball_loss(y_test_clip, p90_pred_test_final, 0.90),
})

# Best test baseline using the validation-selected baseline name
best_eta_test_baseline_pred = eta_baseline_pred_test[best_eta_baseline_name].to_numpy()

best_eta_test_baseline_metrics = evaluate_eta_predictions(
    y_true_raw=y_test_raw,
    y_pred=best_eta_test_baseline_pred,
    name="best_baseline_" + best_eta_baseline_name,
    split="test",
)

eta_test_comparison = pd.DataFrame([
    {
        "stage": "best_baseline_" + best_eta_baseline_name,
        "P50_MedAE_sec": best_eta_test_baseline_metrics["MedAE_sec"],
        "P50_MAE_sec": best_eta_test_baseline_metrics["MAE_sec"],
        "P50_RMSE_sec": best_eta_test_baseline_metrics["RMSE_sec"],
        "P50_Hit_2m_percent": best_eta_test_baseline_metrics["Hit_2m_percent"],
        "P50_Hit_5m_percent": best_eta_test_baseline_metrics["Hit_5m_percent"],
        "P50_Hit_10m_percent": best_eta_test_baseline_metrics["Hit_10m_percent"],
        "P90_coverage_raw_percent": np.nan,
        "P90_median_width_sec": np.nan,
        "P90_p90_width_sec": np.nan,
    },
    {
        "stage": "lgbm_raw_selected",
        "P50_MedAE_sec": p50_test_base_metrics["MedAE_sec"],
        "P50_MAE_sec": p50_test_base_metrics["MAE_sec"],
        "P50_RMSE_sec": p50_test_base_metrics["RMSE_sec"],
        "P50_Hit_2m_percent": p50_test_base_metrics["Hit_2m_percent"],
        "P50_Hit_5m_percent": p50_test_base_metrics["Hit_5m_percent"],
        "P50_Hit_10m_percent": p50_test_base_metrics["Hit_10m_percent"],
        "P90_coverage_raw_percent": p90_test_base_metrics["P90_coverage_raw_percent"],
        "P90_median_width_sec": p90_test_base_metrics["P90_median_width_sec"],
        "P90_p90_width_sec": p90_test_base_metrics["P90_p90_width_sec"],
    },
    {
        "stage": "lgbm_final_calibrated_postprocessed",
        "P50_MedAE_sec": p50_test_final_metrics["MedAE_sec"],
        "P50_MAE_sec": p50_test_final_metrics["MAE_sec"],
        "P50_RMSE_sec": p50_test_final_metrics["RMSE_sec"],
        "P50_Hit_2m_percent": p50_test_final_metrics["Hit_2m_percent"],
        "P50_Hit_5m_percent": p50_test_final_metrics["Hit_5m_percent"],
        "P50_Hit_10m_percent": p50_test_final_metrics["Hit_10m_percent"],
        "P90_coverage_raw_percent": p90_test_final_metrics["P90_coverage_raw_percent"],
        "P90_median_width_sec": p90_test_final_metrics["P90_median_width_sec"],
        "P90_p90_width_sec": p90_test_final_metrics["P90_p90_width_sec"],
    },
])

test_eta_final_df = test_df[
    ["process", "mc_no", "occurred", "mc_status", "next_target_type", "y_time_sec", "y_time_sec_clip"]
].copy()

test_eta_final_df["p50_pred_sec"] = p50_pred_test_final
test_eta_final_df["p90_pred_sec"] = p90_pred_test_final
test_eta_final_df["abs_error_p50_sec"] = np.abs(test_eta_final_df["y_time_sec"] - test_eta_final_df["p50_pred_sec"])
test_eta_final_df["p90_covered_raw"] = test_eta_final_df["y_time_sec"] <= test_eta_final_df["p90_pred_sec"]
test_eta_final_df["p90_width_sec"] = test_eta_final_df["p90_pred_sec"] - test_eta_final_df["p50_pred_sec"]
test_eta_final_df["eta_bound_source"] = eta_bound_source_test.values

eta_test_by_current_status = (
    test_eta_final_df
    .groupby("mc_status")
    .agg(
        rows=("mc_status", "size"),
        actual_median_sec=("y_time_sec", "median"),
        actual_p90_sec=("y_time_sec", lambda x: x.quantile(0.90)),
        p50_pred_median_sec=("p50_pred_sec", "median"),
        p90_pred_median_sec=("p90_pred_sec", "median"),
        MedAE_sec=("abs_error_p50_sec", "median"),
        MAE_sec=("abs_error_p50_sec", "mean"),
        Hit_5m_percent=("abs_error_p50_sec", lambda x: (x <= 300).mean() * 100),
        Hit_10m_percent=("abs_error_p50_sec", lambda x: (x <= 600).mean() * 100),
        P90_coverage_raw_percent=("p90_covered_raw", lambda x: x.mean() * 100),
        P90_median_width_sec=("p90_width_sec", "median"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

eta_test_by_next_target_type = (
    test_eta_final_df
    .groupby("next_target_type")
    .agg(
        rows=("next_target_type", "size"),
        actual_median_sec=("y_time_sec", "median"),
        actual_p90_sec=("y_time_sec", lambda x: x.quantile(0.90)),
        p50_pred_median_sec=("p50_pred_sec", "median"),
        p90_pred_median_sec=("p90_pred_sec", "median"),
        MedAE_sec=("abs_error_p50_sec", "median"),
        MAE_sec=("abs_error_p50_sec", "mean"),
        Hit_5m_percent=("abs_error_p50_sec", lambda x: (x <= 300).mean() * 100),
        Hit_10m_percent=("abs_error_p50_sec", lambda x: (x <= 600).mean() * 100),
        P90_coverage_raw_percent=("p90_covered_raw", lambda x: x.mean() * 100),
        P90_median_width_sec=("p90_width_sec", "median"),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

eta_test_by_machine = (
    test_eta_final_df
    .groupby(["process", "mc_no"])
    .agg(
        rows=("mc_no", "size"),
        actual_median_sec=("y_time_sec", "median"),
        actual_p90_sec=("y_time_sec", lambda x: x.quantile(0.90)),
        p50_pred_median_sec=("p50_pred_sec", "median"),
        MedAE_sec=("abs_error_p50_sec", "median"),
        Hit_5m_percent=("abs_error_p50_sec", lambda x: (x <= 300).mean() * 100),
        Hit_10m_percent=("abs_error_p50_sec", lambda x: (x <= 600).mean() * 100),
        P90_coverage_raw_percent=("p90_covered_raw", lambda x: x.mean() * 100),
        P90_median_width_sec=("p90_width_sec", "median"),
    )
    .reset_index()
)

eta_test_by_machine_n200 = eta_test_by_machine[eta_test_by_machine["rows"] >= 200].copy()

eta_test_summary = {
    "best_eta_test_baseline": best_eta_baseline_name,
    "baseline_test_MedAE_sec": float(best_eta_test_baseline_metrics["MedAE_sec"]),
    "baseline_test_Hit_5m_percent": float(best_eta_test_baseline_metrics["Hit_5m_percent"]),
    "baseline_test_Hit_10m_percent": float(best_eta_test_baseline_metrics["Hit_10m_percent"]),

    "p50_test_MedAE_sec": float(p50_test_final_metrics["MedAE_sec"]),
    "p50_test_MAE_sec": float(p50_test_final_metrics["MAE_sec"]),
    "p50_test_RMSE_sec": float(p50_test_final_metrics["RMSE_sec"]),
    "p50_test_Hit_2m_percent": float(p50_test_final_metrics["Hit_2m_percent"]),
    "p50_test_Hit_5m_percent": float(p50_test_final_metrics["Hit_5m_percent"]),
    "p50_test_Hit_10m_percent": float(p50_test_final_metrics["Hit_10m_percent"]),

    "p90_test_coverage_raw_percent": float(p90_test_final_metrics["P90_coverage_raw_percent"]),
    "p90_test_coverage_clip_percent": float(p90_test_final_metrics["P90_coverage_clip_percent"]),
    "p90_test_median_width_sec": float(p90_test_final_metrics["P90_median_width_sec"]),
    "p90_test_p90_width_sec": float(p90_test_final_metrics["P90_p90_width_sec"]),

    "test_machine_n200_count": int(len(eta_test_by_machine_n200)),
    "test_machine_n200_min_p90_coverage": float(eta_test_by_machine_n200["P90_coverage_raw_percent"].min()) if len(eta_test_by_machine_n200) else np.nan,
    "test_machine_n200_median_p90_coverage": float(eta_test_by_machine_n200["P90_coverage_raw_percent"].median()) if len(eta_test_by_machine_n200) else np.nan,
}

print("=" * 100)
print("P4-30 — FINAL ETA TEST EVALUATION")
print("=" * 100)

print("\nETA test summary:")
print(json.dumps(eta_test_summary, indent=2, ensure_ascii=False))

print("\nETA test comparison:")
display(eta_test_comparison)

print("\nETA test by current status:")
display(eta_test_by_current_status)

print("\nETA test by next target type:")
display(eta_test_by_next_target_type)

print("\nPer-machine ETA test summary:")
display(
    eta_test_by_machine[
        ["rows", "MedAE_sec", "Hit_5m_percent", "Hit_10m_percent", "P90_coverage_raw_percent", "P90_median_width_sec"]
    ].describe()
)

print("\nPer-machine ETA test summary for n>=200:")
display(
    eta_test_by_machine_n200[
        ["rows", "MedAE_sec", "Hit_5m_percent", "Hit_10m_percent", "P90_coverage_raw_percent", "P90_median_width_sec"]
    ].describe()
)

print("\nLowest 30 test P90 coverage machines, n>=200:")
display(
    eta_test_by_machine_n200
    .sort_values("P90_coverage_raw_percent", ascending=True)
    .head(30)
)

print("=" * 100)
print("P4-30 ready")
print("=" * 100)

P4-31 — Final next-type test evaluation

In [ ]:
# ============================================================
# P4-31 — Final next-type test evaluation
# ============================================================

type_proba_test = lgbm_type_final.predict_proba(
    X_test,
    num_iteration=lgbm_type_final.best_iteration_,
)

type_pred_test_enc = np.argmax(type_proba_test, axis=1)
type_conf_test = np.max(type_proba_test, axis=1)
type_pred_test = type_label_encoder.inverse_transform(type_pred_test_enc)

type_test_metrics = evaluate_type_predictions(
    y_true=y_type_test,
    y_pred=type_pred_test,
    name="lgbm_type_final",
    split="test",
)

# Test baseline selected from validation
best_type_test_baseline_pred = type_baseline_pred_test[best_type_baseline_name].astype(str).to_numpy()

best_type_test_baseline_metrics = evaluate_type_predictions(
    y_true=y_type_test,
    y_pred=best_type_test_baseline_pred,
    name="best_baseline_" + best_type_baseline_name,
    split="test",
)

type_test_report_dict = classification_report(
    y_type_test,
    type_pred_test,
    labels=type_classes,
    output_dict=True,
    zero_division=0,
)

type_test_report_df = pd.DataFrame(type_test_report_dict).T

type_test_confusion_df = pd.DataFrame(
    confusion_matrix(y_type_test, type_pred_test, labels=type_classes),
    index=[f"true_{c}" for c in type_classes],
    columns=[f"pred_{c}" for c in type_classes],
)

type_test_eval_df = test_df[
    ["process", "mc_no", "occurred", "mc_status", "next_target_type", "y_time_sec"]
].copy()

type_test_eval_df["type_pred"] = type_pred_test
type_test_eval_df["type_conf"] = type_conf_test
type_test_eval_df["type_correct"] = (
    type_test_eval_df["type_pred"].astype(str)
    == type_test_eval_df["next_target_type"].astype(str)
)

type_test_eval_df["display_type_global_threshold"] = (
    type_test_eval_df["type_conf"] >= TYPE_CONF_THRESHOLD_GLOBAL
)

type_test_displayed_df = type_test_eval_df[type_test_eval_df["display_type_global_threshold"]].copy()
type_test_hidden_df = type_test_eval_df[~type_test_eval_df["display_type_global_threshold"]].copy()

type_test_confidence_summary = {
    "global_confidence_threshold": TYPE_CONF_THRESHOLD_GLOBAL,
    "total_rows": int(len(type_test_eval_df)),
    "displayed_rows": int(len(type_test_displayed_df)),
    "hidden_rows": int(len(type_test_hidden_df)),
    "displayed_percent": float(len(type_test_displayed_df) / len(type_test_eval_df) * 100),
    "hidden_percent": float(len(type_test_hidden_df) / len(type_test_eval_df) * 100),
    "accuracy_all_rows": float(type_test_eval_df["type_correct"].mean()),
    "accuracy_displayed_rows": float(type_test_displayed_df["type_correct"].mean()) if len(type_test_displayed_df) else np.nan,
    "accuracy_hidden_rows": float(type_test_hidden_df["type_correct"].mean()) if len(type_test_hidden_df) else np.nan,
    "mean_confidence_all": float(type_test_eval_df["type_conf"].mean()),
    "median_confidence_all": float(type_test_eval_df["type_conf"].median()),
    "p10_confidence_all": float(type_test_eval_df["type_conf"].quantile(0.10)),
    "p90_confidence_all": float(type_test_eval_df["type_conf"].quantile(0.90)),
}

type_test_by_prediction = (
    type_test_eval_df
    .groupby("type_pred")
    .agg(
        rows=("type_pred", "size"),
        accuracy=("type_correct", "mean"),
        mean_conf=("type_conf", "mean"),
        median_conf=("type_conf", "median"),
        p10_conf=("type_conf", lambda x: x.quantile(0.10)),
        p90_conf=("type_conf", lambda x: x.quantile(0.90)),
        displayed_percent=("display_type_global_threshold", lambda x: x.mean() * 100),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

type_test_by_true_type = (
    type_test_eval_df
    .groupby("next_target_type")
    .agg(
        rows=("next_target_type", "size"),
        accuracy=("type_correct", "mean"),
        mean_conf=("type_conf", "mean"),
        median_conf=("type_conf", "median"),
        displayed_percent=("display_type_global_threshold", lambda x: x.mean() * 100),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

type_test_threshold_rows = []

for th in np.arange(0.30, 0.951, 0.05):
    mask = type_test_eval_df["type_conf"] >= th
    subset = type_test_eval_df[mask]

    type_test_threshold_rows.append({
        "threshold": round(float(th), 2),
        "displayed_rows": int(mask.sum()),
        "displayed_percent": float(mask.mean() * 100),
        "accuracy_displayed": float(subset["type_correct"].mean()) if len(subset) else np.nan,
        "hidden_percent": float((~mask).mean() * 100),
    })

type_test_threshold_sweep_df = pd.DataFrame(type_test_threshold_rows)

type_test_comparison = pd.DataFrame([
    {
        "model": "best_baseline_" + best_type_baseline_name,
        "accuracy": best_type_test_baseline_metrics["accuracy"],
        "macro_f1": best_type_test_baseline_metrics["macro_f1"],
        "weighted_f1": best_type_test_baseline_metrics["weighted_f1"],
    },
    {
        "model": "lgbm_type_final",
        "accuracy": type_test_metrics["accuracy"],
        "macro_f1": type_test_metrics["macro_f1"],
        "weighted_f1": type_test_metrics["weighted_f1"],
    },
    {
        "model": f"lgbm_type_final_displayed_conf_ge_{TYPE_CONF_THRESHOLD_GLOBAL}",
        "accuracy": type_test_confidence_summary["accuracy_displayed_rows"],
        "macro_f1": np.nan,
        "weighted_f1": np.nan,
    },
])

type_test_summary = {
    "best_type_test_baseline": best_type_baseline_name,
    "baseline_test_accuracy": float(best_type_test_baseline_metrics["accuracy"]),
    "baseline_test_macro_f1": float(best_type_test_baseline_metrics["macro_f1"]),
    "baseline_test_weighted_f1": float(best_type_test_baseline_metrics["weighted_f1"]),

    "type_test_accuracy": float(type_test_metrics["accuracy"]),
    "type_test_macro_f1": float(type_test_metrics["macro_f1"]),
    "type_test_weighted_f1": float(type_test_metrics["weighted_f1"]),

    "type_test_accuracy_improvement_point": float(type_test_metrics["accuracy"] - best_type_test_baseline_metrics["accuracy"]),
    "type_test_macro_f1_improvement_point": float(type_test_metrics["macro_f1"] - best_type_test_baseline_metrics["macro_f1"]),
    "type_test_weighted_f1_improvement_point": float(type_test_metrics["weighted_f1"] - best_type_test_baseline_metrics["weighted_f1"]),

    "displayed_accuracy_at_global_threshold": float(type_test_confidence_summary["accuracy_displayed_rows"]),
    "displayed_percent_at_global_threshold": float(type_test_confidence_summary["displayed_percent"]),
}

print("=" * 100)
print("P4-31 — FINAL NEXT-TYPE TEST EVALUATION")
print("=" * 100)

print("\nType test summary:")
print(json.dumps(type_test_summary, indent=2, ensure_ascii=False))

print("\nType test comparison:")
display(type_test_comparison)

print("\nFull test classification report:")
display(type_test_report_df)

print("\nTest confusion matrix:")
display(type_test_confusion_df)

print("\nTest confidence summary:")
print(json.dumps(type_test_confidence_summary, indent=2, ensure_ascii=False))

print("\nTest confidence by predicted type:")
display(type_test_by_prediction)

print("\nTest confidence by true next target type:")
display(type_test_by_true_type)

print("\nTest threshold sweep:")
display(type_test_threshold_sweep_df)

print("\nTop 30 high-confidence wrong test predictions:")
display(
    type_test_eval_df[
        ~type_test_eval_df["type_correct"]
    ]
    .sort_values("type_conf", ascending=False)
    .head(30)
)

print("=" * 100)
print("P4-31 ready")
print("=" * 100)

P4-32 — Final acceptance gate report

In [ ]:
# ============================================================
# P4-32 — Final acceptance gate report
# ============================================================

ACCEPTANCE_GATES = {
    "eta_medae_max_sec": 60.0,
    "eta_hit_5m_min_percent": 75.0,
    "eta_hit_10m_min_percent": 80.0,
    "p90_global_min_percent": 88.0,
    "p90_global_max_percent": 93.0,
    "p90_machine_n200_min_percent": 84.0,
    "type_display_conf_threshold": TYPE_CONF_THRESHOLD_GLOBAL,
    "type_displayed_accuracy_min_percent": 90.0,
    "type_displayed_percent_min": 50.0,
}

acceptance_rows = []

def add_acceptance_gate(name, passed, value, threshold, detail, severity="ERROR"):
    acceptance_rows.append({
        "gate": name,
        "severity": severity,
        "passed": bool(passed),
        "value": value,
        "threshold": threshold,
        "detail": detail,
    })


# ETA gates
add_acceptance_gate(
    "ETA MedAE <= gate",
    p50_test_final_metrics["MedAE_sec"] <= ACCEPTANCE_GATES["eta_medae_max_sec"],
    float(p50_test_final_metrics["MedAE_sec"]),
    f"<= {ACCEPTANCE_GATES['eta_medae_max_sec']} sec",
    f"Test MedAE={p50_test_final_metrics['MedAE_sec']:.3f}s",
)

add_acceptance_gate(
    "ETA Hit@±5m >= gate",
    p50_test_final_metrics["Hit_5m_percent"] >= ACCEPTANCE_GATES["eta_hit_5m_min_percent"],
    float(p50_test_final_metrics["Hit_5m_percent"]),
    f">= {ACCEPTANCE_GATES['eta_hit_5m_min_percent']}%",
    f"Test Hit@5m={p50_test_final_metrics['Hit_5m_percent']:.3f}%",
)

add_acceptance_gate(
    "ETA Hit@±10m >= gate",
    p50_test_final_metrics["Hit_10m_percent"] >= ACCEPTANCE_GATES["eta_hit_10m_min_percent"],
    float(p50_test_final_metrics["Hit_10m_percent"]),
    f">= {ACCEPTANCE_GATES['eta_hit_10m_min_percent']}%",
    f"Test Hit@10m={p50_test_final_metrics['Hit_10m_percent']:.3f}%",
)

add_acceptance_gate(
    "P90 global coverage within gate",
    (
        ACCEPTANCE_GATES["p90_global_min_percent"]
        <= p90_test_final_metrics["P90_coverage_raw_percent"]
        <= ACCEPTANCE_GATES["p90_global_max_percent"]
    ),
    float(p90_test_final_metrics["P90_coverage_raw_percent"]),
    f"{ACCEPTANCE_GATES['p90_global_min_percent']}–{ACCEPTANCE_GATES['p90_global_max_percent']}%",
    f"Test P90 global coverage={p90_test_final_metrics['P90_coverage_raw_percent']:.3f}%",
)

# Per-machine P90 gate.
# This is often noisy with short data windows, so keep as WARNING.
if len(eta_test_by_machine_n200) > 0:
    machine_n200_min_cov = float(eta_test_by_machine_n200["P90_coverage_raw_percent"].min())
    machine_n200_median_cov = float(eta_test_by_machine_n200["P90_coverage_raw_percent"].median())
else:
    machine_n200_min_cov = np.nan
    machine_n200_median_cov = np.nan

add_acceptance_gate(
    "P90 per-machine n>=200 min coverage >= gate",
    pd.isna(machine_n200_min_cov) or machine_n200_min_cov >= ACCEPTANCE_GATES["p90_machine_n200_min_percent"],
    machine_n200_min_cov,
    f">= {ACCEPTANCE_GATES['p90_machine_n200_min_percent']}%",
    f"n>=200 machines={len(eta_test_by_machine_n200)}, min={machine_n200_min_cov}, median={machine_n200_median_cov}",
    severity="WARNING",
)

add_acceptance_gate(
    "P90 per-machine n>=200 median coverage >= gate",
    pd.isna(machine_n200_median_cov) or machine_n200_median_cov >= ACCEPTANCE_GATES["p90_machine_n200_min_percent"],
    machine_n200_median_cov,
    f">= {ACCEPTANCE_GATES['p90_machine_n200_min_percent']}%",
    f"n>=200 machines={len(eta_test_by_machine_n200)}, median={machine_n200_median_cov}",
    severity="WARNING",
)

# Type gates
add_acceptance_gate(
    "Type displayed accuracy >= gate",
    type_test_confidence_summary["accuracy_displayed_rows"] * 100 >= ACCEPTANCE_GATES["type_displayed_accuracy_min_percent"],
    float(type_test_confidence_summary["accuracy_displayed_rows"] * 100),
    f">= {ACCEPTANCE_GATES['type_displayed_accuracy_min_percent']}%",
    f"Displayed type accuracy={type_test_confidence_summary['accuracy_displayed_rows'] * 100:.3f}% at conf >= {TYPE_CONF_THRESHOLD_GLOBAL}",
)

add_acceptance_gate(
    "Type displayed percent >= gate",
    type_test_confidence_summary["displayed_percent"] >= ACCEPTANCE_GATES["type_displayed_percent_min"],
    float(type_test_confidence_summary["displayed_percent"]),
    f">= {ACCEPTANCE_GATES['type_displayed_percent_min']}%",
    f"Displayed percent={type_test_confidence_summary['displayed_percent']:.3f}% at conf >= {TYPE_CONF_THRESHOLD_GLOBAL}",
)

add_acceptance_gate(
    "Type classifier weighted F1 beats baseline",
    type_test_metrics["weighted_f1"] >= best_type_test_baseline_metrics["weighted_f1"],
    float(type_test_metrics["weighted_f1"]),
    f">= baseline {best_type_test_baseline_metrics['weighted_f1']:.4f}",
    f"LGBM weighted F1={type_test_metrics['weighted_f1']:.4f}, baseline={best_type_test_baseline_metrics['weighted_f1']:.4f}",
    severity="WARNING",
)

add_acceptance_gate(
    "Type classifier macro F1 beats baseline",
    type_test_metrics["macro_f1"] >= best_type_test_baseline_metrics["macro_f1"],
    float(type_test_metrics["macro_f1"]),
    f">= baseline {best_type_test_baseline_metrics['macro_f1']:.4f}",
    f"LGBM macro F1={type_test_metrics['macro_f1']:.4f}, baseline={best_type_test_baseline_metrics['macro_f1']:.4f}",
    severity="WARNING",
)

final_acceptance_df = pd.DataFrame(acceptance_rows)

failed_error_gates = final_acceptance_df[
    (final_acceptance_df["severity"].eq("ERROR")) &
    (~final_acceptance_df["passed"])
]

failed_warning_gates = final_acceptance_df[
    (final_acceptance_df["severity"].eq("WARNING")) &
    (~final_acceptance_df["passed"])
]

if len(failed_error_gates) == 0 and len(failed_warning_gates) == 0:
    DEPLOYMENT_READINESS = "PASS"
elif len(failed_error_gates) == 0 and len(failed_warning_gates) > 0:
    DEPLOYMENT_READINESS = "PASS_WITH_WARNINGS"
else:
    DEPLOYMENT_READINESS = "FAIL"

final_model_card_summary = {
    "process_id": CFG.process_id,
    "rows_total_labeled": int(len(df_labeled)),
    "train_rows": int(len(train_df)),
    "val_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "feature_count": int(len(feature_cols)),
    "normal_statuses": sorted(NORMAL_STATUSES),
    "target_statuses": sorted(TARGET_STATUSES),

    "selected_p50_model": best_p50_name,
    "selected_p50_target_mode": p50_target_mode_final,
    "selected_p90_model": best_p90_name,
    "selected_p90_target_mode": p90_target_mode_final,
    "selected_type_model": best_type_name,

    "test_eta_medae_sec": float(p50_test_final_metrics["MedAE_sec"]),
    "test_eta_mae_sec": float(p50_test_final_metrics["MAE_sec"]),
    "test_eta_hit_5m_percent": float(p50_test_final_metrics["Hit_5m_percent"]),
    "test_eta_hit_10m_percent": float(p50_test_final_metrics["Hit_10m_percent"]),
    "test_p90_coverage_percent": float(p90_test_final_metrics["P90_coverage_raw_percent"]),
    "test_p90_median_width_sec": float(p90_test_final_metrics["P90_median_width_sec"]),

    "test_type_accuracy": float(type_test_metrics["accuracy"]),
    "test_type_macro_f1": float(type_test_metrics["macro_f1"]),
    "test_type_weighted_f1": float(type_test_metrics["weighted_f1"]),
    "test_type_displayed_accuracy_percent": float(type_test_confidence_summary["accuracy_displayed_rows"] * 100),
    "test_type_displayed_percent": float(type_test_confidence_summary["displayed_percent"]),

    "deployment_readiness": DEPLOYMENT_READINESS,
}

READY_FOR_BATCH_10 = DEPLOYMENT_READINESS in ["PASS", "PASS_WITH_WARNINGS"]

print("=" * 100)
print("P4-32 — FINAL ACCEPTANCE GATE REPORT")
print("=" * 100)

print("\nFinal model card summary:")
print(json.dumps(final_model_card_summary, indent=2, ensure_ascii=False))

print("\nAcceptance gates:")
display(final_acceptance_df)

print("\nFailed ERROR gates:")
display(failed_error_gates)

print("\nFailed WARNING gates:")
display(failed_warning_gates)

print("=" * 100)
print(f"DEPLOYMENT_READINESS = {DEPLOYMENT_READINESS}")
print(f"READY_FOR_BATCH_10 = {READY_FOR_BATCH_10}")
print("=" * 100)

P4-33 — Build artifact metadata dictionary

In [ ]:
# ============================================================
# P4-33 — Build artifact metadata dictionary
# ============================================================

import platform
import pickle
import joblib
from datetime import datetime, timezone

def utc_now_iso():
    return datetime.now(timezone.utc).isoformat()


def safe_json_scalar(x):
    """
    Convert common numpy/pandas scalars into JSON-friendly Python scalars.
    """
    if isinstance(x, (np.integer,)):
        return int(x)

    if isinstance(x, (np.floating,)):
        if np.isnan(x):
            return None
        return float(x)

    if isinstance(x, (np.bool_,)):
        return bool(x)

    if isinstance(x, pd.Timestamp):
        return x.isoformat()

    if pd.isna(x) if not isinstance(x, (list, dict, tuple, set, pd.DataFrame, pd.Series)) else False:
        return None

    return x


def dataframe_to_records(df):
    if df is None:
        return None

    out = df.copy()

    for c in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[c]):
            out[c] = out[c].astype(str)

    return out.to_dict(orient="records")


def stringify_tuple_key_dict(d):
    """
    JSON-friendly version of dictionaries with tuple keys.
    Pickle artifact keeps original tuple keys.
    JSON summary uses string keys only.
    """
    if d is None:
        return None

    out = {}

    for k, v in d.items():
        if isinstance(k, tuple):
            k2 = "||".join(map(str, k))
        else:
            k2 = str(k)

        out[k2] = v

    return out


# ------------------------------------------------------------
# Phase 5 production output contract
# ------------------------------------------------------------

phase5_output_contract = {
    "required_output_fields": [
        "eta_p50_sec",
        "eta_p90_sec",
        "next_type",
        "type_conf",
    ],
    "field_definitions": {
        "eta_p50_sec": "Final calibrated/postprocessed P50 ETA in seconds.",
        "eta_p90_sec": "Final calibrated/postprocessed P90 ETA upper-bound in seconds.",
        "next_type": "Predicted next target status type.",
        "type_conf": "Classifier confidence/probability of predicted next_type.",
    },
    "status_targeting_contract": {
        "normal_statuses": sorted(NORMAL_STATUSES),
        "target_statuses": sorted(TARGET_STATUSES),
        "ignored_statuses": sorted(IGNORED_STATUSES),
        "target_status_rule": (
            "Rows whose canonical mc_status is in target_statuses are target events. "
            "For each row, the label is the next future target event within the same process + machine."
        ),
    },
    "serving_notes": {
        "registered_column_required": False,
        "required_input_columns": ["occurred", "mc_no", "mc_status", "process"],
        "timestamp_column": "occurred",
        "machine_column": "mc_no",
        "status_column": "mc_status",
        "process_column": "process",
    },
}

# ------------------------------------------------------------
# Artifact metadata bundle
# ------------------------------------------------------------

artifacts_phase4 = {
    # -----------------------------
    # Identity
    # -----------------------------
    "artifact_schema_version": CFG.artifact_schema_version,
    "created_at_utc": utc_now_iso(),
    "pipeline_name": CFG.pipeline_name,
    "process_id": CFG.process_id,
    "feature_version": CFG.feature_version,

    # -----------------------------
    # Runtime info
    # -----------------------------
    "runtime": {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "lightgbm_version": lgb.__version__,
        "random_seed": SEED,
    },

    # -----------------------------
    # Input/schema contract
    # -----------------------------
    "input_schema": {
        "required_columns": ["occurred", "mc_no", "mc_status", "process"],
        "occurred_col": "occurred",
        "machine_col": "mc_no",
        "status_col": "mc_status",
        "process_col": "process",
        "registered_col_used": False,
    },

    # -----------------------------
    # Config/status contract
    # -----------------------------
    "config": asdict(CFG),
    "status_config": {
        "all_statuses": list(ALL_STATUSES),
        "normal_statuses": sorted(NORMAL_STATUSES),
        "target_statuses": sorted(TARGET_STATUSES),
        "ignored_statuses": sorted(IGNORED_STATUSES),
        "status_alias_map": CFG.status_alias_map,
        "status_role": STATUS_ROLE,
        "hidden_statuses": CFG.hidden_statuses,
    },

    # -----------------------------
    # Feature contract
    # -----------------------------
    "feature_contract": {
        "feature_count": int(len(feature_cols)),
        "feature_cols": list(feature_cols),
        "feature_groups": feature_metadata["feature_groups"],
        "feature_fill_values": feature_fill_values,
        "rolling_windows_min": CFG.rolling_windows_min,
        "behavior_maps": behavior_maps,
        "machine_behavior_cols": MACHINE_BEHAVIOR_COLS,
        "status_behavior_cols": STATUS_BEHAVIOR_COLS,
        "ratio_cols": RATIO_COLS,
    },

    # -----------------------------
    # ETA model contract
    # -----------------------------
    "eta_model_contract": {
        "p50_model_file": "lgbm_quantile_p50_final.pkl",
        "p90_model_file": "lgbm_quantile_p90_final.pkl",

        "selected_p50_model": best_p50_name,
        "selected_p50_target_mode": p50_target_mode_final,
        "selected_p50_best_iteration": int(lgbm_p50_final.best_iteration_ or 0),

        "selected_p90_model": best_p90_name,
        "selected_p90_target_mode": p90_target_mode_final,
        "selected_p90_best_iteration": int(lgbm_p90_final.best_iteration_ or 0),

        "clip_max_sec": float(clip_max),
        "target_clip_quantile": float(CFG.target_clip_quantile),

        "p50_calibration": {
            "global_scale": float(p50_scale_global),
            "scale_by_machine": p50_scale_by_machine,
            "scale_clip": P50_SCALE_CLIP,
            "min_machine_cal_rows": P50_MIN_MACHINE_CAL_ROWS,
            "shrinkage_k": P50_SHRINKAGE_K,
        },

        "p90_calibration": {
            "global_multiplier": float(p90_multiplier_global),
            "multiplier_by_machine": p90_multiplier_by_machine,
            "multiplier_clip": P90_MULTIPLIER_CLIP,
            "target_coverage": P90_TARGET_COVERAGE,
            "min_machine_cal_rows": P90_MIN_MACHINE_CAL_ROWS,
            "shrinkage_k": P90_SHRINKAGE_K,
        },

        "eta_postprocess_maps": eta_postprocess_maps,
        "eta_min_floor_sec": ETA_MIN_FLOOR_SEC,
        "eta_max_cap_sec": ETA_MAX_CAP_SEC,
    },

    # -----------------------------
    # Type classifier contract
    # -----------------------------
    "type_model_contract": {
        "type_model_file": "lgbm_next_type.pkl",
        "selected_type_model": best_type_name,
        "selected_type_best_iteration": int(lgbm_type_final.best_iteration_ or 0),

        "type_label_encoder": type_label_encoder,
        "type_classes": type_classes,
        "type_class_to_idx": type_class_to_idx,
        "type_idx_to_class": type_idx_to_class,

        "global_confidence_threshold": float(TYPE_CONF_THRESHOLD_GLOBAL),
        "confidence_threshold_by_class": type_conf_threshold_by_class,
        "per_class_threshold_df": per_class_threshold_df,

        "class_weight_by_class": class_weight_by_class,
        "class_weight_by_idx": class_weight_by_idx,
    },

    # -----------------------------
    # Baselines and fallbacks
    # -----------------------------
    "baseline_contract": {
        "eta_baseline_maps": eta_baseline_maps,
        "type_baseline_maps": type_baseline_maps,
        "best_eta_baseline_name": best_eta_baseline_name,
        "best_type_baseline_name": best_type_baseline_name,
    },

    # -----------------------------
    # Metrics
    # -----------------------------
    "metrics": {
        "final_model_card_summary": final_model_card_summary,

        "validation": {
            "eta_final_validation_summary": eta_final_validation_summary,
            "type_selection_summary": type_selection_summary,
            "type_vs_baseline": type_vs_baseline,
        },

        "test": {
            "eta_test_summary": eta_test_summary,
            "type_test_summary": type_test_summary,
        },

        "acceptance": {
            "deployment_readiness": DEPLOYMENT_READINESS,
            "acceptance_gates": ACCEPTANCE_GATES,
            "acceptance_df": final_acceptance_df,
            "failed_error_gates": failed_error_gates,
            "failed_warning_gates": failed_warning_gates,
        },
    },

    # -----------------------------
    # Phase 5 output contract
    # -----------------------------
    "phase5_output_contract": phase5_output_contract,
}

# ------------------------------------------------------------
# Lightweight JSON-safe summary
# ------------------------------------------------------------

artifact_summary_json = {
    "artifact_schema_version": artifacts_phase4["artifact_schema_version"],
    "created_at_utc": artifacts_phase4["created_at_utc"],
    "pipeline_name": artifacts_phase4["pipeline_name"],
    "process_id": artifacts_phase4["process_id"],
    "feature_version": artifacts_phase4["feature_version"],
    "feature_count": int(len(feature_cols)),

    "normal_statuses": sorted(NORMAL_STATUSES),
    "target_statuses": sorted(TARGET_STATUSES),
    "ignored_statuses": sorted(IGNORED_STATUSES),

    "selected_models": {
        "p50": best_p50_name,
        "p50_target_mode": p50_target_mode_final,
        "p90": best_p90_name,
        "p90_target_mode": p90_target_mode_final,
        "type": best_type_name,
    },

    "test_metrics": {
        "eta_medae_sec": float(p50_test_final_metrics["MedAE_sec"]),
        "eta_mae_sec": float(p50_test_final_metrics["MAE_sec"]),
        "eta_hit_5m_percent": float(p50_test_final_metrics["Hit_5m_percent"]),
        "eta_hit_10m_percent": float(p50_test_final_metrics["Hit_10m_percent"]),
        "p90_coverage_raw_percent": float(p90_test_final_metrics["P90_coverage_raw_percent"]),
        "p90_median_width_sec": float(p90_test_final_metrics["P90_median_width_sec"]),
        "type_accuracy": float(type_test_metrics["accuracy"]),
        "type_macro_f1": float(type_test_metrics["macro_f1"]),
        "type_weighted_f1": float(type_test_metrics["weighted_f1"]),
        "type_displayed_accuracy_percent": float(type_test_confidence_summary["accuracy_displayed_rows"] * 100),
        "type_displayed_percent": float(type_test_confidence_summary["displayed_percent"]),
    },

    "deployment_readiness": DEPLOYMENT_READINESS,
    "phase5_output_contract": phase5_output_contract,
}

print("P4-33 ready")
print("Artifact metadata dictionary created.")

print("\nArtifact summary:")
print(json.dumps(artifact_summary_json, indent=2, ensure_ascii=False))

P4-34 — Save production artifact files

This saves the four filenames that Phase 5 expects:
- lgbm_quantile_p50_final.pkl
- lgbm_quantile_p90_final.pkl
- lgbm_next_type.pkl
- artifacts_phase4.pkl

In [ ]:
# ============================================================
# P4-34 — Save production artifact files
# ============================================================

import os
import json
import hashlib
from pathlib import Path

artifact_dir = Path(CFG.artifact_dir)
artifact_dir.mkdir(parents=True, exist_ok=True)

p50_model_path = artifact_dir / "lgbm_quantile_p50_final.pkl"
p90_model_path = artifact_dir / "lgbm_quantile_p90_final.pkl"
type_model_path = artifact_dir / "lgbm_next_type.pkl"
metadata_path = artifact_dir / "artifacts_phase4.pkl"

summary_json_path = artifact_dir / "artifacts_phase4_summary.json"
model_card_json_path = artifact_dir / "model_card_phase4.json"

# ------------------------------------------------------------
# Save model files
# ------------------------------------------------------------

joblib.dump(lgbm_p50_final, p50_model_path)
joblib.dump(lgbm_p90_final, p90_model_path)
joblib.dump(lgbm_type_final, type_model_path)

# Save artifact metadata bundle
joblib.dump(artifacts_phase4, metadata_path)

# Save lightweight human-readable summaries
with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(artifact_summary_json, f, indent=2, ensure_ascii=False)

with open(model_card_json_path, "w", encoding="utf-8") as f:
    json.dump(final_model_card_summary, f, indent=2, ensure_ascii=False)


# ------------------------------------------------------------
# Checksum helpers
# ------------------------------------------------------------

def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            b = f.read(block_size)
            if not b:
                break
            h.update(b)

    return h.hexdigest()


saved_files = [
    p50_model_path,
    p90_model_path,
    type_model_path,
    metadata_path,
    summary_json_path,
    model_card_json_path,
]

artifact_file_report = pd.DataFrame([
    {
        "file": p.name,
        "path": str(p.resolve()),
        "size_mb": p.stat().st_size / (1024 ** 2),
        "sha256": sha256_file(p),
    }
    for p in saved_files
])

# ------------------------------------------------------------
# Minimal save validation
# ------------------------------------------------------------

required_phase5_files = [
    "lgbm_quantile_p50_final.pkl",
    "lgbm_quantile_p90_final.pkl",
    "lgbm_next_type.pkl",
    "artifacts_phase4.pkl",
]

save_checks = []

def add_save_check(name, passed, detail, severity="ERROR"):
    save_checks.append({
        "check": name,
        "severity": severity,
        "passed": bool(passed),
        "detail": detail,
    })

for fname in required_phase5_files:
    fpath = artifact_dir / fname
    add_save_check(
        f"required_file_exists__{fname}",
        fpath.exists() and fpath.stat().st_size > 0,
        f"path={str(fpath.resolve())}, size_bytes={fpath.stat().st_size if fpath.exists() else 0}",
    )

add_save_check(
    "metadata_contains_phase5_output_contract",
    "phase5_output_contract" in artifacts_phase4,
    f"phase5_output_contract_keys={list(artifacts_phase4.get('phase5_output_contract', {}).keys())}",
)

add_save_check(
    "metadata_feature_count_matches",
    len(artifacts_phase4["feature_contract"]["feature_cols"]) == len(feature_cols),
    f"metadata_feature_count={len(artifacts_phase4['feature_contract']['feature_cols'])}, feature_cols={len(feature_cols)}",
)

add_save_check(
    "metadata_target_statuses_match",
    set(artifacts_phase4["status_config"]["target_statuses"]) == set(TARGET_STATUSES),
    f"metadata_target_statuses={artifacts_phase4['status_config']['target_statuses']}, current={sorted(TARGET_STATUSES)}",
)

add_save_check(
    "metadata_normal_statuses_match",
    set(artifacts_phase4["status_config"]["normal_statuses"]) == set(NORMAL_STATUSES),
    f"metadata_normal_statuses={artifacts_phase4['status_config']['normal_statuses']}, current={sorted(NORMAL_STATUSES)}",
)

add_save_check(
    "deployment_readiness_saved",
    artifacts_phase4["metrics"]["acceptance"]["deployment_readiness"] == DEPLOYMENT_READINESS,
    f"deployment_readiness={DEPLOYMENT_READINESS}",
)

save_validation_df = pd.DataFrame(save_checks)

failed_save_errors = save_validation_df[
    (save_validation_df["severity"].eq("ERROR")) &
    (~save_validation_df["passed"])
]

READY_FOR_BATCH_11 = len(failed_save_errors) == 0

print("=" * 100)
print("P4-34 — ARTIFACT SAVE REPORT")
print("=" * 100)

print(f"Artifact directory: {artifact_dir.resolve()}")

print("\nSaved artifact files:")
display(artifact_file_report)

print("\nPhase 5 required files:")
for fname in required_phase5_files:
    print(f"- {fname}")

print("\nSave validation gates:")
display(save_validation_df)

print("\nArtifact summary JSON:")
print(json.dumps(artifact_summary_json, indent=2, ensure_ascii=False))

print("=" * 100)
if READY_FOR_BATCH_11:
    print("READY_FOR_BATCH_11 = True")
    print("Batch 10 complete. Send this output before moving to Batch 11 smoke test.")
else:
    print("READY_FOR_BATCH_11 = False")
    print("Batch 10 has failed ERROR gates. Fix before moving to Batch 11.")
print("=" * 100)

Artifact reload and Phase 5 smoke test.

This verifies:

1. Saved artifacts can be reloaded from disk
2. Reloaded models can produce predictions
3. Output matches Phase 5 contract:
   - eta_p50_sec
   - eta_p90_sec
   - next_type
   - type_conf
4. P90 >= P50
5. next_type is one of target_statuses
6. registered column is not required

P4-35 — Reload saved artifacts

In [ ]:
# ============================================================
# P4-35 — Reload saved artifacts
# ============================================================

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import json

artifact_dir = Path(CFG.artifact_dir)

loaded_p50_path = artifact_dir / "lgbm_quantile_p50_final.pkl"
loaded_p90_path = artifact_dir / "lgbm_quantile_p90_final.pkl"
loaded_type_path = artifact_dir / "lgbm_next_type.pkl"
loaded_meta_path = artifact_dir / "artifacts_phase4.pkl"

loaded_p50_model = joblib.load(loaded_p50_path)
loaded_p90_model = joblib.load(loaded_p90_path)
loaded_type_model = joblib.load(loaded_type_path)
loaded_artifacts_phase4 = joblib.load(loaded_meta_path)

loaded_feature_cols = loaded_artifacts_phase4["feature_contract"]["feature_cols"]
loaded_feature_fill_values = loaded_artifacts_phase4["feature_contract"]["feature_fill_values"]

loaded_status_config = loaded_artifacts_phase4["status_config"]
loaded_eta_contract = loaded_artifacts_phase4["eta_model_contract"]
loaded_type_contract = loaded_artifacts_phase4["type_model_contract"]
loaded_phase5_contract = loaded_artifacts_phase4["phase5_output_contract"]

loaded_target_statuses = set(loaded_status_config["target_statuses"])
loaded_normal_statuses = set(loaded_status_config["normal_statuses"])

reload_report = {
    "artifact_dir": str(artifact_dir.resolve()),
    "p50_model_loaded": loaded_p50_model is not None,
    "p90_model_loaded": loaded_p90_model is not None,
    "type_model_loaded": loaded_type_model is not None,
    "metadata_loaded": loaded_artifacts_phase4 is not None,
    "feature_count_loaded": len(loaded_feature_cols),
    "phase5_required_output_fields": loaded_phase5_contract["required_output_fields"],
    "registered_column_required": loaded_phase5_contract["serving_notes"]["registered_column_required"],
    "normal_statuses": sorted(loaded_normal_statuses),
    "target_statuses": sorted(loaded_target_statuses),
    "deployment_readiness": loaded_artifacts_phase4["metrics"]["acceptance"]["deployment_readiness"],
}

print("=" * 100)
print("P4-35 — RELOAD SAVED ARTIFACTS")
print("=" * 100)

print(json.dumps(reload_report, indent=2, ensure_ascii=False))

print("\nLoaded model best iterations:")
print("P50:", getattr(loaded_p50_model, "best_iteration_", None))
print("P90:", getattr(loaded_p90_model, "best_iteration_", None))
print("Type:", getattr(loaded_type_model, "best_iteration_", None))

print("\nFirst 30 loaded feature columns:")
for i, c in enumerate(loaded_feature_cols[:30], start=1):
    print(f"{i:03d}. {c}")

print("=" * 100)
print("P4-35 ready")
print("=" * 100)

P4-36 — Run production-style prediction on one test row

In [ ]:
# ============================================================
# P4-36 — Run production-style prediction on one test row
# ============================================================

def loaded_inverse_transform_pred(pred, target_mode):
    pred = np.asarray(pred, dtype=np.float64)

    if target_mode == "raw":
        return pred

    if target_mode == "log1p":
        return np.expm1(pred)

    raise ValueError(f"Unknown target mode: {target_mode}")


def loaded_clean_eta_pred(pred, clip_upper=None):
    pred = np.asarray(pred, dtype=np.float64)
    pred = np.nan_to_num(
        pred,
        nan=0.0,
        posinf=clip_upper if clip_upper is not None else 1e9,
        neginf=0.0,
    )
    pred = np.maximum(pred, 1.0)

    if clip_upper is not None:
        pred = np.minimum(pred, clip_upper)

    return pred


def build_loaded_model_matrix(df, feature_cols, fill_values):
    X = df[feature_cols].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(fill_values)

    missing_cols = [c for c in feature_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing feature columns: {missing_cols[:20]}")

    bad_nan_cols = X.columns[X.isna().any()].tolist()
    if bad_nan_cols:
        raise ValueError(f"NaN still found after fill. Example columns: {bad_nan_cols[:20]}")

    inf_counts = np.isinf(X.select_dtypes(include=[np.number])).sum()
    inf_cols = inf_counts[inf_counts > 0].index.tolist()
    if inf_cols:
        raise ValueError(f"Inf still found after fill. Example columns: {inf_cols[:20]}")

    return X.astype(np.float32)


def loaded_get_eta_bounds(row, eta_postprocess_maps):
    process = row["process"]
    mc_no = row["mc_no"]
    status = row["mc_status"]

    key_ms = (process, mc_no, status)
    key_m = (process, mc_no)

    if key_ms in eta_postprocess_maps["by_machine_status"]:
        return eta_postprocess_maps["by_machine_status"][key_ms], "machine_status"

    if key_m in eta_postprocess_maps["by_machine"]:
        return eta_postprocess_maps["by_machine"][key_m], "machine"

    if status in eta_postprocess_maps["by_status"]:
        return eta_postprocess_maps["by_status"][status], "status"

    return eta_postprocess_maps["global"], "global"


def loaded_predict_one_phase5(row_df):
    """
    Smoke-test production-style prediction using already engineered feature columns.

    In real Phase 5, the streaming/serving layer must build the same feature columns
    from live machine state before calling these models.
    """
    if len(row_df) != 1:
        raise ValueError("loaded_predict_one_phase5 expects exactly one row.")

    X_one = build_loaded_model_matrix(
        row_df,
        loaded_feature_cols,
        loaded_feature_fill_values,
    )

    row = row_df.iloc[0]

    # -----------------------------
    # P50 base prediction
    # -----------------------------
    p50_target_mode = loaded_eta_contract["selected_p50_target_mode"]
    p50_clip_max = float(loaded_eta_contract["clip_max_sec"])

    p50_base_model_space = loaded_p50_model.predict(
        X_one,
        num_iteration=loaded_p50_model.best_iteration_,
    )

    p50_base_sec = loaded_inverse_transform_pred(
        p50_base_model_space,
        p50_target_mode,
    )

    p50_base_sec = loaded_clean_eta_pred(
        p50_base_sec,
        clip_upper=p50_clip_max,
    )[0]

    # -----------------------------
    # P90 base prediction
    # -----------------------------
    p90_target_mode = loaded_eta_contract["selected_p90_target_mode"]

    p90_base_model_space = loaded_p90_model.predict(
        X_one,
        num_iteration=loaded_p90_model.best_iteration_,
    )

    p90_base_sec = loaded_inverse_transform_pred(
        p90_base_model_space,
        p90_target_mode,
    )

    p90_base_sec = loaded_clean_eta_pred(
        p90_base_sec,
        clip_upper=p50_clip_max,
    )[0]

    p90_base_sec = max(p90_base_sec, p50_base_sec + 1.0)

    # -----------------------------
    # Apply saved P50 calibration
    # -----------------------------
    process = row["process"]
    mc_no = row["mc_no"]

    p50_calibration = loaded_eta_contract["p50_calibration"]
    p50_global_scale = float(p50_calibration["global_scale"])
    p50_scale_by_machine = p50_calibration["scale_by_machine"]

    p50_scale = float(
        p50_scale_by_machine.get(
            (process, mc_no),
            p50_global_scale,
        )
    )

    p50_cal_sec = p50_base_sec * p50_scale
    p50_cal_sec = loaded_clean_eta_pred(
        [p50_cal_sec],
        clip_upper=p50_clip_max,
    )[0]

    # -----------------------------
    # Apply saved P90 calibration
    # -----------------------------
    p90_calibration = loaded_eta_contract["p90_calibration"]
    p90_global_multiplier = float(p90_calibration["global_multiplier"])
    p90_multiplier_by_machine = p90_calibration["multiplier_by_machine"]

    p90_multiplier = float(
        p90_multiplier_by_machine.get(
            (process, mc_no),
            p90_global_multiplier,
        )
    )

    p90_cal_sec = p90_base_sec * p90_multiplier
    p90_cal_sec = max(p90_cal_sec, p50_cal_sec + 1.0)

    # -----------------------------
    # Apply saved post-processing bounds
    # -----------------------------
    eta_postprocess_maps_loaded = loaded_eta_contract["eta_postprocess_maps"]
    eta_max_cap_sec = float(loaded_eta_contract["eta_max_cap_sec"])

    bounds, eta_bound_source = loaded_get_eta_bounds(
        row,
        eta_postprocess_maps_loaded,
    )

    floor_sec = float(bounds["floor"])
    median_sec = float(bounds["median"])
    cap_sec = float(bounds["cap"])

    p50_final_sec = float(np.clip(p50_cal_sec, floor_sec, cap_sec))

    p90_final_sec = max(p90_cal_sec, p50_final_sec + 1.0)
    p90_final_sec = max(p90_final_sec, median_sec)
    p90_final_sec = float(np.clip(p90_final_sec, p50_final_sec + 1.0, eta_max_cap_sec * 1.50))

    # -----------------------------
    # Type prediction
    # -----------------------------
    type_proba = loaded_type_model.predict_proba(
        X_one,
        num_iteration=loaded_type_model.best_iteration_,
    )

    type_pred_idx = int(np.argmax(type_proba[0]))
    type_conf = float(np.max(type_proba[0]))

    loaded_type_label_encoder = loaded_type_contract["type_label_encoder"]
    next_type = str(loaded_type_label_encoder.inverse_transform([type_pred_idx])[0])

    output = {
        "eta_p50_sec": p50_final_sec,
        "eta_p90_sec": p90_final_sec,
        "next_type": next_type,
        "type_conf": type_conf,
    }

    debug = {
        "p50_base_sec": float(p50_base_sec),
        "p50_scale": float(p50_scale),
        "p50_cal_sec": float(p50_cal_sec),
        "p90_base_sec": float(p90_base_sec),
        "p90_multiplier": float(p90_multiplier),
        "p90_cal_sec": float(p90_cal_sec),
        "eta_bound_source": eta_bound_source,
        "eta_floor_sec": floor_sec,
        "eta_median_sec": median_sec,
        "eta_cap_sec": cap_sec,
        "type_proba": {
            str(cls): float(prob)
            for cls, prob in zip(loaded_type_contract["type_classes"], type_proba[0])
        },
    }

    return output, debug


# Pick a deterministic smoke-test row from test split.
SMOKE_TEST_POSITION = min(1000, len(test_df) - 1)

smoke_row_df = test_df.iloc[[SMOKE_TEST_POSITION]].copy()
smoke_X = build_loaded_model_matrix(
    smoke_row_df,
    loaded_feature_cols,
    loaded_feature_fill_values,
)

phase5_smoke_output, phase5_smoke_debug = loaded_predict_one_phase5(smoke_row_df)

smoke_actual = {
    "actual_y_time_sec": float(smoke_row_df["y_time_sec"].iloc[0]),
    "actual_next_target_type": str(smoke_row_df["next_target_type"].iloc[0]),
    "current_status": str(smoke_row_df["mc_status"].iloc[0]),
    "process": str(smoke_row_df["process"].iloc[0]),
    "mc_no": str(smoke_row_df["mc_no"].iloc[0]),
    "occurred": str(smoke_row_df["occurred"].iloc[0]),
}

print("=" * 100)
print("P4-36 — ONE-ROW PHASE 5 SMOKE PREDICTION")
print("=" * 100)

print("\nSmoke-test input row:")
display(
    smoke_row_df[
        [
            "process",
            "mc_no",
            "occurred",
            "mc_status",
            "next_target_time",
            "next_target_type",
            "y_time_sec",
        ]
    ]
)

print("\nPhase 5 smoke output:")
print(json.dumps(phase5_smoke_output, indent=2, ensure_ascii=False))

print("\nActual label for comparison:")
print(json.dumps(smoke_actual, indent=2, ensure_ascii=False))

print("\nSmoke debug info:")
print(json.dumps(phase5_smoke_debug, indent=2, ensure_ascii=False))

print("=" * 100)
print("P4-36 ready")
print("=" * 100)

P4-37 — Validate Phase 5 output contract

In [ ]:
# ============================================================
# P4-37 — Validate Phase 5 output contract
# ============================================================

# ------------------------------------------------------------
# Batch smoke test on first N test rows
# ------------------------------------------------------------

SMOKE_BATCH_N = min(5000, len(test_df))

smoke_batch_df = test_df.iloc[:SMOKE_BATCH_N].copy()
X_smoke_batch = build_loaded_model_matrix(
    smoke_batch_df,
    loaded_feature_cols,
    loaded_feature_fill_values,
)

# P50 batch
p50_batch_base = loaded_p50_model.predict(
    X_smoke_batch,
    num_iteration=loaded_p50_model.best_iteration_,
)

p50_batch_base = loaded_inverse_transform_pred(
    p50_batch_base,
    loaded_eta_contract["selected_p50_target_mode"],
)

p50_batch_base = loaded_clean_eta_pred(
    p50_batch_base,
    clip_upper=float(loaded_eta_contract["clip_max_sec"]),
)

# P90 batch
p90_batch_base = loaded_p90_model.predict(
    X_smoke_batch,
    num_iteration=loaded_p90_model.best_iteration_,
)

p90_batch_base = loaded_inverse_transform_pred(
    p90_batch_base,
    loaded_eta_contract["selected_p90_target_mode"],
)

p90_batch_base = loaded_clean_eta_pred(
    p90_batch_base,
    clip_upper=float(loaded_eta_contract["clip_max_sec"]),
)

p90_batch_base = np.maximum(p90_batch_base, p50_batch_base + 1.0)

# Apply row-by-row loaded production wrapper for final outputs.
# This is intentionally slower but verifies the exact one-row serving path.
batch_outputs = []

for i in range(SMOKE_BATCH_N):
    row_df = smoke_batch_df.iloc[[i]].copy()
    output_i, _ = loaded_predict_one_phase5(row_df)
    batch_outputs.append(output_i)

batch_output_df = pd.DataFrame(batch_outputs)

# ------------------------------------------------------------
# Contract validation
# ------------------------------------------------------------

phase5_required_fields = loaded_phase5_contract["required_output_fields"]

contract_checks = []

def add_contract_check(name, passed, detail, severity="ERROR"):
    contract_checks.append({
        "check": name,
        "severity": severity,
        "passed": bool(passed),
        "detail": detail,
    })


# Required files still exist
for fname in [
    "lgbm_quantile_p50_final.pkl",
    "lgbm_quantile_p90_final.pkl",
    "lgbm_next_type.pkl",
    "artifacts_phase4.pkl",
]:
    fpath = artifact_dir / fname
    add_contract_check(
        f"required_artifact_exists__{fname}",
        fpath.exists() and fpath.stat().st_size > 0,
        f"path={str(fpath.resolve())}, size_bytes={fpath.stat().st_size if fpath.exists() else 0}",
    )


# Output field contract
add_contract_check(
    "one_row_output_has_required_fields",
    all(k in phase5_smoke_output for k in phase5_required_fields),
    f"required={phase5_required_fields}, actual={list(phase5_smoke_output.keys())}",
)

add_contract_check(
    "batch_output_has_required_fields",
    all(k in batch_output_df.columns for k in phase5_required_fields),
    f"required={phase5_required_fields}, actual={list(batch_output_df.columns)}",
)

# ETA contract
add_contract_check(
    "one_row_eta_p50_positive",
    phase5_smoke_output["eta_p50_sec"] > 0,
    f"eta_p50_sec={phase5_smoke_output['eta_p50_sec']}",
)

add_contract_check(
    "one_row_eta_p90_greater_equal_p50",
    phase5_smoke_output["eta_p90_sec"] >= phase5_smoke_output["eta_p50_sec"],
    f"eta_p50_sec={phase5_smoke_output['eta_p50_sec']}, eta_p90_sec={phase5_smoke_output['eta_p90_sec']}",
)

add_contract_check(
    "batch_eta_p50_positive",
    (batch_output_df["eta_p50_sec"] > 0).all(),
    f"min_eta_p50_sec={batch_output_df['eta_p50_sec'].min()}",
)

add_contract_check(
    "batch_eta_p90_greater_equal_p50",
    (batch_output_df["eta_p90_sec"] >= batch_output_df["eta_p50_sec"]).all(),
    f"violations={int((batch_output_df['eta_p90_sec'] < batch_output_df['eta_p50_sec']).sum())}",
)

add_contract_check(
    "batch_eta_outputs_no_nan",
    batch_output_df[["eta_p50_sec", "eta_p90_sec"]].notna().all().all(),
    f"nan_counts={batch_output_df[['eta_p50_sec', 'eta_p90_sec']].isna().sum().to_dict()}",
)

# Type contract
add_contract_check(
    "one_row_next_type_is_target_status",
    phase5_smoke_output["next_type"] in loaded_target_statuses,
    f"next_type={phase5_smoke_output['next_type']}, target_statuses={sorted(loaded_target_statuses)}",
)

add_contract_check(
    "batch_next_type_is_target_status",
    batch_output_df["next_type"].isin(loaded_target_statuses).all(),
    f"bad_next_type_count={int((~batch_output_df['next_type'].isin(loaded_target_statuses)).sum())}",
)

add_contract_check(
    "one_row_type_conf_between_0_and_1",
    0.0 <= phase5_smoke_output["type_conf"] <= 1.0,
    f"type_conf={phase5_smoke_output['type_conf']}",
)

add_contract_check(
    "batch_type_conf_between_0_and_1",
    batch_output_df["type_conf"].between(0, 1).all(),
    f"min={batch_output_df['type_conf'].min()}, max={batch_output_df['type_conf'].max()}",
)

# Metadata contract
add_contract_check(
    "registered_column_not_required",
    loaded_phase5_contract["serving_notes"]["registered_column_required"] is False,
    f"registered_column_required={loaded_phase5_contract['serving_notes']['registered_column_required']}",
)

add_contract_check(
    "input_schema_has_exact_required_columns",
    loaded_phase5_contract["serving_notes"]["required_input_columns"] == ["occurred", "mc_no", "mc_status", "process"],
    f"required_input_columns={loaded_phase5_contract['serving_notes']['required_input_columns']}",
)

add_contract_check(
    "feature_count_matches_loaded_matrix",
    X_smoke_batch.shape[1] == len(loaded_feature_cols),
    f"X_smoke_features={X_smoke_batch.shape[1]}, loaded_feature_cols={len(loaded_feature_cols)}",
)

add_contract_check(
    "deployment_readiness_is_acceptable",
    loaded_artifacts_phase4["metrics"]["acceptance"]["deployment_readiness"] in ["PASS", "PASS_WITH_WARNINGS"],
    f"deployment_readiness={loaded_artifacts_phase4['metrics']['acceptance']['deployment_readiness']}",
)

contract_validation_df = pd.DataFrame(contract_checks)

failed_contract_errors = contract_validation_df[
    (contract_validation_df["severity"].eq("ERROR")) &
    (~contract_validation_df["passed"])
]

READY_FOR_PHASE5_INTEGRATION = len(failed_contract_errors) == 0

# ------------------------------------------------------------
# Smoke batch summary
# ------------------------------------------------------------

smoke_batch_summary = {
    "smoke_batch_rows": int(SMOKE_BATCH_N),
    "eta_p50_median_sec": float(batch_output_df["eta_p50_sec"].median()),
    "eta_p50_p90_sec": float(batch_output_df["eta_p50_sec"].quantile(0.90)),
    "eta_p90_median_sec": float(batch_output_df["eta_p90_sec"].median()),
    "eta_p90_p90_sec": float(batch_output_df["eta_p90_sec"].quantile(0.90)),
    "type_conf_mean": float(batch_output_df["type_conf"].mean()),
    "type_conf_median": float(batch_output_df["type_conf"].median()),
    "displayed_percent_conf_ge_threshold": float((batch_output_df["type_conf"] >= loaded_type_contract["global_confidence_threshold"]).mean() * 100),
    "predicted_type_distribution": batch_output_df["next_type"].value_counts().to_dict(),
}

print("=" * 100)
print("P4-37 — PHASE 5 OUTPUT CONTRACT VALIDATION")
print("=" * 100)

print("\nOne-row Phase 5 smoke output:")
print(json.dumps(phase5_smoke_output, indent=2, ensure_ascii=False))

print("\nSmoke batch summary:")
print(json.dumps(smoke_batch_summary, indent=2, ensure_ascii=False))

print("\nSmoke batch output sample:")
display(batch_output_df.head(20))

print("\nContract validation gates:")
display(contract_validation_df)

print("\nFailed contract ERROR gates:")
display(failed_contract_errors)

print("=" * 100)
if READY_FOR_PHASE5_INTEGRATION:
    print("READY_FOR_PHASE5_INTEGRATION = True")
    print("Batch 11 passed. Artifact bundle is ready for Phase 5 integration.")
else:
    print("READY_FOR_PHASE5_INTEGRATION = False")
    print("Batch 11 failed contract validation. Fix before Phase 5 integration.")
print("=" * 100)

P4-38 — Export feature importance reports

In [ ]:
# ============================================================
# P4-38 — Export feature importance reports
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

artifact_dir = Path(CFG.artifact_dir)
artifact_dir.mkdir(parents=True, exist_ok=True)

def get_lgbm_importance_df(model, feature_cols, model_name):
    booster = model.booster_

    importance_gain = booster.feature_importance(importance_type="gain")
    importance_split = booster.feature_importance(importance_type="split")

    out = pd.DataFrame({
        "feature": feature_cols,
        "importance_gain": importance_gain,
        "importance_split": importance_split,
    })

    out["model"] = model_name

    total_gain = out["importance_gain"].sum()
    total_split = out["importance_split"].sum()

    out["importance_gain_percent"] = np.where(
        total_gain > 0,
        out["importance_gain"] / total_gain * 100,
        0,
    )

    out["importance_split_percent"] = np.where(
        total_split > 0,
        out["importance_split"] / total_split * 100,
        0,
    )

    out = out.sort_values("importance_gain", ascending=False).reset_index(drop=True)
    out["rank_gain"] = np.arange(1, len(out) + 1)

    return out


p50_importance_df = get_lgbm_importance_df(
    lgbm_p50_final,
    feature_cols,
    "lgbm_quantile_p50_final",
)

p90_importance_df = get_lgbm_importance_df(
    lgbm_p90_final,
    feature_cols,
    "lgbm_quantile_p90_final",
)

type_importance_df = get_lgbm_importance_df(
    lgbm_type_final,
    feature_cols,
    "lgbm_next_type",
)

combined_importance_df = pd.concat(
    [
        p50_importance_df,
        p90_importance_df,
        type_importance_df,
    ],
    ignore_index=True,
)

# Aggregate importance across all three models
importance_agg_df = (
    combined_importance_df
    .groupby("feature")
    .agg(
        total_gain=("importance_gain", "sum"),
        total_split=("importance_split", "sum"),
        mean_gain_percent=("importance_gain_percent", "mean"),
        max_gain_percent=("importance_gain_percent", "max"),
        used_by_models=("model", "nunique"),
    )
    .reset_index()
)

importance_agg_df = importance_agg_df.sort_values(
    "total_gain",
    ascending=False,
).reset_index(drop=True)

importance_agg_df["rank_total_gain"] = np.arange(1, len(importance_agg_df) + 1)

# Save files
p50_importance_path = artifact_dir / "feature_importance_p50.csv"
p90_importance_path = artifact_dir / "feature_importance_p90.csv"
type_importance_path = artifact_dir / "feature_importance_next_type.csv"
combined_importance_path = artifact_dir / "feature_importance_combined.csv"
importance_agg_path = artifact_dir / "feature_importance_aggregate.csv"

p50_importance_df.to_csv(p50_importance_path, index=False)
p90_importance_df.to_csv(p90_importance_path, index=False)
type_importance_df.to_csv(type_importance_path, index=False)
combined_importance_df.to_csv(combined_importance_path, index=False)
importance_agg_df.to_csv(importance_agg_path, index=False)

importance_file_report = pd.DataFrame([
    {
        "file": p.name,
        "path": str(p.resolve()),
        "size_mb": p.stat().st_size / (1024 ** 2),
    }
    for p in [
        p50_importance_path,
        p90_importance_path,
        type_importance_path,
        combined_importance_path,
        importance_agg_path,
    ]
])

print("=" * 100)
print("P4-38 — FEATURE IMPORTANCE REPORT")
print("=" * 100)

print("\nTop 30 P50 features by gain:")
display(p50_importance_df.head(30))

print("\nTop 30 P90 features by gain:")
display(p90_importance_df.head(30))

print("\nTop 30 next-type classifier features by gain:")
display(type_importance_df.head(30))

print("\nTop 30 aggregate features across all models:")
display(importance_agg_df.head(30))

print("\nSaved feature importance files:")
display(importance_file_report)

print("=" * 100)
print("P4-38 ready")
print("=" * 100)

P4-39 — Save final model card and Phase 5 checklist

In [ ]:
# ============================================================
# P4-39 — Save final model card and Phase 5 checklist
# ============================================================

from pathlib import Path
import json
from datetime import datetime, timezone
from IPython.display import Markdown, display

artifact_dir = Path(CFG.artifact_dir)

model_card_md_path = artifact_dir / "model_card_phase4.md"
phase5_checklist_md_path = artifact_dir / "phase5_integration_checklist.md"

def fmt_pct(x):
    return f"{float(x):.3f}%"

def fmt_sec(x):
    return f"{float(x):.3f}s"

created_at = datetime.now(timezone.utc).isoformat()

summary = artifact_summary_json
metrics = summary["test_metrics"]

model_card_lines = []

model_card_lines.append("# Phase 4 Automatic Model Training — Model Card")
model_card_lines.append("")
model_card_lines.append(f"Created at UTC: `{created_at}`")
model_card_lines.append("")
model_card_lines.append("## Identity")
model_card_lines.append("")
model_card_lines.append(f"- Pipeline: `{summary['pipeline_name']}`")
model_card_lines.append(f"- Process ID: `{summary['process_id']}`")
model_card_lines.append(f"- Artifact schema version: `{summary['artifact_schema_version']}`")
model_card_lines.append(f"- Feature version: `{summary['feature_version']}`")
model_card_lines.append(f"- Deployment readiness: `{summary['deployment_readiness']}`")
model_card_lines.append("")
model_card_lines.append("## Status Configuration")
model_card_lines.append("")
model_card_lines.append(f"- Normal statuses: `{summary['normal_statuses']}`")
model_card_lines.append(f"- Target statuses: `{summary['target_statuses']}`")
model_card_lines.append(f"- Ignored statuses: `{summary['ignored_statuses']}`")
model_card_lines.append("")
model_card_lines.append("## Selected Models")
model_card_lines.append("")
model_card_lines.append(f"- P50 ETA model: `{summary['selected_models']['p50']}`")
model_card_lines.append(f"- P50 target mode: `{summary['selected_models']['p50_target_mode']}`")
model_card_lines.append(f"- P90 ETA model: `{summary['selected_models']['p90']}`")
model_card_lines.append(f"- P90 target mode: `{summary['selected_models']['p90_target_mode']}`")
model_card_lines.append(f"- Next-type classifier: `{summary['selected_models']['type']}`")
model_card_lines.append("")
model_card_lines.append("## Test Metrics")
model_card_lines.append("")
model_card_lines.append(f"- ETA MedAE: `{fmt_sec(metrics['eta_medae_sec'])}`")
model_card_lines.append(f"- ETA MAE: `{fmt_sec(metrics['eta_mae_sec'])}`")
model_card_lines.append(f"- ETA Hit@±5m: `{fmt_pct(metrics['eta_hit_5m_percent'])}`")
model_card_lines.append(f"- ETA Hit@±10m: `{fmt_pct(metrics['eta_hit_10m_percent'])}`")
model_card_lines.append(f"- P90 raw coverage: `{fmt_pct(metrics['p90_coverage_raw_percent'])}`")
model_card_lines.append(f"- P90 median width: `{fmt_sec(metrics['p90_median_width_sec'])}`")
model_card_lines.append(f"- Type accuracy: `{metrics['type_accuracy']:.6f}`")
model_card_lines.append(f"- Type macro F1: `{metrics['type_macro_f1']:.6f}`")
model_card_lines.append(f"- Type weighted F1: `{metrics['type_weighted_f1']:.6f}`")
model_card_lines.append(f"- Type displayed accuracy: `{fmt_pct(metrics['type_displayed_accuracy_percent'])}`")
model_card_lines.append(f"- Type displayed percent: `{fmt_pct(metrics['type_displayed_percent'])}`")
model_card_lines.append("")
model_card_lines.append("## Required Phase 5 Output Contract")
model_card_lines.append("")
for field in summary["phase5_output_contract"]["required_output_fields"]:
    definition = summary["phase5_output_contract"]["field_definitions"][field]
    model_card_lines.append(f"- `{field}` — {definition}")
model_card_lines.append("")
model_card_lines.append("## Required Phase 5 Input Columns")
model_card_lines.append("")
for col in summary["phase5_output_contract"]["serving_notes"]["required_input_columns"]:
    model_card_lines.append(f"- `{col}`")
model_card_lines.append("")
model_card_lines.append("`registered` is not required and was not used by this Phase 4 training pipeline.")
model_card_lines.append("")
model_card_lines.append("## Known Warnings / Limitations")
model_card_lines.append("")
model_card_lines.append("- Deployment readiness is `PASS_WITH_WARNINGS`, not pure `PASS`.")
model_card_lines.append("- The only failed acceptance warning was per-machine P90 minimum coverage for n>=200 machines.")
model_card_lines.append("- Global P90 coverage passed, but some individual machines remain under-covered.")
model_card_lines.append("- `mc_alarm` is the weakest next-type class and should be monitored in production.")
model_card_lines.append("- The dataset window is short, so retraining with a longer history should improve rare/long-gap cases.")
model_card_lines.append("")
model_card_lines.append("## Phase 5 Required Artifact Files")
model_card_lines.append("")
model_card_lines.append("- `lgbm_quantile_p50_final.pkl`")
model_card_lines.append("- `lgbm_quantile_p90_final.pkl`")
model_card_lines.append("- `lgbm_next_type.pkl`")
model_card_lines.append("- `artifacts_phase4.pkl`")

model_card_md = "\n".join(model_card_lines)

with open(model_card_md_path, "w", encoding="utf-8") as f:
    f.write(model_card_md)


checklist_lines = []

checklist_lines.append("# Phase 5 Integration Checklist")
checklist_lines.append("")
checklist_lines.append("## Load these files")
checklist_lines.append("")
checklist_lines.append("- [ ] `lgbm_quantile_p50_final.pkl`")
checklist_lines.append("- [ ] `lgbm_quantile_p90_final.pkl`")
checklist_lines.append("- [ ] `lgbm_next_type.pkl`")
checklist_lines.append("- [ ] `artifacts_phase4.pkl`")
checklist_lines.append("")
checklist_lines.append("## Input event contract")
checklist_lines.append("")
checklist_lines.append("- [ ] Input has `occurred`")
checklist_lines.append("- [ ] Input has `mc_no`")
checklist_lines.append("- [ ] Input has `mc_status`")
checklist_lines.append("- [ ] Input has `process`")
checklist_lines.append("- [ ] Do not require `registered`")
checklist_lines.append("")
checklist_lines.append("## Status logic")
checklist_lines.append("")
checklist_lines.append(f"- [ ] Treat `{summary['normal_statuses']}` as normal statuses")
checklist_lines.append(f"- [ ] Treat `{summary['target_statuses']}` as target statuses")
checklist_lines.append("- [ ] Only statuses in `target_statuses` are valid `next_type` outputs")
checklist_lines.append("")
checklist_lines.append("## Feature contract")
checklist_lines.append("")
checklist_lines.append(f"- [ ] Build exactly `{summary['feature_count']}` model features")
checklist_lines.append("- [ ] Use `feature_cols` from `artifacts_phase4.pkl`")
checklist_lines.append("- [ ] Use `feature_fill_values` from `artifacts_phase4.pkl`")
checklist_lines.append("- [ ] Replace NaN/inf before prediction")
checklist_lines.append("")
checklist_lines.append("## Prediction post-processing")
checklist_lines.append("")
checklist_lines.append("- [ ] Apply P50 inverse target transform")
checklist_lines.append("- [ ] Apply P90 inverse target transform")
checklist_lines.append("- [ ] Apply saved P50 calibration")
checklist_lines.append("- [ ] Apply saved P90 calibration")
checklist_lines.append("- [ ] Apply saved ETA floor/median/cap post-processing")
checklist_lines.append("- [ ] Enforce `eta_p90_sec >= eta_p50_sec`")
checklist_lines.append("")
checklist_lines.append("## Output contract")
checklist_lines.append("")
checklist_lines.append("- [ ] Return `eta_p50_sec`")
checklist_lines.append("- [ ] Return `eta_p90_sec`")
checklist_lines.append("- [ ] Return `next_type`")
checklist_lines.append("- [ ] Return `type_conf`")
checklist_lines.append("")
checklist_lines.append("## Dashboard/display logic")
checklist_lines.append("")
checklist_lines.append(f"- [ ] Use global confidence threshold `{CFG.confidence_threshold}` unless per-class thresholding is intentionally enabled")
checklist_lines.append("- [ ] Hide low-confidence type predictions from alert dashboard if needed")
checklist_lines.append("- [ ] Log all predictions, even hidden ones, for monitoring")
checklist_lines.append("")
checklist_lines.append("## Monitoring")
checklist_lines.append("")
checklist_lines.append("- [ ] Track P50 absolute error against actual target events")
checklist_lines.append("- [ ] Track P90 coverage globally and per machine")
checklist_lines.append("- [ ] Track next-type accuracy/confusion matrix")
checklist_lines.append("- [ ] Track confidence distribution drift")
checklist_lines.append("- [ ] Track new/unseen statuses")
checklist_lines.append("- [ ] Retrain when machine/status behavior drifts")

phase5_checklist_md = "\n".join(checklist_lines)

with open(phase5_checklist_md_path, "w", encoding="utf-8") as f:
    f.write(phase5_checklist_md)

report_files = pd.DataFrame([
    {
        "file": model_card_md_path.name,
        "path": str(model_card_md_path.resolve()),
        "size_kb": model_card_md_path.stat().st_size / 1024,
    },
    {
        "file": phase5_checklist_md_path.name,
        "path": str(phase5_checklist_md_path.resolve()),
        "size_kb": phase5_checklist_md_path.stat().st_size / 1024,
    },
])

print("=" * 100)
print("P4-39 — FINAL MODEL CARD AND PHASE 5 CHECKLIST")
print("=" * 100)

print("\nSaved report files:")
display(report_files)

print("\nModel card preview:")
display(Markdown(model_card_md[:4000]))

print("=" * 100)
print("P4-39 ready")
print("=" * 100)

P4-40 — Create final artifact manifest and ZIP bundle

In [ ]:
# ============================================================
# P4-40 — SAFE final artifact manifest and ZIP bundle
# ============================================================

import hashlib
import zipfile
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd

artifact_dir = Path(CFG.artifact_dir).resolve()
zip_output_dir = artifact_dir.parent.resolve()
zip_path = zip_output_dir / f"phase4_artifacts_{CFG.process_id}.zip"

# Remove old zip if exists
if zip_path.exists():
    print("Deleting old zip:", zip_path)
    zip_path.unlink()

def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            b = f.read(block_size)
            if not b:
                break
            h.update(b)

    return h.hexdigest()


# Files to include
required_phase5_files = [
    "lgbm_quantile_p50_final.pkl",
    "lgbm_quantile_p90_final.pkl",
    "lgbm_next_type.pkl",
    "artifacts_phase4.pkl",
]

optional_report_files = [
    "artifacts_phase4_summary.json",
    "model_card_phase4.json",
    "model_card_phase4.md",
    "phase5_integration_checklist.md",
    "feature_importance_p50.csv",
    "feature_importance_p90.csv",
    "feature_importance_next_type.csv",
    "feature_importance_combined.csv",
    "feature_importance_aggregate.csv",
]

files_to_package = []

for fname in required_phase5_files + optional_report_files:
    p = artifact_dir / fname
    if p.exists() and p.is_file():
        files_to_package.append(p)

# Build manifest before zip
manifest_records = []

for p in files_to_package:
    manifest_records.append({
        "file": p.name,
        "path": str(p),
        "size_bytes": int(p.stat().st_size),
        "size_mb": float(p.stat().st_size / (1024 ** 2)),
        "sha256": sha256_file(p),
    })

artifact_manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "artifact_dir": str(artifact_dir),
    "zip_output_dir": str(zip_output_dir),
    "process_id": CFG.process_id,
    "pipeline_name": CFG.pipeline_name,
    "deployment_readiness": DEPLOYMENT_READINESS,
    "ready_for_phase5_integration": bool(READY_FOR_PHASE5_INTEGRATION),
    "required_phase5_files": required_phase5_files,
    "files": manifest_records,
}

manifest_path = artifact_dir / "artifact_manifest.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(artifact_manifest, f, indent=2, ensure_ascii=False)

files_to_package.append(manifest_path)

# Create zip safely
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for p in files_to_package:
        zf.write(p, arcname=p.name)
        print("Added:", p.name)

zip_report = {
    "zip_file": zip_path.name,
    "zip_path": str(zip_path),
    "zip_size_mb": zip_path.stat().st_size / (1024 ** 2),
    "zip_sha256": sha256_file(zip_path),
}

manifest_df = pd.DataFrame([
    {
        "file": p.name,
        "size_mb": p.stat().st_size / (1024 ** 2),
        "included_in_zip": True,
    }
    for p in files_to_package
])

final_checks = []

def add_final_check(name, passed, detail, severity="ERROR"):
    final_checks.append({
        "check": name,
        "severity": severity,
        "passed": bool(passed),
        "detail": detail,
    })

for fname in required_phase5_files:
    fpath = artifact_dir / fname
    add_final_check(
        f"final_required_file_exists__{fname}",
        fpath.exists() and fpath.stat().st_size > 0,
        f"path={str(fpath)}, size_bytes={fpath.stat().st_size if fpath.exists() else 0}",
    )

add_final_check(
    "artifact_manifest_exists",
    manifest_path.exists() and manifest_path.stat().st_size > 0,
    f"path={str(manifest_path)}, size_bytes={manifest_path.stat().st_size if manifest_path.exists() else 0}",
)

add_final_check(
    "zip_bundle_exists",
    zip_path.exists() and zip_path.stat().st_size > 0,
    f"path={str(zip_path)}, size_mb={zip_path.stat().st_size / (1024 ** 2):.3f}",
)

add_final_check(
    "zip_is_outside_artifact_dir",
    zip_path.parent != artifact_dir,
    f"zip_parent={zip_path.parent}, artifact_dir={artifact_dir}",
)

add_final_check(
    "phase5_integration_ready",
    READY_FOR_PHASE5_INTEGRATION is True,
    f"READY_FOR_PHASE5_INTEGRATION={READY_FOR_PHASE5_INTEGRATION}",
)

final_packaging_validation_df = pd.DataFrame(final_checks)

failed_final_packaging_errors = final_packaging_validation_df[
    (final_packaging_validation_df["severity"].eq("ERROR")) &
    (~final_packaging_validation_df["passed"])
]

AUTOTRAIN_PIPELINE_COMPLETE = len(failed_final_packaging_errors) == 0

print("=" * 100)
print("P4-40 — SAFE FINAL ARTIFACT MANIFEST AND ZIP BUNDLE")
print("=" * 100)

print("\nPackaged files:")
display(manifest_df)

print("\nZIP bundle report:")
print(json.dumps(zip_report, indent=2, ensure_ascii=False))

print("\nFinal packaging validation gates:")
display(final_packaging_validation_df)

print("\nFailed final packaging ERROR gates:")
display(failed_final_packaging_errors)

print("=" * 100)
if AUTOTRAIN_PIPELINE_COMPLETE:
    print("AUTOTRAIN_PIPELINE_COMPLETE = True")
    print("Automatic Model Training Pipeline is complete.")
    print("Artifact bundle is ready for Phase 5 integration.")
else:
    print("AUTOTRAIN_PIPELINE_COMPLETE = False")
    print("Final packaging failed. Fix before handoff.")
print("=" * 100)